# KOM82 – Bundestagsprotokollanalyse · 20. Wahlperiode, Sitzungen 146–212

| | |
|:--|:--|
| **Autor** | Stefan Stadtherr |
| **Notebook** | `KOM82_NLP_v01.ipynb` |
| **Stand** | 29.10.2026 |
| **Eingabe** | Original-PDFs `data/pdf/20_146.pdf … 20_212.pdf` (Download außerhalb dieses Notebooks) |
| **Referenz (nur Evaluation)** | `data/validation/dip/`, `data/validation/xml/` – werden ausschließlich in Kapitel 14 gelesen |

**Leitidee.** Das Protokoll trägt seine Struktur in Typografie, Einzug und Inhaltsverzeichnis. Die Pipeline liest diese Struktur beim Extrahieren, statt sie nachträglich aus Fließtext zu rekonstruieren: Sprecherköpfe über die Kopfschrift, Zwischenrufe über den Einzug, Gesetzestitel über den Fettdruck, Debatten und Redner über das Inhaltsverzeichnis als Soll. Deterministische Verfahren liefern die Kandidaten, das lokale LLM ergänzt nur frei formulierte Information – jeder übernommene LLM-Wert muss wörtlich im PDF belegt sein.

### Inhaltsverzeichnis<a class="anchor" id="toc"></a>

- [0) Aufgabe, Leitprinzipien und Architektur](#k0)
- [1) Konfiguration](#k1)
- [2) Laufzeitumgebung und Modelle (lokal / RunPod)](#k2)
- [3) Datenbasis und Split](#k3)
- [4) Layout: Wörter → Zeilen in Lesereihenfolge](#k4)
- [5) Dokumentmodell: Inhaltsverzeichnis, Beiträge, Debatten](#k5)
- [6) Redeeinheiten, Kennzahlen und LLM-taugliches Textformat](#k6)
- [7) Gesetze: Extraktion, Identität, Abstimmung](#k7)
- [8) Beteiligung und Hybrid-Retrieval (BM25 + Dense, RRF)](#k8)
- [9) Sentiment](#k9)
- [10) Ergebnistabellen: Redner und Parteien](#k10)
- [11) Graph der Relationen (SQLite, Neo4j-Export)](#k11)
- [12) Visualisierung](#k12)
- [13) LLM-basierte Auswertung (RAG mit Belegen)](#k13)
- [14) Evaluation](#k14)
- [15) Export und Laufzeitbericht](#k15)
- [16) Grenzen und Ausblick](#k16)

## [↩](#toc) 0) Aufgabe, Leitprinzipien und Architektur<a class="anchor" id="k0"></a>

**Fragestellungen.** (1) Welche Gesetze wurden diskutiert oder abgestimmt, und wer (inkl. Partei) war beteiligt? (2) Welche Abgeordneten haben wie oft und wie lange gesprochen, mit welchem Sentiment? Grundlage ist eine vollständige Pipeline vom PDF bis zur Visualisierung mit einem LLM-basierten Auswertungsschritt.

**Leitprinzipien** (abgeleitet aus den Schwächen früherer Versionen)

| Prinzip | Umsetzung |
|:--|:--|
| Struktur vor Text | Wörter mit Schrift, Größe und Position (`rawdict`); Sprecherköpfe, Zwischenrufe, Tagesordnungstext und Tabellen entstehen beim Zeilenaufbau, nicht durch Nachreparatur |
| Das Inhaltsverzeichnis ist das Soll | Debatten, Vorlagen und Redner (mit Druckseite und Quadrant) aus dem IV; jeder Redebeitrag wird gegen das IV verknüpft – die Abdeckung ist eine referenzfreie Kennzahl je Protokoll |
| Die Sitzungsleitung erklärt jeden Beitrag | Kurzintervention, Erwiderung, Zwischenfrage, Geschäftsordnung aus der Worterteilung; Abstimmungsergebnisse aus ihren Äußerungen |
| Deterministisch zuerst, LLM für den Rest | Gesetzestitel = fett gesetzte Spanne des IV-Eintrags; das LLM klassifiziert Grenzfälle und liest Abstimmungsergebnisse – nur wörtlich belegte Werte werden übernommen |
| Evidenz statt Behauptung | Jede Zeile trägt Seite, Spalte, Druckseite und Quadrant; Parteien nur, wenn im PDF belegt (Kopf → Abstimmungsliste → Zwischenruf), sonst „nicht im PDF belegt“ |
| Messen ab Tag 1, Test nur einmal | eingefrorener Split 43/12/12; Entwicklung nur auf DEV, Validierung nach dem Methoden-Freeze, Test einmalig (protokolliert) |
| Methode und Messung trennen | die Evaluation (Kap. 14) hat eigene Normierer und ist nur über Ergebnistabellen gekoppelt |

**Architektur**

| Stufe | Eingabe → Ausgabe | Verfahren | Kap. | Prüfung |
|:--|:--|:--|:--|:--|
| Layout | PDF → Zeilen (Spalte, Einzug, Schrift, Fundstelle) | PyMuPDF `rawdict`, Grundlinien, Spaltenlücke, Bänder | 4 | Demo-Seite |
| Dokumentmodell | Zeilen → IV, Beiträge, Zwischenrufe, Debatten | IV-Grammatik, Kopfschrift, Klammertiefe, Soll-Ist-Verknüpfung | 5 | IV-Abdeckung je Protokoll |
| Reden | Beiträge → Redeeinheiten, Kennzahlen, Markdown | Fortsetzungsregel, Kennzahlen, Text-Export | 6 | Verteilungen |
| Gesetze | IV-Gegenstände → Vorlagen, Gesetze, Ergebnisse | Regel + LLM (JSON-Schema, Belegprüfung), Drucksache als Schlüssel | 7 | Regel/LLM/Hybrid gegen DIP |
| Beteiligung | Reden × Vorlagen | Debatte (Struktur) und Inhalt (BM25 + Dense, RRF) | 8 | gegen DIP-Vorgangsbezug |
| Sentiment | Sätze → Rede | ParlaSent (XLM-R), Formelfilter | 9 | κ gegen Annotation |
| Graph | Tabellen → Property-Graph | SQLite + rekursive CTEs, Neo4j-CSV | 11 | Abfragen |
| Auswertung | Frage → Antwort mit Belegen | Graph-Filter + Hybrid-RRF + LLM | 13 | Belegnummern |

**Werkzeugwahl**

| Baustein | Werkzeug | Begründung |
|:--|:--|:--|
| PDF | PyMuPDF (`rawdict`), Rückfall pdfplumber | Zeichen mit Schriftname, Größe, Grundlinie; schnell (≈ 1 s/Protokoll); Lesereihenfolge wird selbst aus der Geometrie gebildet und ist damit backendunabhängig |
| Datenmodell | pandas, Pickle-Stufencache, CSV/Parquet/Excel | Tabellen mit Provenienz; Cache-Schlüssel = PDF-Hash + Parameter + **Bytecode der Verfahren** |
| LLM / Embeddings | Ollama: `qwen3:8b`, `qwen3-embedding:0.6b` | lokal auf 8 GB VRAM; JSON-Schema-Decoding, Temperatur 0, fester Seed; identische Modelle auf RunPod (Digest-Prüfung) |
| Retrieval | eigenes BM25 + Dense, Reciprocal Rank Fusion | Fusion ohne Score-Kalibrierung; robust bei unterschiedlichen Score-Skalen |
| Graph | SQLite (Standardbibliothek) | eine Datei, rekursive CTEs für Pfade/Nachbarschaften, Export für Neo4j |
| Sentiment | `classla/xlm-r-parlasent` | auf Parlamentsreden trainiert, deterministisch, > 100 000 Sätze in Minuten |

**Warum kein flächiges RAG für die Gesetzesextraktion?** Gesucht sind *alle* Gesetze einer Sitzung. Ähnlichkeitssuche findet zuverlässig *einen* relevanten Abschnitt, aber nicht verlässlich *alle* benötigten; Chunking verliert zudem Dokumentmetadaten wie Sprecher und Tagesordnungspunkt (Banerjee et al., *The Bottleneck Is Retrieval, Not Reading*). Das Inhaltsverzeichnis liefert dagegen eine vollständige, deterministische Kandidatenmenge. Retrieval wird dort eingesetzt, wo es Mehrwert hat: Zuordnung von Reden zu Vorlagen in verbundenen Debatten und belegte Antworten auf Fragen (Kap. 8, 13). Das Kontextformat der RAG-Antworten folgt „DOS RAG“ (Originalreihenfolge der Passagen; Laitenberger et al., *Stronger Baselines for RAG*).

## [↩](#toc) 1) Konfiguration<a class="anchor" id="k1"></a>

Alle Parameter stehen in **einer** Zelle. Schwellen der Verfahren (Layout, Zuordnung) sind in den Klassen als dokumentierte Standardwerte hinterlegt und werden hier nur bei Bedarf überschrieben.

| Parameter | Bedeutung / Begründung |
|:--|:--|
| `MODUS` | `dev`: Verarbeitung + Evaluation gegen DIP/XML · `betrieb`: alle 67 Protokolle, LLM Pflicht, keine Referenzdaten |
| `SITZUNGEN` | nur `dev`: Teilmenge, z. B. `[146, 199, 205]` |
| `PDF_BACKEND` | `auto` = PyMuPDF, falls installiert; pdfplumber liefert dieselbe Wörtertabelle (≈ 20× langsamer) |
| `MODELL_PARAMETER` | Ollama-URL, LLM, Embeddingmodell, Parallelität. `qwen3:8b` (≈ 5,2 GB Q4) passt vollständig in 8 GB VRAM; `qwen3-embedding:0.6b` (≈ 0,6 GB) ist mehrsprachig und instruktionsfähig. Für einen Qualitätsvergleich: `qwen3-embedding:4b` (2,5 GB) |
| `RUNPOD_AKTIV` | `True`: Pod starten, **dieselben** Modelle laden, Digests lokal/Pod vergleichen, SSH-Tunnel; RunPod dient nur der Geschwindigkeit |
| `DENSE_UMFANG` | `alle`: Dense-Index über alle Reden (RAG über den Korpus) · `verbundene`: nur Debatten mit mehreren Vorlagen (schneller) |
| `RRF_K` | 60 nach Cormack et al. (2009): dämpft den Einfluss einzelner Spitzenränge |
| `ZUORDNUNG_DELTA` | Toleranz zum besten fusionierten Score bei der Zuordnung Rede → Vorlage; einziger Zuordnungsparameter, Kalibrierung nur auf DEV (Kap. 14) |
| `GESETZ_MODUS` | `regel` / `llm` / `hybrid`; Vergleich aller drei gegen DIP in Kap. 14 |
| `EVAL_SPLITS`, `TEST_FREIGABE` | Entwicklung → Validierung (nach Freeze) → Test (einmalig, protokolliert) |

In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import math
import os
import pickle
import re
import sqlite3
import sys
import threading
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Iterable, Literal

import numpy as np
import pandas as pd
import requests

try:
    from IPython.display import display, Markdown
except ImportError:                                   # Ausführung außerhalb von Jupyter (Tests)
    def display(*a, **k):
        for x in a:
            print(x)
    Markdown = str


def _modul_da(name: str) -> bool:
    import importlib.util
    return importlib.util.find_spec(name) is not None

In [2]:
# ---------------------------------------------------------------------
# Zentrale Konfiguration – alle Parameter mit Begründung im Markdown oben
# ---------------------------------------------------------------------
PROJECT_DIR = Path(os.environ.get("KOM82_PROJECT_DIR", Path.cwd())).resolve()   # Ordner mit data/
MODUS = "dev"                    # "dev": + Evaluation gegen DIP/XML | "betrieb": alle 67 Protokolle, ohne Referenz
SITZUNGEN = None                 # nur dev: z. B. [146, 199, 205]; None = alle vorhandenen 146–212
PDF_BACKEND = "auto"             # "pymupdf" (Standard) | "pdfplumber" (Rückfall, ca. 20× langsamer)
BACKEND_VERGLEICH = False        # dev: PyMuPDF vs. pdfplumber auf einem Protokoll (Strukturkennzahlen müssen gleich sein)

DATA_DIR = PROJECT_DIR / "data"
PDF_DIR = DATA_DIR / "pdf"                         # 20_146.pdf … 20_212.pdf (Downloader liegt außerhalb)
VALIDATION_DIR = DATA_DIR / "validation"           # dip/, xml/ – nur Kapitel 14 (Evaluation)
SPLIT_PATH = PROJECT_DIR / "evaluation" / "protocol_split.json"
OUTPUT_DIR = PROJECT_DIR / "output"
CACHE_DIR, ERGEBNIS_DIR = OUTPUT_DIR / "cache", OUTPUT_DIR / "ergebnisse"
TEXT_DIR, FIGUR_DIR, EVAL_DIR = OUTPUT_DIR / "text", OUTPUT_DIR / "abbildungen", OUTPUT_DIR / "evaluation"
for _o in (CACHE_DIR, ERGEBNIS_DIR, TEXT_DIR, FIGUR_DIR, EVAL_DIR):
    _o.mkdir(parents=True, exist_ok=True)

# Modelle: lokal und auf RunPod IDENTISCH (RunPod nur für Geschwindigkeit)
LLM_AKTIV = True
MODELL_PARAMETER = dict(
    url=os.environ.get("OLLAMA_URL", "http://localhost:11434"),
    llm="qwen3:8b",                  # 5,2 GB Q4: passt vollständig in 8 GB VRAM
    embed="qwen3-embedding:0.6b",    # 0,6 GB, mehrsprachig, instruktionsfähig; Alternative: "qwen3-embedding:4b", "bge-m3"
    parallel=1)
RUNPOD_AKTIV = False             # True: Pod starten, Modelle prüfen (Digest-Gleichheit), Tunnel, danach beenden
RUNPOD = {"gpu": "NVIDIA A40", "parallel": 4, "modul": "my_utils.runpod_pod", "port": 11435}
DENSE_UMFANG = "alle"            # "alle" Reden (RAG über den Korpus) | "verbundene" (nur Zuordnung)
RRF_K = 60                       # Reciprocal Rank Fusion, k nach Cormack et al. (2009)
ZUORDNUNG_DELTA = 0.10           # Rede -> Vorlage: Toleranz zum besten RRF-Score (Kalibrierung Kap. 14, nur DEV)
GESETZ_MODUS = "hybrid"          # "regel" | "llm" | "hybrid" (Vergleich in Kap. 14)

SENTIMENT_AKTIV = True
SENTIMENT_MODELL = "classla/xlm-r-parlasent"

EVAL_SPLITS = ("entwicklung",)   # nach dem Methoden-Freeze: ("validierung",), zuletzt einmalig ("test",)
TEST_FREIGABE = False            # Test nur mit ausdrücklicher Freigabe; jede Auswertung wird protokolliert
ZIP_ERSTELLEN = False

SITZUNGEN_ALLE = tuple(range(146, 213))
VERARBEITUNG = SITZUNGEN_ALLE if MODUS == "betrieb" or SITZUNGEN is None else tuple(int(s) for s in SITZUNGEN)

In [3]:
# ---------------------------------------------------------------------
# Ein Logger für alle Kapitel, Laufzeitmessung je Stufe
# ---------------------------------------------------------------------
def get_logger(name: str, log_dir: Path, level: int = logging.INFO) -> logging.Logger:
    log = logging.getLogger(name)
    log.setLevel(level); log.propagate = False
    if not log.handlers:
        fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
        for h in (logging.StreamHandler(), logging.FileHandler(log_dir / f"{name}_{datetime.now():%Y-%m-%d}.log", encoding="utf-8")):
            h.setFormatter(fmt); log.addHandler(h)
    return log


(PROJECT_DIR / "logs").mkdir(exist_ok=True)
logger = get_logger("KOM82_NLP", PROJECT_DIR / "logs")
LAUFZEITEN: dict[str, float] = {}


@contextmanager
def stufe(name: str):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        LAUFZEITEN[name] = round(time.perf_counter() - t0, 1)
        logger.info("Stufe %-28s %8.1f s", name, LAUFZEITEN[name])


logger.info("Start | Modus=%s | Protokolle=%d | Backend=%s", MODUS, len(VERARBEITUNG), PDF_BACKEND)

20:52:46 | INFO    | Start | Modus=dev | Protokolle=67 | Backend=auto


## [↩](#toc) 2) Laufzeitumgebung und Modelle (lokal / RunPod)<a class="anchor" id="k2"></a>

**Modellwahl bei 8 GB VRAM.** Maßgeblich ist nicht die Downloadgröße allein, sondern Gewichte + KV-Cache. `qwen3:8b` (Q4, ≈ 5,2 GB) läuft mit `num_ctx = 8192` vollständig auf der GPU; die Aufgaben sind kurz und eng belegt (ein Tagesordnungseintrag, eine Abstimmungspassage). Größere Modelle (`qwen3:14b`, `gpt-oss:20b`) würden auf die CPU auslagern und sind für den lokalen Massendurchlauf ungeeignet. Embeddings und LLM werden nacheinander genutzt, sodass Ollama das jeweils aktive Modell im VRAM hält.

**Backend-Vertrag (`OllamaBackend`).** `/api/chat` mit `format` = JSON-Schema (constrained decoding), `temperature 0`, fester Seed, `think: false`; bei ungültiger Antwort ein zweiter Versuch mit anderem Seed. Alle Antworten und Embeddings werden in SQLite gecacht; der Schlüssel enthält den **Modell-Digest**. Daraus folgt: (1) Wiederholungsläufe sind ohne GPU reproduzierbar (Nachvollzug aus dem Cache), (2) lokale und RunPod-Ergebnisse sind austauschbar, weil gleiche Gewichte denselben Schlüssel erzeugen.

**RunPod (`RunPodSitzung`).** Nutzt die vorhandenen Module `my_utils.runpod_pod` (Pod, Wächter, Setup, Tunnel). Garantien: keine laufenden Pods vor dem Start; Ollama-Version wie lokal; Abbruch, wenn ein Modell-Digest lokal ≠ Pod; Zugriff nur über `localhost:11435`; nach Kapitel 15 wird der Pod beendet und geprüft, dass keiner mehr läuft. Auf dem Pod wird mit `OLLAMA_NUM_PARALLEL = 4` parallel angefragt. Bricht das Notebook vorher ab, beendet `RUNPOD_SITZUNG.__exit__(None, None, None)` (oder `rp.beenden()`) den Pod; zusätzlich terminiert der Wächter ihn nach der eingestellten Leerlauf- bzw. Höchstzeit.

In [4]:
# =====================================================================
# Modell-Backend: Ollama lokal oder per SSH-Tunnel auf RunPod (gleiche Modelle, gleiche Gewichte)
# =====================================================================
@dataclass
class ModellKonfig:
    url: str = "http://localhost:11434"
    llm: str = "qwen3:8b"
    embed: str = "qwen3-embedding:0.6b"
    optionen: dict = field(default_factory=lambda: {"temperature": 0.0, "seed": 42, "num_ctx": 8192,
                                                    "num_predict": 1024})
    timeout: int = 300
    parallel: int = 1                     # gleichzeitige Anfragen (RunPod: = OLLAMA_NUM_PARALLEL)
    embed_batch: int = 32
    embed_max_zeichen: int = 6000         # Schutz vor Überlänge (Chunks sind ohnehin ≤ 1.500 Zeichen)


class OllamaBackend:
    """Schlanker HTTP-Client für /api/chat (JSON-Schema, Temperatur 0, fester Seed, ohne 'thinking')
    und /api/embed. Alle Antworten werden persistent gecacht (SQLite), Schlüssel = Modell-Digest +
    Eingabe + Optionen. Damit sind Läufe reproduzierbar und lokale/RunPod-Ergebnisse austauschbar."""

    def __init__(self, k: ModellKonfig, cache: Path, logger: logging.Logger, name: str = "lokal"):
        self.k, self.log, self.name = k, logger, name
        self.url = k.url.rstrip("/")
        self.db = sqlite3.connect(str(cache), check_same_thread=False)
        self.db.execute("CREATE TABLE IF NOT EXISTS chat (key TEXT PRIMARY KEY, modell TEXT, antwort TEXT, "
                        "sekunden REAL, server TEXT, zeit TEXT)")
        self.db.execute("CREATE TABLE IF NOT EXISTS emb (key TEXT PRIMARY KEY, modell TEXT, dim INT, vec BLOB)")
        self.db.execute("CREATE TABLE IF NOT EXISTS modelle (name TEXT PRIMARY KEY, digest TEXT)")
        self.db.commit()
        self._digest: dict[str, str] = {}
        self.statistik = Counter()
        self._lock = threading.Lock()

    # ---------------------------------------------------------------- Status
    def status(self) -> dict:
        try:
            v = requests.get(f"{self.url}/api/version", timeout=5).json().get("version")
            tags = requests.get(f"{self.url}/api/tags", timeout=5).json().get("models", [])
        except requests.RequestException as exc:
            return {"erreichbar": False, "fehler": type(exc).__name__}
        self._digest = {m["name"]: m.get("digest", "") for m in tags}
        with self._lock:
            self.db.executemany("INSERT OR REPLACE INTO modelle VALUES (?,?)", self._digest.items())
            self.db.commit()
        return {"erreichbar": True, "version": v, "modelle": sorted(self._digest)}

    def verfuegbar(self, modell: str) -> bool:
        if not self._digest:
            self.status()
        return modell in self._digest or f"{modell}:latest" in self._digest

    def digest(self, modell: str) -> str:
        """Modell-Digest (identische Gewichte = identischer Cache-Schlüssel). Ohne Server (Nachvollzug aus
        dem Cache) wird der zuletzt gesehene Digest aus der Cache-Datenbank verwendet."""
        if not self._digest:
            self.status()
        if not self._digest:
            self._digest = dict(self.db.execute("SELECT name, digest FROM modelle").fetchall())
        return self._digest.get(modell) or self._digest.get(f"{modell}:latest") or modell

    # ---------------------------------------------------------------- Chat
    def chat_json(self, system: str, nutzer: str, schema: dict, modell: str | None = None) -> dict | None:
        modell = modell or self.k.llm
        key = hashlib.sha256(json.dumps([self.digest(modell), system, nutzer, schema, self.k.optionen],
                                        sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        with self._lock:
            row = self.db.execute("SELECT antwort FROM chat WHERE key=?", (key,)).fetchone()
        if row:
            self.statistik["chat_cache"] += 1
            return json.loads(row[0])
        t0 = time.perf_counter()
        for versuch in range(2):
            opt = dict(self.k.optionen, seed=self.k.optionen.get("seed", 42) + versuch)
            try:
                r = requests.post(f"{self.url}/api/chat", timeout=self.k.timeout, json={
                    "model": modell, "stream": False, "think": False, "format": schema, "options": opt,
                    "messages": [{"role": "system", "content": system}, {"role": "user", "content": nutzer}]})
                r.raise_for_status()
                inhalt = r.json()["message"]["content"]
                antwort = json.loads(inhalt)
            except (requests.RequestException, json.JSONDecodeError, KeyError) as exc:
                self.log.warning("LLM-Antwort ungültig (Versuch %d): %s", versuch + 1, type(exc).__name__)
                continue
            dauer = time.perf_counter() - t0
            with self._lock:
                self.db.execute("INSERT OR REPLACE INTO chat VALUES (?,?,?,?,?,?)",
                                (key, modell, json.dumps(antwort, ensure_ascii=False), dauer, self.name,
                                 datetime.now().isoformat(timespec="seconds")))
                self.db.commit()
            self.statistik["chat_neu"] += 1
            return antwort
        self.statistik["chat_fehler"] += 1
        return None

    def chat_viele(self, aufgaben: list[tuple[str, str, dict]]) -> list[dict | None]:
        """Mehrere Anfragen; parallel, wenn der Server es erlaubt (RunPod: OLLAMA_NUM_PARALLEL)."""
        if self.k.parallel <= 1:
            return [self.chat_json(*a) for a in aufgaben]
        with ThreadPoolExecutor(self.k.parallel) as ex:
            return list(ex.map(lambda a: self.chat_json(*a), aufgaben))

    # ---------------------------------------------------------------- Embeddings
    def embed(self, texte: list[str], modell: str | None = None, praefix: str = "") -> np.ndarray:
        modell = modell or self.k.embed
        dg = self.digest(modell)
        texte = [(praefix + t)[: self.k.embed_max_zeichen] for t in texte]
        keys = [hashlib.sha256(f"{dg}|{t}".encode()).hexdigest() for t in texte]
        vorhanden = {}
        for i in range(0, len(keys), 900):
            teil = keys[i:i + 900]
            q = f"SELECT key, dim, vec FROM emb WHERE key IN ({','.join('?' * len(teil))})"
            for key, dim, vec in self.db.execute(q, teil):
                vorhanden[key] = np.frombuffer(vec, dtype=np.float32)
        fehlend = [i for i, key in enumerate(keys) if key not in vorhanden]
        self.statistik["emb_cache"] += len(keys) - len(fehlend)
        stapel = [fehlend[i:i + self.k.embed_batch] for i in range(0, len(fehlend), self.k.embed_batch)]
        def hole(idx):
            r = requests.post(f"{self.url}/api/embed", timeout=self.k.timeout,
                              json={"model": modell, "input": [texte[i] for i in idx], "truncate": True})
            r.raise_for_status()
            return idx, r.json()["embeddings"]
        if self.k.parallel <= 1:
            ergebnisse = map(hole, stapel)
        else:
            with ThreadPoolExecutor(self.k.parallel) as ex:
                ergebnisse = list(ex.map(hole, stapel))
        for idx, vecs in ergebnisse:
            with self._lock:
                for i, v in zip(idx, vecs):
                    a = np.asarray(v, dtype=np.float32)
                    a /= max(float(np.linalg.norm(a)), 1e-12)
                    vorhanden[keys[i]] = a
                    self.db.execute("INSERT OR REPLACE INTO emb VALUES (?,?,?,?)", (keys[i], modell, len(a), a.tobytes()))
                self.db.commit()
            self.statistik["emb_neu"] += len(idx)
        return np.vstack([vorhanden[k] for k in keys]) if keys else np.zeros((0, 1), dtype=np.float32)


class RunPodSitzung:
    """Kontextmanager für einen RunPod-Pod mit Ollama (Module aus my_utils, siehe Projektordner).
    Garantien: (1) vor dem Start und nach dem Ende keine laufenden Pods, (2) dieselbe Ollama-Version
    wie lokal, (3) identische Modell-Digests lokal/Pod – sonst Abbruch, (4) Zugriff nur per SSH-Tunnel.
    Der Pod dient ausschließlich der Geschwindigkeit; Modelle und Ergebnisse bleiben gleich."""

    def __init__(self, modelle: list[str], logger: logging.Logger, gpu: str = "NVIDIA A40",
                 lokaler_port: int = 11435, parallel: int = 4, modul: str = "my_utils.runpod_pod",
                 auto_stop_min: int = 120, hard_stop_min: int = 360):
        import importlib
        self.rp = importlib.import_module(modul)
        self.modelle, self.log, self.gpu = list(dict.fromkeys(modelle)), logger, gpu
        self.port, self.parallel = lokaler_port, parallel
        self.auto, self.hard = auto_stop_min, hard_stop_min
        self.pod = self.tunnel = None

    def __enter__(self) -> str:
        vorher = self.rp.laufende()
        if vorher:
            raise RuntimeError(f"Es laufen bereits Pods: {vorher} – erst beenden (rp.beenden)")
        self.pod, ip, port = self.rp.starten(gpu=self.gpu, auto_stop_min=self.auto, hard_stop_min=self.hard)
        try:
            self.rp.einrichten(ip, port, modelle=" ".join(self.modelle), parallel=self.parallel, ollama_version="lokal")
            vergleich = {name: status for name, _, _, status in self.rp.modelle_vergleichen(ip, port)}
            abweichend = [m for m in self.modelle if vergleich.get(m, vergleich.get(f"{m}:latest")) != "identisch"]
            if abweichend:
                raise RuntimeError(f"Modelle lokal/Pod nicht identisch: {abweichend}")
            self.tunnel = self.rp.tunnel(ip, port, lokal=self.port)
        except BaseException:
            self.rp.beenden(self.pod, self.tunnel)
            raise
        self.log.info("RunPod bereit | GPU=%s | Modelle identisch: %s", self.gpu, self.modelle)
        return f"http://localhost:{self.port}"

    def __exit__(self, *exc):
        self.rp.beenden(self.pod, self.tunnel)
        rest = self.rp.laufende()
        if rest:
            self.log.error("Nach dem Beenden laufen noch Pods: %s", rest)
        else:
            self.log.info("RunPod beendet, keine Pods aktiv")
        return False

In [5]:
# ---------------------------------------------------------------------
# Laufzeitumgebung: Bibliotheken, GPU, Modellserver (ohne Inferenz)
# ---------------------------------------------------------------------
def umgebung() -> dict:
    from importlib.metadata import version, PackageNotFoundError
    info = {"Python": sys.version.split()[0], "pandas": pd.__version__}
    for paket in ("pymupdf", "pdfplumber", "transformers", "pyarrow", "networkx"):
        try:
            info[paket] = version(paket)
        except PackageNotFoundError:
            info[paket] = None
    try:
        import torch
        info["torch"], info["CUDA"] = torch.__version__, torch.cuda.is_available()
        if info["CUDA"]:
            p = torch.cuda.get_device_properties(0)
            info["GPU"] = f"{p.name} | {p.total_memory / 2**30:.1f} GB"
    except ImportError:
        info["torch"] = None
    return info


UMGEBUNG = umgebung()
for k, v in UMGEBUNG.items():
    print(f"{k:<14} {v}")

MODELLE = ModellKonfig(**MODELL_PARAMETER)
RUNPOD_SITZUNG = None
if RUNPOD_AKTIV and LLM_AKTIV:                 # Pod läuft bis zum Aufruf von RUNPOD_SITZUNG.__exit__ (Kap. 14)
    RUNPOD_SITZUNG = RunPodSitzung([MODELLE.llm, MODELLE.embed], logger, gpu=RUNPOD["gpu"],
                                   lokaler_port=RUNPOD["port"], parallel=RUNPOD["parallel"], modul=RUNPOD["modul"])
    MODELLE.url, MODELLE.parallel = RUNPOD_SITZUNG.__enter__(), RUNPOD["parallel"]

BACKEND = OllamaBackend(MODELLE, CACHE_DIR / "modelle.sqlite", logger, name="runpod" if RUNPOD_SITZUNG else "lokal")
STATUS = BACKEND.status()
LLM_BEREIT = LLM_AKTIV and STATUS.get("erreichbar", False) and BACKEND.verfuegbar(MODELLE.llm)
EMBED_BEREIT = LLM_AKTIV and STATUS.get("erreichbar", False) and BACKEND.verfuegbar(MODELLE.embed)
print(f"\nModellserver: {MODELLE.url} | {STATUS}")
print(f"LLM {MODELLE.llm}: {'bereit' if LLM_BEREIT else 'NICHT verfügbar (Cache/Regel)'} | "
      f"Embedding {MODELLE.embed}: {'bereit' if EMBED_BEREIT else 'NICHT verfügbar (nur BM25)'}")
if MODUS == "betrieb" and LLM_AKTIV and not LLM_BEREIT:
    raise RuntimeError(f"Betrieb erfordert {MODELLE.llm}: `ollama pull {MODELLE.llm}` und Ollama starten")

Python         3.11.5
pandas         2.2.2
pymupdf        1.28.2
pdfplumber     0.11.9
transformers   5.20.2
pyarrow        16.1.0
networkx       3.3
torch          2.13.0+cu132
CUDA           True
GPU            NVIDIA GeForce RTX 4070 Laptop GPU | 8.0 GB

Modellserver: http://localhost:11434 | {'erreichbar': True, 'version': '0.40.13', 'modelle': ['bge-m3:latest', 'deepseek-coder:latest', 'llama3.2-vision:latest', 'llama3.2:latest', 'nomic-embed-text:latest', 'phi3:latest', 'qwen2.5-coder:7b', 'qwen3-embedding:4b', 'qwen3.5:9b', 'qwen3:8b']}
LLM qwen3:8b: bereit | Embedding qwen3-embedding:0.6b: NICHT verfügbar (nur BM25)


## [↩](#toc) 3) Datenbasis und Split<a class="anchor" id="k3"></a>

Die PDFs liegen in `data/pdf/` (Downloader außerhalb des Notebooks). Der Split **43 Entwicklung / 12 Validierung / 12 Test** wird aus `evaluation/protocol_split.json` gelesen; fehlt die Datei, wird einmalig ein deterministischer, zeitlich geschichteter Split erzeugt (je Zeitblock ein Validierungs- und ein Testprotokoll) und gespeichert – er wird nie neu ausgelost. Verfahren werden ausschließlich auf DEV entwickelt; Validierung und Test dienen der unverzerrten Schätzung.

In [6]:
# =====================================================================
# Datenbasis: eingefrorener Split (Entwicklung / Validierung / Test)
# =====================================================================
class Split:
    """Eingefrorener Protokoll-Split (Entwicklung 43 / Validierung 12 / Test 12). Fehlt die Datei, wird
    einmalig ein deterministischer, zeitlich geschichteter Split erzeugt und gespeichert (nie neu ausgelost).
    Test-Auswertungen werden mit Zeitstempel und Pipelineversion protokolliert (Nachweis der Einmaligkeit)."""

    GROESSEN = {"entwicklung": 43, "validierung": 12, "test": 12}

    def __init__(self, pfad: Path, sitzungen: tuple[int, ...], seed: int = 20261029):
        self.pfad = Path(pfad)
        if self.pfad.is_file():
            roh = json.loads(self.pfad.read_text(encoding="utf-8-sig"))
            self.gruppen = {k: sorted(int(x) for x in roh[k]) for k in self.GROESSEN}
            self.quelle = "eingefroren (Datei)"
        else:
            rng = np.random.default_rng(seed)
            gruppen = {k: [] for k in self.GROESSEN}
            for block in np.array_split(np.array(sitzungen), 12):     # zeitliche Schichtung: je Block 1 VAL, 1 TEST
                block = list(rng.permutation(block))
                gruppen["validierung"].append(int(block.pop())); gruppen["test"].append(int(block.pop()))
                gruppen["entwicklung"] += [int(x) for x in block]
            self.gruppen = {k: sorted(v) for k, v in gruppen.items()}
            self.pfad.parent.mkdir(parents=True, exist_ok=True)
            self.pfad.write_text(json.dumps(self.gruppen, indent=1), encoding="utf-8")
            self.quelle = f"neu erzeugt (Seed {seed}) und gespeichert"
        alle = [s for v in self.gruppen.values() for s in v]
        assert [len(self.gruppen[k]) for k in self.GROESSEN] == list(self.GROESSEN.values()), "Splitgrößen"
        assert len(set(alle)) == len(alle) == len(sitzungen), "Split nicht disjunkt/vollständig"

    def sitzungen(self, gruppen: Iterable[str]) -> list[int]:
        return sorted(s for g in gruppen for s in self.gruppen[g])

    def protokolliere_test(self, ordner: Path, kennzahlen: dict):
        with (Path(ordner) / "test_auswertungen.jsonl").open("a", encoding="utf-8") as f:
            f.write(json.dumps({"zeit": datetime.now().isoformat(timespec="seconds"), "version": PIPELINE_VERSION,
                                **kennzahlen}, ensure_ascii=False, default=str) + "\n")

In [7]:
# ---------------------------------------------------------------------
# Datenbasis: vorhandene PDFs (Download erfolgt außerhalb), Split
# ---------------------------------------------------------------------
vorhanden = sorted(int(p.stem.split("_")[1]) for p in PDF_DIR.glob("20_*.pdf"))
fehlend = sorted(set(VERARBEITUNG) - set(vorhanden))
print(f"PDFs vorhanden: {len(vorhanden)} | angefordert: {len(VERARBEITUNG)} | fehlend: {fehlend or '–'}")
if MODUS == "betrieb" and fehlend:
    raise FileNotFoundError(f"Betrieb erfordert alle 67 Protokolle; es fehlen {fehlend}")
VERARBEITUNG = tuple(s for s in VERARBEITUNG if s in vorhanden)
SPLIT = Split(SPLIT_PATH, SITZUNGEN_ALLE)
print(f"Split: {SPLIT.quelle} | " + " | ".join(f"{k}: {len(v)}" for k, v in SPLIT.gruppen.items()))
print("Verarbeitete Protokolle je Split:", {k: len(set(v) & set(VERARBEITUNG)) for k, v in SPLIT.gruppen.items()})

PDFs vorhanden: 67 | angefordert: 67 | fehlend: –
Split: neu erzeugt (Seed 20261029) und gespeichert | entwicklung: 43 | validierung: 12 | test: 12
Verarbeitete Protokolle je Split: {'entwicklung': 43, 'validierung': 12, 'test': 12}


## [↩](#toc) 4) Layout: Wörter → Zeilen in Lesereihenfolge<a class="anchor" id="k4"></a>

**Verfahren** (`PdfWoerter`, `Layout`)

1. **Wörter** mit Position, **Grundlinie** und Schriftname je Zeichenlauf. Die Grundlinie ist in beiden Backends identisch definiert; daher hängen Zeilen und Spalten nicht vom Backend ab.
2. **Kopf und Rand**: Kolumnentitel/Seitenzahl im oberen Seitenrand (< 9 % der Seitenhöhe), Randmarken `(A)…(D)` außerhalb des Satzspiegels.
3. **Spalten**: breiteste senkrechte Lücke im mittleren Seitendrittel, die kein Wort schneidet. Seiten ohne saubere Lücke (Titelseite, Tabellen) erhalten den **Median des Protokolls** – der Satzspiegel ist konstant.
4. **Zeilen**: Wörter gleicher Grundlinie (Toleranz 1,5 pt); eine Zeile wird nur an einer Lücke **über der Spaltenmitte** geteilt – weite Wortabstände im Blocksatz trennen nicht. Segmente über der Mitte sind seitenbreit (zentrierte Titel).
5. **Lesereihenfolge**: seitenbreite Zeilen trennen horizontale Bänder; im Band erst linke, dann rechte Spalte.
6. **Einzug** relativ zum Satzspiegel (häufigster Zeilenanfang/-schluss je Spalte und Seitenparität). Fließtext 0 pt, Absatzanfang ≈ 10 pt, Sprecherkopf ≈ 9 pt, Zwischenruf beidseitig eingezogen, Tagesordnungstext ≈ 31 pt.
7. **Fundstelle**: Druckseite aus dem Kolumnentitel (gesperrte Ziffern werden zusammengeführt), Quadrant aus den Randmarken (B/D markieren die untere Spaltenhälfte).

Die Zeilen tragen zusätzlich Schriftmerkmale (`fett0`, `sans` = andere Familie als der Fließtext, `kursiv0`, `pt`) und Tabulatorsegmente – die Grundlage für alle Strukturentscheidungen in Kapitel 5.

In [8]:
# =====================================================================
# Layout: PDF -> Wörter mit Schriftmerkmalen -> Zeilen in Lesereihenfolge
# =====================================================================
WORT_SPALTEN = ["seite", "x0", "x1", "yb", "pt", "font", "text"]


class PdfWoerter:
    """Liest alle Wörter eines PDFs mit Position (x0, x1, Grundlinie yb) und Schrift.

    Zwei austauschbare Backends liefern dieselbe Tabelle:
      * ``pymupdf``    – Standard, schnell (``get_text("rawdict")``: Zeichen mit Schrift je Span)
      * ``pdfplumber`` – Rückfall ohne PyMuPDF (pdfminer; ca. 20-mal langsamer)
    Die Grundlinie (nicht die Glyphenbox) ist die Zeilenkoordinate, weil sie in beiden
    Backends identisch definiert ist; damit hängen Zeilenbildung und Spalten nicht vom Backend ab.
    """

    def __init__(self, backend: str = "auto"):
        if backend == "auto":
            backend = "pymupdf" if _modul_da("pymupdf") else "pdfplumber"
        if backend not in ("pymupdf", "pdfplumber"):
            raise ValueError(f"Unbekanntes PDF-Backend: {backend}")
        self.backend = backend

    def version(self) -> str:
        if self.backend == "pymupdf":
            import pymupdf
            return f"pymupdf {pymupdf.VersionBind}"
        import pdfplumber
        return f"pdfplumber {pdfplumber.__version__}"

    def lese(self, pfad: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
        """-> (woerter, seiten) mit seiten = [seite, breite, hoehe]."""
        zeilen, seiten = (self._pymupdf if self.backend == "pymupdf" else self._pdfplumber)(Path(pfad))
        w = pd.DataFrame(zeilen, columns=WORT_SPALTEN)
        w["pt"] = w["pt"].round(1)
        return w, pd.DataFrame(seiten, columns=["seite", "breite", "hoehe"])

    @staticmethod
    def _pymupdf(pfad: Path):
        import pymupdf
        out, seiten = [], []
        with pymupdf.open(pfad) as doc:
            for nr, page in enumerate(doc, 1):
                seiten.append((nr, page.rect.width, page.rect.height))
                raw = page.get_text("rawdict", flags=pymupdf.TEXT_PRESERVE_WHITESPACE
                                    | pymupdf.TEXT_MEDIABOX_CLIP)
                for block in raw["blocks"]:
                    for line in block.get("lines", []):
                        for span in line["spans"]:
                            wort, x0, x1, yb = [], None, None, None
                            for ch in span["chars"]:
                                c = ch["c"]
                                luecke = wort and (ch["bbox"][0] - x1 > 0.15 * span["size"] or ch["bbox"][0] < x1 - 0.5 * span["size"])
                                if c.isspace() or luecke:
                                    if wort:
                                        out.append((nr, x0, x1, yb, span["size"], span["font"], "".join(wort)))
                                    wort = []
                                    if c.isspace():
                                        continue
                                if not wort:
                                    x0, yb = ch["bbox"][0], ch["origin"][1]
                                wort.append(c)
                                x1 = ch["bbox"][2]
                            if wort:
                                out.append((nr, x0, x1, yb, span["size"], span["font"], "".join(wort)))
        return _verbinde_fragmente(out), seiten

    @staticmethod
    def _pdfplumber(pfad: Path):
        import pdfplumber
        out, seiten = [], []
        with pdfplumber.open(pfad) as pdf:
            for nr, page in enumerate(pdf.pages, 1):
                seiten.append((nr, float(page.width), float(page.height)))
                h = float(page.height)
                zeichen = sorted(((h - c["matrix"][5], c) for c in page.chars), key=lambda t: t[0])
                cluster, letzte, k = [], None, -1               # Grundlinien-Cluster (Toleranz 0,5 pt)
                for base, c in zeichen:
                    if letzte is None or base - letzte > 0.5:
                        k += 1
                    cluster.append((k, base, c))
                    letzte = base
                wort, x0, x1, yb, size, font = [], None, None, None, None, None
                for _, base, ch in sorted(cluster, key=lambda t: (t[0], t[2]["x0"])):
                    c = ch["text"]
                    neu = (not wort or c.isspace() or ch["fontname"] != font or abs(base - yb) > 1.0
                           or ch["x0"] - x1 > 0.15 * ch["size"] or ch["x0"] < x1 - 0.5 * ch["size"])
                    if neu and wort:
                        out.append((nr, x0, x1, yb, size, font, "".join(wort)))
                        wort = []
                    if c.isspace():
                        continue
                    if not wort:
                        x0, yb, size, font = ch["x0"], base, ch["size"], ch["fontname"]
                    wort.append(c)
                    x1 = ch["x1"]
                if wort:
                    out.append((nr, x0, x1, yb, size, font, "".join(wort)))
        return _verbinde_fragmente(out), seiten


def _verbinde_fragmente(woerter: list[tuple]) -> list[tuple]:
    """Wortteile ohne Zwischenraum (Schriftwechsel mitten im Wort, z. B. 'Ge|setzes' fett/normal)
    bleiben getrennte Einträge: Die Zeilenbildung setzt Leerzeichen nur bei echtem Abstand."""
    woerter = [w for w in woerter if w[6]]
    return [(s, x0, x1, yb, sz, f.split("+")[-1], t) for s, x0, x1, yb, sz, f, t in woerter]


@dataclass
class LayoutParameter:
    """Alle Layoutschwellen in Punkt (pt) bzw. als Seitenanteil – begründet im Markdown zu 3)."""
    kopf_anteil: float = 0.09          # Kolumnentitel/Seitenzahl im oberen Seitenrand (≈ 76 pt bei A4)
    grundlinie_tol: float = 1.5        # Wörter einer Zeile: Grundlinien weichen < 1,5 pt ab
    wortabstand: float = 0.12          # Leerzeichen, wenn Lücke > 0,12 · Schriftgröße
    einzug: float = 3.5                # Einzug gilt ab 3,5 pt (Absatzeinzug ≈ 9 pt, Blocksatz 0 pt)


RANDMARKE = re.compile(r"^\([A-D]\)$")


class Layout:
    """Zeilen mit Spalte, Einzug, Schrift und Fundstelle (Druckseite + Quadrant A–D).

    Spalten werden je Seite aus den Wortpositionen bestimmt: die breiteste senkrechte Lücke
    im mittleren Seitendrittel, die kein Wort schneidet. Wörter, die die Lücke überqueren,
    gehören zu seitenbreiten Elementen (Titelseite, Überschriften). Lesereihenfolge: Seite ->
    horizontale Bänder zwischen seitenbreiten Elementen -> linke, dann rechte Spalte -> Grundlinie.
    """

    def __init__(self, p: LayoutParameter | None = None):
        self.p = p or LayoutParameter()

    # ---------------------------------------------------------------- Spalten
    def _spaltengrenze(self, w: pd.DataFrame, breite: float) -> tuple[float, float] | None:
        lo, hi = 0.30 * breite, 0.70 * breite
        raster = np.zeros(int(breite) + 2, dtype=int)
        for a, b in zip(w["x0"].to_numpy(), w["x1"].to_numpy()):
            raster[max(int(a), 0):min(int(np.ceil(b)), len(raster))] += 1
        frei = raster == 0
        beste, start = None, None
        for x in range(int(lo), int(hi) + 1):
            if frei[x] and start is None:
                start = x
            if (not frei[x] or x == int(hi)) and start is not None:
                ende = x if not frei[x] else x + 1
                if beste is None or ende - start > beste[1] - beste[0]:
                    beste = (start, ende)
                start = None
        return beste if beste and beste[1] - beste[0] >= 6 else None

    def zeilen(self, woerter: pd.DataFrame, seiten: pd.DataFrame) -> pd.DataFrame:
        p, ausgabe, marken, druck = self.p, [], {}, {}
        koerper_font = woerter["font"].value_counts().idxmax()          # Fließtextschrift des Protokolls
        koerper_familie = _familie(koerper_font)
        vorbereitet = []
        for s in seiten.itertuples(index=False):
            w = woerter.loc[woerter["seite"].eq(s.seite)]
            if w.empty:
                continue
            kopf = w["yb"] < p.kopf_anteil * s.hoehe
            rand = w["text"].str.match(RANDMARKE) & ((w["x1"] < 0.10 * s.breite) | (w["x0"] > 0.90 * s.breite))
            marken[s.seite] = {t.strip("()"): y for t, y in zip(w.loc[rand, "text"], w.loc[rand, "yb"])}
            druck[s.seite] = _druckseite(w.loc[kopf])
            koerper = w.loc[~kopf & ~rand & w["pt"].le(10.5)]
            vorbereitet.append((s, w, kopf, rand, self._spaltengrenze(koerper, s.breite)))
        # Satzspiegel ist protokollweit konstant: Seiten ohne saubere Lücke (Titelseite, Tabellen)
        # erhalten die Median-Spaltengrenze des Protokolls; Wörter über der Grenze sind seitenbreit.
        gefunden = [g for *_, g in vorbereitet if g is not None]
        standard = (tuple(np.median(np.array(gefunden), axis=0)) if gefunden else None)
        for s, w, kopf, rand, grenze in vorbereitet:
            if grenze is None or standard is None or grenze[1] < standard[0] or grenze[0] > standard[1]:
                grenze = standard
            mitte = None if grenze is None else (grenze[0] + grenze[1]) / 2
            luecke = 0.7 * (grenze[1] - grenze[0]) if grenze is not None else 1e9
            for teil, rolle in ((w.loc[kopf], "kopf"), (w.loc[rand], "rand"), (w.loc[~kopf & ~rand], "inhalt")):
                for spalte, gruppe in self._segmente(teil, mitte if rolle == "inhalt" else None, luecke):
                    ausgabe.append(self._zeile(gruppe, rolle, spalte, s))
        z = pd.DataFrame(ausgabe)
        z = self._lesereihenfolge(z)
        z = self._einzug(z)
        z = self._fundstelle(z, marken, druck)
        z["sans"] = z["font0"].map(_familie).ne(koerper_familie)
        return z

    def _segmente(self, w: pd.DataFrame, mitte: float | None, luecke: float):
        """Zeilen = Wörter gleicher Grundlinie; eine Zeile wird nur an einer Lücke über der Spaltenmitte
        (> 70 % der Spaltenlücke) geteilt – weite Wortabstände im Blocksatz trennen nicht. Ein Segment links/rechts der Spaltenmitte gehört zur Spalte 0/1, ein Segment über
        der Mitte ist seitenbreit (-1, z. B. zentrierte Titel)."""
        if w.empty:
            return
        rows = sorted(w.itertuples(index=False), key=lambda r: r.yb)
        zeilen, aktuell = [], [rows[0]]
        for vorher, r in zip(rows, rows[1:]):
            if r.yb - vorher.yb > self.p.grundlinie_tol:
                zeilen.append(aktuell); aktuell = []
            aktuell.append(r)
        zeilen.append(aktuell)
        for z in zeilen:
            z.sort(key=lambda r: r.x0)
            seg = [z[0]]
            for r in z[1:]:
                if (mitte is not None and r.x0 - seg[-1].x1 > luecke
                        and seg[-1].x1 <= mitte + 1 and r.x0 >= mitte - 1):
                    yield self._spalte(seg, mitte), seg
                    seg = []
                seg.append(r)
            yield self._spalte(seg, mitte), seg

    @staticmethod
    def _spalte(seg: list, mitte: float | None) -> int:
        if mitte is None:
            return -1
        x0, x1 = min(r.x0 for r in seg), max(r.x1 for r in seg)
        return 0 if x1 <= mitte + 1 else 1 if x0 >= mitte - 1 else -1

    def _zeile(self, woerter: list, rolle: str, spalte: int, s) -> dict:
        spans, vorher, luecke, segmente = [], None, 0.0, []
        for r in woerter:
            if vorher is not None:
                luecke = max(luecke, r.x0 - vorher.x1)
            if vorher is None or r.x0 - vorher.x1 > 14:          # Tabulatorlücke: neues Segment (Tabellen)
                segmente.append([round(r.x0, 1), r.text, _fett(r.font)])
            else:
                segmente[-1][1] += (" " if r.x0 - vorher.x1 > self.p.wortabstand * r.pt else "") + r.text
            sep = " " if vorher is not None and r.x0 - vorher.x1 > self.p.wortabstand * r.pt else ""
            fett, kursiv = _fett(r.font), _kursiv(r.font)
            if spans and spans[-1][1] == fett and spans[-1][2] == kursiv:
                spans[-1][0] += sep + r.text
            else:
                spans.append([sep + r.text, fett, kursiv])
            vorher = r
        text = "".join(t for t, _, _ in spans)
        n_fett = sum(len(r.text) for r in woerter if _fett(r.font))
        return {"seite": s.seite, "spalte": int(spalte), "rolle": rolle,
                "x0": min(r.x0 for r in woerter), "x1": max(r.x1 for r in woerter),
                "yb": float(np.median([r.yb for r in woerter])),
                "pt": Counter(r.pt for r in woerter).most_common(1)[0][0],
                "font0": woerter[0].font, "fett0": _fett(woerter[0].font),
                "fett_anteil": n_fett / max(sum(len(r.text) for r in woerter), 1),
                "kursiv0": _kursiv(woerter[0].font), "max_luecke": round(luecke, 1),
                "text": text, "spans": [(t, f, k) for t, f, k in spans],
                "segmente": [tuple(s) for s in segmente] if len(segmente) > 1 else None}

    @staticmethod
    def _lesereihenfolge(z: pd.DataFrame) -> pd.DataFrame:
        """Bänder: seitenbreite Zeilen (spalte -1) trennen; im Band links vor rechts."""
        teile = []
        for _, g in z.groupby("seite", sort=True):
            kopf, rand = g[g["rolle"].eq("kopf")], g[g["rolle"].eq("rand")]
            inh = g[g["rolle"].eq("inhalt")].copy()
            breit = inh.loc[inh["spalte"].eq(-1), "yb"].sort_values().to_numpy()
            inh["band"] = np.searchsorted(breit, inh["yb"].to_numpy(), side="left")
            inh.loc[inh["spalte"].eq(-1), "band"] = np.searchsorted(breit, inh.loc[inh["spalte"].eq(-1), "yb"].to_numpy(), side="left")
            inh["_o"] = inh["spalte"].map({-1: -1, 0: 0, 1: 1})
            inh = inh.sort_values(["band", "_o", "yb", "x0"]).drop(columns=["_o"])
            teile += [kopf.sort_values("yb").assign(band=-1), inh, rand.sort_values("yb").assign(band=-2)]
        z = pd.concat(teile, ignore_index=True)
        z.insert(0, "zeile", np.arange(len(z)))
        return z

    def _einzug(self, z: pd.DataFrame) -> pd.DataFrame:
        """Einzug relativ zum Satzspiegel: häufigster Zeilenanfang/-schluss je Spalte und Seitenparität
        im ganzen Protokoll (Blocksatz: die meisten Zeilen beginnen/enden am Spaltenrand). So bleiben
        Seiten mit überwiegend eingezogenen Zeilen (Inhaltsverzeichnis, Tagesordnung) korrekt."""
        inh = z["rolle"].eq("inhalt") & z["spalte"].ge(0)
        g = z.loc[inh].assign(par=z["seite"] % 2, xl=z["x0"].round(0), xr=z["x1"].round(0))
        links = g.groupby(["par", "spalte"])["xl"].agg(lambda s: s.mode().iloc[0])
        rechts = g.groupby(["par", "spalte"])["xr"].agg(lambda s: s.mode().iloc[0])
        key = list(zip(z["seite"] % 2, z["spalte"]))
        z["einzug_l"] = [x - links.get(k, x) for x, k in zip(z["x0"], key)]
        z["einzug_r"] = [rechts.get(k, x) - x for x, k in zip(z["x1"], key)]
        z.loc[~inh, ["einzug_l", "einzug_r"]] = 0.0
        z["eingezogen"] = z["einzug_l"] > self.p.einzug
        return z

    @staticmethod
    def _fundstelle(z: pd.DataFrame, marken: dict, druck: dict) -> pd.DataFrame:
        """Druckseite aus dem Kolumnentitel; Quadrant A–D aus den Randmarken: (B)/(D) markieren den
        Beginn der unteren Spaltenhälfte. Fehlt eine Marke, gilt der Median des Protokolls
        (Satzspiegel ist seitenkonstant)."""
        median = {q: float(np.median([m[q] for m in marken.values() if q in m]))
                  for q in "BD" if any(q in m for m in marken.values())}
        def quadrant(seite, spalte, yb):
            m = marken.get(seite, {})
            if spalte == 0:
                return "A" if yb < m.get("B", median.get("B", 1e9)) - 2 else "B"
            if spalte == 1:
                return "C" if yb < m.get("D", median.get("D", 1e9)) - 2 else "D"
            return None
        z["druckseite"] = z["seite"].map(druck).astype("Int64")
        z["quadrant"] = [quadrant(s, sp, y) for s, sp, y in zip(z["seite"], z["spalte"], z["yb"])]
        return z


MONATE = ("Januar|Februar|März|April|Mai|Juni|Juli|August|September|Oktober|November|Dezember")


def _druckseite(kopf: pd.DataFrame) -> int | None:
    """Seitenzahl im Kolumnentitel: vier- bis sechsstellige Zahl, die nicht die Jahreszahl des Datums
    ist. Gesperrt gesetzte Ziffern (eigene Wörter ohne Abstand) werden vorher zusammengeführt."""
    tokens, letzte = [], None
    for r in kopf.sort_values(["yb", "x0"]).itertuples(index=False):
        if tokens and letzte is not None and r.x0 - letzte.x1 < 1.0 and abs(r.yb - letzte.yb) < 1:
            tokens[-1] += r.text
        else:
            tokens.append(r.text)
        letzte = r
    for i, t in enumerate(tokens):
        if re.fullmatch(r"\d{4,6}", t) and not (i and re.fullmatch(MONATE, tokens[i - 1])):
            return int(t)
    return None


def _familie(font: str) -> str:
    """Schriftfamilie ohne Schnitt: 'TimesNewRomanPS-BoldMT' -> 'TimesNewRoman', 'Arial-BoldMT' -> 'Arial'."""
    return re.sub(r"(?:PS)?(?:MT)?$", "", re.split(r"[-,]", str(font).split("+")[-1])[0])


def _fett(font: str) -> bool:
    return bool(re.search(r"Bold|Black|Heavy|Semibold", str(font), re.I))


def _kursiv(font: str) -> bool:
    return bool(re.search(r"Italic|Oblique", str(font), re.I))

In [9]:
# ---------------------------------------------------------------------
# Demonstration Layout: eine Seite mit Spalten, Einzügen, Zeilentypen
# ---------------------------------------------------------------------
_demo = VERARBEITUNG[0]
_w, _s = PdfWoerter(PDF_BACKEND).lese(PDF_DIR / f"20_{_demo}.pdf")
_z = Layout().zeilen(_w, _s)
_seite = int(_z.loc[_z["text"].str.contains("Beginn:"), "seite"].iloc[0]) + 1
display(_z.loc[_z["seite"].eq(_seite), ["spalte", "rolle", "einzug_l", "einzug_r", "pt", "font0", "druckseite",
                                        "quadrant", "text"]].head(25))

,spalte,rolle,einzug_l,einzug_r,pt,font0,druckseite,quadrant,text
823,-1,kopf,0.000000,0.000000,8.5,TimesNewRomanPSMT,18484,None,18484 Deutscher Bundestag – 20. Wahlperiode – ...
824,-1,kopf,0.000000,0.000000,8.0,TimesNewRomanPS-BoldMT,18484,None,Präsidentin Bärbel Bas
825,0,inhalt,-0.220501,0.477722,10.0,TimesNewRomanPSMT,18484,A,tung eines großen Europäers und unterstreicht ...
826,0,inhalt,-0.220501,0.454834,10.0,TimesNewRomanPSMT,18484,A,dienste von Wolfgang Schäuble für die deutsch-...
827,0,inhalt,-0.220501,152.249939,10.0,TimesNewRomanPSMT,18484,A,sche Freundschaft.
828,0,inhalt,9.762024,0.412964,10.0,TimesNewRomanPSMT,18484,A,"Liebe Kolleginnen und Kollegen, ich bitte Sie,..."
829,0,inhalt,-0.220501,0.433929,10.0,TimesNewRomanPSMT,18484,A,Gedenken an einen großen Demokraten und Staats...
830,0,inhalt,-0.220501,12.122009,10.0,TimesNewRomanPSMT,18484,A,für eine Gedenkminute von Ihren Plätzen zu erh...
831,0,inhalt,48.646057,49.343262,10.0,TimesNewRomanPSMT,18484,A,(Die Anwesenden erheben sich)
832,0,inhalt,-0.220497,152.213104,10.0,TimesNewRomanPSMT,18484,A,– Ich danke Ihnen.


## [↩](#toc) 5) Dokumentmodell: Inhaltsverzeichnis, Beiträge, Debatten<a class="anchor" id="k5"></a>

**Bereiche.** `Beginn: 13.00 Uhr` und `(Schluss: 21.09 Uhr)` werden nur als **ganze Zeile** akzeptiert („zum Schluss:“ im Fließtext passt nicht). Davor liegt das Inhaltsverzeichnis (IV), danach die Anlagen.

**Inhaltsverzeichnis (Soll).** Einträge enden mit dem Seitenverweis „… 25820 A“. Fette Zeilen ohne Verweis sind Blocküberschriften (Tagesordnungspunkt, Zusatzpunkt, Mündliche Frage, Anlage); „in Verbindung mit“ verbindet Blöcke zu **einer Debatte**. Einträge mit fettem Titel sind Gegenstände (Vorlagen); Einträge der Form „Name (Fraktion)“ bzw. „Name, Funktion“ sind Redner. Drucksachen-Zeilen werden dem vorherigen Eintrag angehängt.

**Sprecherköpfe (Ist).** Das Protokoll setzt den Namen im Kopf **serifenlos fett**, Fraktion und Funktion in der Grundschrift. Ein Kopf ist eine Zeile, deren erstes Wort in dieser Kopfschrift steht; er reicht bis zum ersten Doppelpunkt (höchstens drei Zeilen, z. B. „Dr. Robert Habeck, Bundesminister für Wirtschaft ⏎ und Klimaschutz:“) und muss der Kopfgrammatik folgen. Sätze der Sitzungsleitung, die auf „Name (SPD):“ enden, oder Zitate erzeugen daher keine Köpfe.

**Zeilentypen im Beitrag.** `zwischenruf`: eingezogener Klammerblock, die Klammertiefe wird über Zeilen-, Spalten- und Seitenwechsel geführt · `einschub`: tief eingezogener Block ohne Klammer (Tagesordnungstext, Zitat) · `klein`: < 9 pt (Fußnoten, abgedruckte Fragen) · `tabelle`: Tabulatorlücken > 14 pt (Namenslisten) · `rede`: Wortlaut.

**Soll-Ist-Verknüpfung.** IV-Redner ↔ Beitrag über (Druckseite, Quadrant, Nachname), Toleranz ±1 Quadrant. Nicht verknüpfte IV-Einträge sind entweder schriftliche Antworten in Anlagen (ohne Sprecherkopf, korrekt) oder Fehler – die Abdeckung wird je Protokoll ausgewiesen.

**Debatte, Beitragsart, Redeeinheit.**
- Debatte eines Beitrags: aus seinem IV-Eintrag; nicht verzeichnete Beiträge erben die laufende Debatte; ein Beitrag der Sitzungsleitung mit Tagesordnungstext gehört zur **aufgerufenen** Debatte (sein Text davor zur Vordebatte – dort stehen die Abstimmungen). Zu Protokoll gegebene Reden in Anlagen werden über den Verweis „(Tagesordnungspunkt 5)“ ihrer Debatte zugeordnet.
- Beitragsart: Block „Zur Geschäftsordnung“, Debattentyp (Fragestunde/Regierungsbefragung → Frage/Antwort), Anlage (zu Protokoll / Erklärung) oder der Hinweis der Sitzungsleitung in den letzten Sätzen vor dem Beitrag (Kurzintervention, Erwiderung, Zwischenfrage …, Person muss genannt sein).
- **Redeeinheit**: ein IV-verzeichneter Beitrag plus alle nicht verzeichneten Fortsetzungen derselben Person in derselben Debatte. Die Hauptrede bleibt über Unterbrechungen offen – über Ermahnungen der Sitzungsleitung, nicht verzeichnete Zwischenbeiträge und auch über Zwischenfragen, die das IV selbst verzeichnet (erkannt am Hinweis „Ich erlaube die Zwischenfrage von Kollege Rief“). Nicht verzeichnete Beiträge anderer Personen sind Zwischenbeiträge.

**Personenstamm und Partei (Evidenzprinzip).** Person = Name ohne akademische Titel (Ortszusatz nur bei Mehrdeutigkeit). Partei in dieser Reihenfolge: Mehrheit der Fraktionsangaben in ihren Köpfen → **Namenslisten namentlicher Abstimmungen** (Fraktion als fette Zwischenüberschrift; liefert die Partei auch für Regierungsmitglieder mit Mandat) → benannte Zwischenrufe „Name [Fraktion]“ → „nicht im PDF belegt“. Für Gruppierungen gilt die Rollenregel: Wer als Regierungsmitglied spricht, zählt zur Gruppe *Bundesregierung* (`Partei_Gruppe`); die Partei der Person bleibt erhalten.

In [10]:
# =====================================================================
# Text- und Namenswerkzeuge (Pipeline; die Evaluation hat eigene Normierer)
# =====================================================================
FRAKTIONEN = {
    "spd": "SPD", "cdu/csu": "CDU/CSU", "cdu": "CDU/CSU", "csu": "CDU/CSU", "afd": "AfD", "fdp": "FDP",
    "bündnis90/diegrünen": "BÜNDNIS 90/DIE GRÜNEN", "diegrünen": "BÜNDNIS 90/DIE GRÜNEN",
    "grüne": "BÜNDNIS 90/DIE GRÜNEN", "dielinke": "Die Linke", "linke": "Die Linke",
    "bsw": "BSW", "fraktionslos": "fraktionslos",
}
LAENDER = ("Baden-Württemberg|Bayern|Berlin|Brandenburg|Bremen|Hamburg|Hessen|Mecklenburg-Vorpommern|"
           "Niedersachsen|Nordrhein-Westfalen|Rheinland-Pfalz|Saarland|Sachsen|Sachsen-Anhalt|"
           "Schleswig-Holstein|Thüringen")
TITEL = re.compile(r"\b(?:Dr|Prof|Professor|h\.\s?c|med|rer\.\s?nat|phil|jur|Ing|Dipl)\.?(?=\s|$)\s*", re.I)


def txt(x) -> str:
    """Leere/fehlende Werte (None, NaN, pd.NA) als '' – robust für pandas 2 und 3."""
    return x if isinstance(x, str) else ""


def kanon_fraktion(text: str | None) -> str | None:
    if not txt(text):
        return None
    return FRAKTIONEN.get(re.sub(r"[\s.\-–]+", "", str(text)).casefold())


def personen_schluessel(name: str) -> str:
    """Vergleichsschlüssel einer Person: ohne akademische Titel, ohne Klammerzusatz, ohne Akzente,
    kleingeschrieben. (Ortszusätze werden im Personenstamm gesondert behandelt.)"""
    n = re.sub(r"\s*\([^()]*\)", "", str(name))
    n = TITEL.sub("", n)
    n = unicodedata.normalize("NFKD", n)
    n = "".join(c for c in n if not unicodedata.combining(c))
    return " ".join(re.sub(r"[^\w\s-]", " ", n).casefold().split())


def nachname(name: str) -> str:
    teile = personen_schluessel(name).split()
    return teile[-1] if teile else ""


class Silbentrennung:
    """Trennstrich am Zeilenende auflösen. Entscheidung je Fall über das Korpusvokabular:
    belegte Wortform -> zusammenziehen; nur Bindestrichform belegt -> Bindestrich behalten;
    rechter Teil 'und/oder/bzw./sowie' -> Koordination ('Wirtschafts- und'); rechter Teil
    beginnt groß -> Bindestrich ('AfD-|Fraktion'); sonst zusammenziehen."""

    KOORDINATION = {"und", "oder", "bzw", "bzw.", "sowie", "als", "wie", "bis"}

    def __init__(self, vokabular: Counter | None = None):
        self.vok = vokabular or Counter()
        self.stat = Counter()

    @classmethod
    def aus_zeilen(cls, texte: Iterable[str]) -> "Silbentrennung":
        vok = Counter()
        for t in texte:
            woerter = t.split()
            for w in woerter[:-1] if woerter and woerter[-1].endswith("-") else woerter:
                vok[w.strip(".,;:!?()„“\"'").casefold()] += 1
        return cls(vok)

    def verbinde(self, links: str, rechts: str) -> tuple[str, str]:
        """links endet mit '-'; gibt (verbundenes Wort, Grund) zurück."""
        stamm = links[:-1]
        r_wort = re.match(r"[\wÄÖÜäöüß]+", rechts)
        r = r_wort.group(0) if r_wort else rechts
        if r.casefold().rstrip(".") in self.KOORDINATION:
            return links + " " + rechts, "Koordination"
        zusammen = (stamm + r).casefold()
        binde = (stamm + "-" + r).casefold()
        if self.vok.get(zusammen, 0) > self.vok.get(binde, 0):
            return stamm + rechts, "Vokabular_Wort"
        if self.vok.get(binde, 0) > 0:
            return links + rechts, "Vokabular_Bindestrich"
        if r[:1].isupper() or r[:1].isdigit() or stamm[-1:].isdigit():
            return links + rechts, "Großschreibung"
        return stamm + rechts, "Zusammenziehen"

    def fuege(self, zeilen: Iterable[str]) -> str:
        out = ""
        for z in zeilen:
            z = z.strip()
            if not z:
                continue
            if out.endswith("-") and len(out) > 1 and out[-2].isalpha() and z[:1].isalnum():
                kopf, _, letztes = out.rpartition(" ")
                erstes, _, rest = z.partition(" ")
                wort, grund = self.verbinde(letztes, erstes)
                self.stat[grund] += 1
                out = (kopf + " " if kopf else "") + wort + (" " + rest if rest else "")
            else:
                out = (out + " " + z) if out else z
        return out


SATZENDE = re.compile(r"(?<=[.!?…])[\"“”»)]?\s+(?=[„\"»(]?[A-ZÄÖÜ0-9])")
ABKUERZUNG = re.compile(r"(?:\b(?:Dr|Prof|Abs|Nr|Art|bzw|vgl|ca|Mio|Mrd|z\.\s?B|u\.\s?a|d\.\s?h|i\.\s?d\.\s?R|"
                        r"St|Std|Tsd|Jan|Feb|Okt|Nov|Dez|Sept|ggf|evtl|inkl|sog|usw|Ziff|lit|Buchst)\.|\b\d{1,2}\.)$")


def saetze(text: str) -> list[str]:
    """Regelbasierte Satztrennung (deutsche Abkürzungen und Ordnungszahlen bleiben im Satz)."""
    teile, puffer = [], ""
    for stueck in SATZENDE.split(text):
        puffer = f"{puffer} {stueck}".strip() if puffer else stueck
        if not ABKUERZUNG.search(puffer):
            teile.append(puffer)
            puffer = ""
    if puffer:
        teile.append(puffer)
    return [t for t in teile if t.strip()]

In [11]:
# =====================================================================
# Dokumentmodell: Bereiche, Inhaltsverzeichnis (Soll), Beiträge (Ist), Verknüpfung
# =====================================================================
BEGINN = re.compile(r"^Beginn:\s*\d{1,2}[.:]\d{2}\s*Uhr$")
SCHLUSS = re.compile(r"^\(Schluss:\s*\d{1,2}[.:]\d{2}\s*Uhr\)$")
INHALT = re.compile(r"^I\s*n\s*h\s*a\s*l\s*t\s*:$")
IMPRESSUM = re.compile(r"^Gesamtherstellung:")

# Inhaltsverzeichnis: Seitenverweis "… 25820 A" am Zeilenende, Blocküberschriften (fett),
# Rollenmarken der Fragestunde. Die Bezeichnungen sind das feste Vokabular der amtlichen Gliederung.
SEITENVERWEIS = re.compile(r"(?:\s*\.)*\s*(\d{4,6})\s+([A-D])$")
IV_BLOCK = re.compile(r"^(?P<art>Tagesordnungspunkte?|Zusatzpunkte?|Mündliche Frage|Dringliche Frage|"
                      r"Frage|Anlage)\s+(?P<label>[\w.]+(?:\s*(?:,|und|bis)\s*[\w.]+)*)"
                      r"(?:\s*\((?:Fortsetzung)\))?\s*:?$")
IV_ROLLE = re.compile(r"^(Antwort|Zusatzfragen?)$")
DRUCKSACHE = re.compile(r"\b(?:Drucksachen?|Bundestagsdrucksachen?)\s+((?:\d{1,2}/\s?\d{2,6}(?:\s*\(?(?:neu|Nr\.\s*[\d.]+)\)?)?"
                        r"(?:\s*(?:,|und|sowie)\s*)?)+)")
DS_NUMMER = re.compile(r"\b(\d{2})/\s?(\d{2,6})\b")

# Sprecherkopf: "Name [(Zusatz)] [(Fraktion)] [, Funktion [(Land)]]:" – geprüft am erkannten Kopfblock
KOPF = re.compile(r"^(?P<name>[^():,]{2,90}?)\s*(?P<klammern>(?:\([^()]{1,60}\)\s*)*)"
                  r"(?:,\s*(?P<funktion>[^:]{2,250}?))?\s*:$")
SITZUNGSLEITUNG = re.compile(r"^(?P<amt>(?:Alters|Vize)?[Pp]räsident(?:in)?)\s+(?P<name>.+)$")
REGIERUNG = re.compile(r"Bundes(?:kanzler|minister)|Staatsminister|Staatssekretär|Beauftragte[r]? der Bundesregierung", re.I)
LAND = re.compile(rf"\((?:{LAENDER})\)|Ministerpräsident|Senator|Landesminister|Bürgermeister", re.I)


class Bereiche:
    """Gliedert die Zeilen in titel/iv/sitzung/anlagen/impressum über ganze Zeilen mit fester
    Form ('Beginn: 13.00 Uhr', '(Schluss: 21.09 Uhr)'); Fließtext wie 'zum Schluss:' passt nicht."""

    @staticmethod
    def markiere(z: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
        hinweise, inh = [], z["rolle"].eq("inhalt")
        def erste(rx, ab=-1):
            treffer = z.index[inh & z["text"].str.strip().str.match(rx) & (z["zeile"] > ab)]
            return int(z.loc[treffer[0], "zeile"]) if len(treffer) else None
        i_inhalt, i_beginn = erste(INHALT), erste(BEGINN)
        i_schluss = erste(SCHLUSS, i_beginn or -1)
        i_impressum = erste(IMPRESSUM, i_schluss or -1)
        if i_beginn is None:
            raise ValueError("Sitzungsbeginn ('Beginn: … Uhr') nicht gefunden")
        if i_schluss is None:
            hinweise.append("Sitzungsende ('(Schluss: … Uhr)') nicht gefunden – Anlagen nicht abgegrenzt")
            i_schluss = int(z["zeile"].max())
        i_impressum = i_impressum if i_impressum is not None else int(z["zeile"].max()) + 1
        zl = z["zeile"]
        z["bereich"] = np.select(
            [zl < (i_inhalt or 0), zl < i_beginn, zl <= i_schluss, zl < i_impressum],
            ["titel", "iv", "sitzung", "anlagen"], "impressum")
        z.loc[~inh, "bereich"] = "layout"                 # Kolumnentitel, Randmarken
        return z, hinweise


class Inhaltsverzeichnis:
    """Parst das Inhaltsverzeichnis (IV) in Blöcke (TOP/ZP/Frage/Anlage), Gegenstände (Vorlagen,
    mit fettem Titel) und Personeneinträge (Name, Fraktion/Funktion, Druckseite, Quadrant).
    'in Verbindung mit' verbindet Blöcke zu einer Debatte. Das IV ist das Soll der Sitzung."""

    def __init__(self, sitzung: int):
        self.sitzung = sitzung

    def parse(self, z: pd.DataFrame, trenner: "Silbentrennung") -> pd.DataFrame:
        iv = z.loc[z["bereich"].eq("iv") & ~z["text"].str.match(INHALT)]
        titel = iv.index[iv["pt"].ge(12)]                     # Sitzungstitel ('146. Sitzung') beendet das IV
        if len(titel):
            iv = iv.loc[:titel[0] - 1]
        eintraege, offen, block, debatte, rolle_hint = [], [], None, 0, None
        verbinden, nr_block, block_ober = False, 0, None

        def schliesse(zeilen, seite=None, quadrant=None, art=None):
            if not zeilen:
                return
            text = trenner.fuege([r.text for r in zeilen])
            text = re.sub(r"(?:\s*\.){3,}\s*$", "", SEITENVERWEIS.sub("", text) if seite else text).strip()
            fett = trenner.fuege([t for r in zeilen for t, f, _ in r.spans if f]).strip(" .:")
            kursiv = all(r.kursiv0 for r in zeilen)
            eintraege.append({
                "Sitzung": self.sitzung, "IV_Nr": len(eintraege) + 1, "Block": block_ober, "Unterblock": block,
                "Block_Nr": nr_block,
                "Debatte_Nr": debatte, "Art": art, "Rolle_Hinweis": rolle_hint, "Text": text,
                "Titel_fett": fett or None, "Kursiv": kursiv, "Druckseite": seite, "Quadrant": quadrant,
                "Drucksachen": [], "Zeilen": [int(r.zeile) for r in zeilen]})

        for r in iv.itertuples(index=False):
            t = r.text.strip()
            if not t:
                continue
            m_ref = SEITENVERWEIS.search(t)
            if r.fett_anteil > 0.9 and not m_ref and (IV_BLOCK.match(t) or (t.endswith(":") and len(t.split()) <= 5)):
                schliesse(offen, art="frage" if offen else None); offen = []
                m = IV_BLOCK.match(t)
                block = (f"{m.group('art')} {m.group('label')}" if m else t.rstrip(":")).strip()
                block = " ".join(block.split())
                nr_block += 1
                unterblock = bool(m) and m.group("art").endswith("Frage") and block_ober is not None
                if not verbinden and not unterblock:
                    debatte += 1
                if not unterblock:
                    block_ober = block
                verbinden, rolle_hint = False, None
                continue
            if re.fullmatch(r"in Verbindung mit", t, re.I):
                schliesse(offen); offen = []
                verbinden = True
                continue
            if IV_ROLLE.match(t):
                schliesse(offen, art="frage"); offen = []
                rolle_hint = IV_ROLLE.match(t).group(1)
                continue
            if not offen and eintraege and re.match(r"^(?:Drucksachen?\b|Nr\.|\d{1,2}/?\s?\d|\d{3,6}\b|20/)", t):
                eintraege[-1]["Anhang"] = (eintraege[-1].get("Anhang", "") + " " + t).strip()
                continue
            offen.append(r)
            if m_ref:
                schliesse(offen, int(m_ref.group(1)), m_ref.group(2)); offen = []
        schliesse(offen)
        df = pd.DataFrame(eintraege)
        if df.empty:
            raise ValueError("Inhaltsverzeichnis leer")
        if "Anhang" not in df:
            df["Anhang"] = None
        df["Drucksachen"] = [sorted({f"{a}/{b}" for a, b in DS_NUMMER.findall(
            " ".join(x for x in (t, an) if isinstance(x, str)))}) for t, an in zip(df["Text"], df["Anhang"])]
        df["Art"] = [a if isinstance(a, str) else self._art(t, f) for a, t, f in zip(df["Art"], df["Text"], df["Titel_fett"])]
        df["Name"] = [self._name(t) if a == "person" else None for a, t in zip(df["Art"], df["Text"])]
        df.insert(0, "IV_ID", [f"20/{self.sitzung}:IV{i:04d}" for i in df["IV_Nr"]])
        return df

    VORLAGE = re.compile(r"\b(?:Antrag|Anträge|Entwurfs?|Gesetzentwurf|Beschlussempfehlung|Bericht|Unterrichtung|"
                         r"Wahl|Beratung|Regierungserklärung|Vereinbarte Debatte|Aktuelle Stunde)\b")

    @classmethod
    def _art(cls, text: str, fett) -> str:
        if isinstance(fett, str) and len(fett) > 3:
            return "gegenstand"
        kern = re.split(r"\s*\(|,\s*", text, maxsplit=1)[0]
        name_form = re.fullmatch(r"(?:(?:Dr|Prof)\.\s*(?:h\.\s?c\.\s*|med\.\s*)?)*[A-ZÄÖÜ][^\s,()]*"
                                 r"(?:\s+(?:[A-ZÄÖÜ][^\s,()]*|von|van|de|der|den|zu|di|da|del|vom|ten|y|al|bin))+", kern)
        if name_form and ("(" in text or "," in text):
            return "person"
        if cls.VORLAGE.search(text) and not re.match(r"Erklärung|Zu Protokoll", text):
            return "gegenstand"
        return "sonstiges"

    @staticmethod
    def _name(text: str) -> str:
        return re.split(r"\s*\(|,\s*", text, maxsplit=1)[0].strip()


@dataclass
class Kopf:
    name: str
    rolle: str
    fraktion: str | None
    funktion: str | None
    zusatz: str | None
    amt: str | None


def parse_kopf(kopftext: str) -> Kopf | None:
    """Zerlegt 'Dr. X Y (Ort) (SPD):', 'X Y, Bundesminister …:', 'Vizepräsidentin X Y:'."""
    m = KOPF.match(" ".join(kopftext.split()))
    if not m:
        return None
    name, funktion = m.group("name").strip(), (m.group("funktion") or "").strip() or None
    klammern = re.findall(r"\(([^()]+)\)", m.group("klammern") or "")
    fraktion = next((kanon_fraktion(k) for k in klammern if kanon_fraktion(k)), None)
    zusatz = next((k for k in klammern if not kanon_fraktion(k)), None)
    ms = SITZUNGSLEITUNG.match(name)
    if ms and not funktion and not klammern:
        return Kopf(ms.group("name"), "Sitzungsleitung", None, None, None, ms.group("amt"))
    if len(name.split()) > 7 or re.search(r"\d", name):
        return None
    if fraktion:
        rolle = "Abgeordnete"
    elif funktion and LAND.search(funktion + " " + " ".join(f"({k})" for k in klammern)):
        rolle = "Länder"
    elif funktion and REGIERUNG.search(funktion):
        rolle = "Bundesregierung"
    else:
        rolle = "Sonstige"
    return Kopf(name, rolle, fraktion, funktion, zusatz, None)


class Beitraege:
    """Erkennt Sprecherköpfe typografisch und klassifiziert jede Folgezeile.

    Kopf: Zeile, deren erstes Wort in der Kopfschrift steht (fett und andere Schriftfamilie als der
    Fließtext, im Protokoll serifenlos fett). Der Kopf reicht bis zum ersten ':' (höchstens 3 Zeilen).
    Zeilentypen im Beitrag:
      zwischenruf  – zentrierter Klammerblock '( … )' (Einzug links ≈ rechts); Klammertiefe über Zeilen/Spalten/Seiten
      einschub     – tief eingezogener Block ohne Klammer (Tagesordnungstext, Zitat, Frage im Wortlaut)
      klein        – kleinerer Schriftgrad (< 9 pt: Fußnoten, abgedruckte Fragen)
      tabelle      – Zeile mit Tabulatorlücken > 14 pt (Namenslisten namentlicher Abstimmungen)
      rede         – übriger Text (Wortlaut des Beitrags)
    """

    def __init__(self, sitzung: int, tief: float = 18.0):
        self.sitzung, self.tief = sitzung, tief

    def parse(self, z: pd.DataFrame, trenner: "Silbentrennung") -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        z = z.loc[z["bereich"].isin(["sitzung", "anlagen"])].reset_index(drop=True)
        koepfe = self._koepfe(z)
        z["beitrag"] = -1
        z["typ"] = "rahmen"
        beitraege, rufe = [], []
        grenzen = [k["start"] for k in koepfe] + [len(z)]
        for i, k in enumerate(koepfe):
            von, bis = k["start"], grenzen[i + 1]
            z.loc[von:bis - 1, "beitrag"] = i
            z.loc[von:k["ende"], "typ"] = "kopf"
            typen, ruf_bloecke = self._zeilentypen(z.loc[k["ende"] + 1:bis - 1], k["rest"])
            z.loc[k["ende"] + 1:bis - 1, "typ"] = typen
            rede_zeilen = z.loc[k["ende"] + 1:bis - 1].loc[[t == "rede" for t in typen]]
            text = trenner.fuege(([k["rest"]] if k["rest"] else []) + rede_zeilen["text"].tolist())
            einschub = trenner.fuege(z.loc[k["ende"] + 1:bis - 1].loc[[t == "einschub" for t in typen], "text"])
            erster_einschub = next((j for j, t in enumerate(typen) if t == "einschub"), len(typen))
            vor = trenner.fuege(([k["rest"]] if k["rest"] else []) + [
                t for t, ty in zip(z.loc[k["ende"] + 1:bis - 1, "text"], typen[:erster_einschub]) if ty == "rede"])
            absaetze = int((rede_zeilen["einzug_l"].between(5, 16)).sum()) + 1
            erste = z.loc[von]
            beitraege.append({
                "Sitzung": self.sitzung, "Beitrag_Nr": i + 1, "Bereich": erste["bereich"],
                "Kopftext": k["kopftext"], "Name": k["kopf"].name, "Rolle": k["kopf"].rolle,
                "Fraktion_Kopf": k["kopf"].fraktion, "Funktion": k["kopf"].funktion,
                "Zusatz": k["kopf"].zusatz, "Amt": k["kopf"].amt,
                "Druckseite": erste["druckseite"], "Quadrant": erste["quadrant"], "PDF_Seite": int(erste["seite"]),
                "Zeile_von": int(erste["zeile"]), "Zeile_bis": int(z.loc[bis - 1, "zeile"]),
                "Text": text, "Einschub": einschub, "Text_vor_Einschub": vor, "Absaetze": absaetze,
                "Woerter": len(text.split()), "Zeichen": len(text),
                "Zwischenrufe": len(ruf_bloecke)})
            for j, blk in enumerate(ruf_bloecke):
                rufe.append({"Sitzung": self.sitzung, "Beitrag_Nr": i + 1, "Ruf_Nr": j + 1,
                             "Text": trenner.fuege(blk)})
        df = pd.DataFrame(beitraege)
        df.insert(0, "Beitrag_ID", [f"20/{self.sitzung}:B{n:04d}" for n in df["Beitrag_Nr"]])
        return df, pd.DataFrame(rufe, columns=["Sitzung", "Beitrag_Nr", "Ruf_Nr", "Text"]), z

    def _koepfe(self, z: pd.DataFrame) -> list[dict]:
        koepfe, i, n = [], 0, len(z)
        kandidat = (z["fett0"] & z["sans"] & z["pt"].between(8.5, 10.5) & z["rolle"].eq("inhalt")).to_numpy()
        texte = z["text"].tolist()
        while i < n:
            if not kandidat[i] or texte[i].startswith("("):
                i += 1
                continue
            kopf, j = texte[i], i
            while ":" not in kopf and j + 1 < min(n, i + 3):
                j += 1
                kopf = (kopf[:-1] if kopf.endswith("-") and texte[j][:1].islower() else kopf + " ") + texte[j]
            if ":" not in kopf:
                i += 1
                continue
            kopftext, _, rest = kopf.partition(":")
            k = parse_kopf(kopftext + ":")
            if k is None:
                i += 1
                continue
            koepfe.append({"start": i, "ende": j, "kopftext": kopftext.strip(), "rest": rest.strip(), "kopf": k})
            i = j + 1
        return koepfe

    def _zeilentypen(self, zeilen: pd.DataFrame, rest: str) -> tuple[list[str], list[list[str]]]:
        typen, bloecke, tiefe, puffer = [], [], 0, []
        for r in zeilen.itertuples(index=False):
            t = r.text.strip()
            eingerueckt = r.einzug_l > 3.5
            if tiefe > 0 and (eingerueckt or r.einzug_r > 3.5):
                puffer.append(t)
                tiefe += t.count("(") - t.count(")")
                typen.append("zwischenruf")
                if tiefe <= 0:
                    bloecke.append(puffer); puffer, tiefe = [], 0
                continue
            if tiefe > 0:                              # Block endet ohne schließende Klammer
                bloecke.append(puffer); puffer, tiefe = [], 0
            if t.startswith("(") and eingerueckt and abs(r.einzug_l - r.einzug_r) <= 6:   # zentrierter Klammerblock
                tiefe = t.count("(") - t.count(")")
                if tiefe > 0:
                    puffer = [t]
                else:
                    bloecke.append([t])
                typen.append("zwischenruf")
                continue
            if r.pt < 9.0:
                typen.append("klein")
            elif r.max_luecke > 14:
                typen.append("tabelle")
            elif r.einzug_l >= self.tief:
                typen.append("einschub")
            else:
                typen.append("rede")
        if puffer:
            bloecke.append(puffer)
        return typen, bloecke


class Verknuepfung:
    """Soll-Ist-Abgleich: IV-Personeneintrag <-> Beitrag über (Druckseite, Quadrant, Nachname).
    Der Quadrant darf um eine Position abweichen (Kopf am Quadrantenrand), dann wird der
    Nachname zwingend. Ergebnis: Beitrag -> IV_ID, Block, Debatte, Rollenhinweis."""

    @staticmethod
    def verknuepfe(iv: pd.DataFrame, b: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        personen = iv.loc[iv["Art"].eq("person")].copy()
        index = defaultdict(list)
        for r in b.itertuples(index=False):
            if r.Rolle != "Sitzungsleitung" and pd.notna(r.Druckseite):
                index[(int(r.Druckseite), r.Quadrant)].append(r)
        def pos(seite, q):
            return int(seite) * 4 + "ABCD".index(q)
        vergeben, links = set(), []
        for p in personen.itertuples(index=False):
            if pd.isna(p.Druckseite):
                continue
            nn = nachname(p.Name)
            kandidaten = []
            for d in (0, 1, -1, 2):
                k = pos(p.Druckseite, p.Quadrant) + d
                for r in index.get((k // 4, "ABCD"[k % 4]), []):
                    if r.Beitrag_ID not in vergeben and nachname(r.Name) == nn:
                        kandidaten.append((abs(d), r.Zeile_von, r))
                if kandidaten:
                    break
            if kandidaten:
                _, _, r = min(kandidaten, key=lambda x: (x[0], x[1]))
                vergeben.add(r.Beitrag_ID)
                links.append({"IV_ID": p.IV_ID, "Beitrag_ID": r.Beitrag_ID})
        links = pd.DataFrame(links, columns=["IV_ID", "Beitrag_ID"])
        b = b.merge(links, on="Beitrag_ID", how="left").merge(
            iv[["IV_ID", "Block", "Block_Nr", "Debatte_Nr", "Rolle_Hinweis", "Kursiv"]], on="IV_ID", how="left")
        personen["verknuepft"] = personen["IV_ID"].isin(links["IV_ID"])
        return b, personen


class Abstimmungslisten:
    """Namenslisten namentlicher Abstimmungen ('Endgültiges Ergebnis' … 'Ja' / Fraktion / Namen) als
    zusätzliche PDF-Quelle der Fraktionszugehörigkeit – auch für Regierungsmitglieder mit Mandat, deren
    Partei im Sprecherkopf fehlt. Tabellenzeilen werden an Tabulatorlücken in Segmente geteilt; Segmente
    werden je Teilspalte (x-Position) von oben nach unten gelesen; fette Segmente setzen Stimme/Fraktion."""

    MARKE = re.compile(r"Endgültiges Ergebnis|Abgegebene Stimmen")
    STIMME = {"ja": "Ja", "nein": "Nein", "enthalten": "Enthalten"}
    NAME = re.compile(r"^(?:(?:Dr|Prof)\.\s)*[A-ZÄÖÜ][\w'’.-]+(?:\s(?:von|van|de|der|zu|di|da|del|vom|ten|[A-ZÄÖÜ][\w'’.-]+)){1,4}"
                      r"(?:\s\([^()\d]{2,30}\))?$")

    @classmethod
    def parse(cls, zb: pd.DataFrame, sitzung: int) -> pd.DataFrame:
        zeilen = []
        for nr, g in zb.loc[zb["beitrag"].ge(0)].groupby("beitrag", sort=True):
            marke = g.index[g["text"].str.contains(cls.MARKE)]
            if not len(marke):
                continue
            segs = []
            for r in g.loc[marke[0]:].itertuples(index=False):
                teile = r.segmente if isinstance(r.segmente, (list, tuple)) else [(r.x0, r.text, r.fett_anteil > 0.9)]
                segs += [(r.seite, r.spalte, x, r.yb, t.strip(), f) for x, t, f in teile]
            if not segs:
                continue
            df = pd.DataFrame(segs, columns=["seite", "spalte", "x", "yb", "text", "fett"])
            df["teilspalte"] = df.groupby(["seite", "spalte"])["x"].transform(
                lambda x: np.searchsorted(np.unique(np.round(x / 25) * 25), np.round(x / 25) * 25))
            stimme, fraktion = None, None
            for r in df.sort_values(["seite", "spalte", "teilspalte", "yb"]).itertuples(index=False):
                t = r.text.rstrip(":")
                if r.fett and t.casefold() in cls.STIMME:
                    stimme, fraktion = cls.STIMME[t.casefold()], None
                elif r.fett and kanon_fraktion(t):
                    fraktion = kanon_fraktion(t)
                elif stimme and fraktion and cls.NAME.match(t):
                    zeilen.append({"Sitzung": sitzung, "Beitrag_Nr": nr + 1, "Name": t, "Fraktion": fraktion, "Stimme": stimme})
        return pd.DataFrame(zeilen, columns=["Sitzung", "Beitrag_Nr", "Name", "Fraktion", "Stimme"])

In [12]:
# =====================================================================
# Sitzungsmodell: Debatten, Beitragsart, Redeeinheiten, Zwischenrufe
# =====================================================================
DEBATTENTYP = [("Fragestunde", r"\bFragestunde\b"), ("Regierungsbefragung", r"Befragung der Bundesregierung"),
               ("Aktuelle Stunde", r"Aktuelle Stunde"), ("Regierungserklärung", r"Regierungserklärung"),
               ("Vereinbarte Debatte", r"Vereinbarte Debatte"), ("Wahl", r"^(?:\S+\s){0,3}Wahl\b"),
               ("Geschäftsordnung", r"^Zur Geschäftsordnung$")]
QA_TYPEN = {"Fragestunde", "Regierungsbefragung"}
TOP_VERWEIS = re.compile(r"\((?P<art>Tagesordnungspunkt|Zusatzpunkt)\s+(?P<label>[\w.]+)")

# Hinweise der Sitzungsleitung auf die Art des folgenden Beitrags (Begriffe der GO-BT)
HINWEIS = [("Kurzintervention", re.compile(r"Kurzintervention", re.I)),
           ("Erwiderung", re.compile(r"\berwider|Erwiderung", re.I)),
           ("Zwischenfrage", re.compile(r"Zwischenfrage|Frage oder Bemerkung|Zwischenbemerkung", re.I)),
           ("Geschäftsordnung", re.compile(r"zur Geschäftsordnung", re.I)),
           ("Erklärung", re.compile(r"Erklärung nach §\s?3[01]|persönliche Erklärung|Erklärung zur Aussprache", re.I))]


class Sitzungsmodell:
    """Debatten aus dem IV (Blöcke, die 'in Verbindung mit' aufgerufen werden, bilden eine Debatte),
    Zuordnung Beitrag -> Debatte, Beitragsart und Redeeinheiten.

    Redeeinheit: ein im IV verzeichneter Beitrag zusammen mit allen nicht verzeichneten Fortsetzungen
    derselben Person in derselben Debatte, bis eine andere Person mit IV-Eintrag spricht (Unterbrechung
    durch Zwischenfrage, Ermahnung oder Antwort der Sitzungsleitung). Nicht verzeichnete Beiträge
    anderer Personen sind Zwischenbeiträge (Zwischenfrage, Nachfrage)."""

    def __init__(self, sitzung: int, iv: pd.DataFrame, beitraege: pd.DataFrame):
        self.sitzung, self.iv, self.b = sitzung, iv, beitraege.copy()

    # ---------------------------------------------------------------- Debatten
    def debatten(self) -> pd.DataFrame:
        zeilen = []
        for nr, g in self.iv.groupby("Debatte_Nr", sort=True):
            gg = g.loc[g["Art"].eq("gegenstand")]
            texte = " | ".join(gg["Text"])
            bloecke = [b for b in dict.fromkeys(g["Block"].dropna())]
            typ = next((t for t, rx in DEBATTENTYP if re.search(rx, texte) or any(re.search(rx, b) for b in bloecke)), None)
            anlage = bool(bloecke) and all(str(b).startswith("Anlage") for b in bloecke)
            ref = None
            if anlage:
                kopf = str(g["Text"].iloc[0])                    # erster Eintrag benennt den Anlageninhalt
                m = TOP_VERWEIS.search(kopf)
                ref = f"{'Tagesordnungspunkt' if m.group('art').startswith('T') else 'Zusatzpunkt'} {m.group('label')}" if m else None
                typ = ("zu Protokoll" if re.search(r"zu Protokoll", kopf, re.I)
                       else "Erklärung" if re.search(r"^Erklärung", kopf) else "Anlage")
            zeilen.append({"Sitzung": self.sitzung, "Debatte_Nr": int(nr),
                           "Debatte_ID": f"20/{self.sitzung}:D{int(nr):02d}",
                           "Bloecke": bloecke, "Typ": typ or ("Anlage" if anlage else "Beratung"),
                           "Anlage": anlage, "Bezug_Block": ref,
                           "Gegenstaende": int(len(gg)), "Titel": texte[:300],
                           "IV_Personen": int(g["Art"].eq("person").sum())})
        d = pd.DataFrame(zeilen)
        # Anlage mit Verweis "(Tagesordnungspunkt 5)" -> Debatte dieses Blocks
        block_debatte = {b: n for n, bl in zip(d["Debatte_Nr"], d["Bloecke"]) for b in bl}
        d["Bezug_Debatte_Nr"] = [block_debatte.get(r) if isinstance(r, str) else None for r in d["Bezug_Block"]]
        return d

    # ---------------------------------------------------------------- Beiträge
    def beitraege(self, debatten: pd.DataFrame) -> pd.DataFrame:
        b = self.b.sort_values("Zeile_von").reset_index(drop=True)
        typ = debatten.set_index("Debatte_Nr")["Typ"].to_dict()
        bezug = debatten.set_index("Debatte_Nr")["Bezug_Debatte_Nr"].to_dict()
        # 1) Debatte: IV-Verknüpfung; Sitzungsleitung mit Aufruf -> nächste Debatte; sonst Vorgänger
        deb = b["Debatte_Nr"].astype("Float64")
        ruft_auf = b["Rolle"].eq("Sitzungsleitung") & b["Einschub"].str.len().gt(0)
        nach_vorne = deb.bfill()
        b["Debatte_vorher"] = deb.ffill().astype("Int64")      # Sitzungsleitung vor einem Aufruf: Abstimmung der Vordebatte
        deb = deb.where(deb.notna() | ~ruft_auf, nach_vorne).ffill().bfill()
        b["Debatte_Nr"] = deb.astype("Int64")
        b["Debatte_Quelle"] = np.where(b["IV_ID"].notna(), "IV", "Position")
        b["Debattentyp"] = b["Debatte_Nr"].map(typ)
        # Anlagen: Bezugsdebatte übernehmen (zu Protokoll gegebene Reden gehören zur Beratung des TOP)
        b["Anlage_Debatte_Nr"] = np.where(b["Bereich"].eq("anlagen"), b["Debatte_Nr"], pd.NA)
        b["Debatte_Nr"] = [int(bezug.get(d)) if (bez := bezug.get(d)) is not None and pd.notna(bez) else d
                           for d in b["Debatte_Nr"]]
        b["Debatte_ID"] = [f"20/{self.sitzung}:D{int(d):02d}" for d in b["Debatte_Nr"]]
        b["Beitragsart"] = self._beitragsart(b)
        return self._redeeinheiten(b)

    def _beitragsart(self, b: pd.DataFrame) -> list[str]:
        arten, fenster, sl_zuletzt = [], [], ""
        for r in b.itertuples(index=False):
            if r.Rolle == "Sitzungsleitung":
                arten.append("Sitzungsleitung")
                sl_zuletzt = r.Text
            elif r.Bereich == "anlagen":
                arten.append("Rede (zu Protokoll)" if r.Debattentyp == "zu Protokoll" else "Erklärung (schriftlich)")
            elif isinstance(r.Block, str) and "Geschäftsordnung" in r.Block:
                arten.append("Geschäftsordnung")
            elif r.Debattentyp in QA_TYPEN:
                arten.append("Antwort" if r.Rolle == "Bundesregierung" else "Frage")
            else:
                arten.append(self._hinweis(" ".join(fenster[-3:]), r.Name, sl_zuletzt) or "Rede")
            fenster.append(r.Text[-600:])
        return arten

    @staticmethod
    def _hinweis(kontext: str, name: str, sl_zuletzt: str = "") -> str | None:
        """Art aus der Worterteilung: Hinweis in den letzten Sätzen der vorangehenden (bis zu drei) Beiträge,
        in denen die Person genannt ist ('… erteile ich das Wort für eine Kurzintervention an Stefan Keuter',
        'Ich erlaube die Zwischenfrage von Kollege Rief'). 'Erwiderung' nur aus der unmittelbar vorangehenden
        Äußerung der Sitzungsleitung ('Möchten Sie erwidern?' – ohne Namen)."""
        nn = nachname(name)
        for s in saetze(sl_zuletzt)[-2:]:
            if HINWEIS[1][1].search(s):
                return "Erwiderung"
        if not kontext:
            return None
        for s in reversed(saetze(kontext)[-6:]):
            if re.search(rf"\b{re.escape(nn)}\b", personen_schluessel(s)):
                for art, rx in HINWEIS:
                    if art != "Erwiderung" and rx.search(s):
                        return art
        return None

    def _redeeinheiten(self, b: pd.DataFrame) -> pd.DataFrame:
        """Hauptrede bleibt offen über Zwischenfragen/Zwischenbeiträge anderer Personen; ein nicht im IV
        verzeichneter Beitrag des Hauptredners in derselben Debatte setzt sie fort."""
        b["Rede_Nr"] = pd.array([pd.NA] * len(b), dtype="Int64")
        b["Fortsetzung"] = False
        haupt, nr = None, 0                                    # haupt = (Person, Debatte, Rede_Nr)
        for i, r in b.iterrows():
            if r["Rolle"] == "Sitzungsleitung":
                continue
            key, deb, art = personen_schluessel(r["Name"]), r["Debatte_Nr"], r["Beitragsart"]
            if r["Debattentyp"] in QA_TYPEN or r["Bereich"] == "anlagen":
                nr += 1; b.at[i, "Rede_Nr"] = nr; haupt = None
                continue
            if haupt and pd.isna(r["IV_ID"]) and key == haupt[0] and deb == haupt[1]:
                b.at[i, "Rede_Nr"], b.at[i, "Fortsetzung"] = haupt[2], True
                continue
            nr += 1
            b.at[i, "Rede_Nr"] = nr
            if pd.notna(r["IV_ID"]) and art != "Zwischenfrage":
                haupt = (key, deb, nr)                         # Rede, Kurzintervention, Erwiderung …
            elif pd.isna(r["IV_ID"]) and art == "Rede":
                b.at[i, "Beitragsart"] = "Zwischenbeitrag"
        b["Rede_ID"] = [f"20/{self.sitzung}:R{int(n):04d}" if pd.notna(n) else None for n in b["Rede_Nr"]]
        return b


# ---------------------------------------------------------------- Zwischenrufe
REAKTION = re.compile(r"^(?P<art>(?:(?:Lebhafter|Langanhaltender|Lang anhaltender|Anhaltender|Erneuter|Weiterer|Weitere|"
                      r"Starker|Vereinzelter|Vereinzelte|Allgemeiner)\s+)?(?:Beifall|Lachen|Heiterkeit|Widerspruch|Zurufe?|"
                      r"Gegenrufe?|Unruhe|Zustimmung|Buhrufe?|Pfiffe|Rufe|Glocke|Bravo-?Rufe?)"
                      r"|Die Abgeordneten\b|Abg\.)\b(?P<rest>.*)$", re.S)
RUFER = re.compile(r"(?:(?:Zuruf|Gegenruf) (?:des|der) Abg\.\s+)?(?P<name>[A-ZÄÖÜ][^\[\]():–]{1,60}?)\s*"
                   r"(?:\[(?P<ort>[^\]]{2,30})\]\s*)?\[(?P<fraktion>[^\]]{2,40})\](?:,[^:]{0,80}?gewandt)?(?:\s*:\s*(?P<text>.+))?$", re.S)
PARTEI_IM_TEXT = [("SPD", r"\bSPD\b"), ("CDU/CSU", r"CDU/CSU"), ("AfD", r"\bAfD\b"), ("FDP", r"\bFDP\b"),
                  ("BÜNDNIS 90/DIE GRÜNEN", r"BÜNDNIS(?:SES)? 90/DIE GRÜNEN|GRÜNEN"),
                  ("Die Linke", r"\bLinke[n]?\b|DIE LINKE"), ("BSW", r"\bBSW\b"), ("fraktionslos", r"fraktionslos")]


def zerlege_zwischenrufe(rufe: pd.DataFrame) -> pd.DataFrame:
    """Ein Klammerblock '(Beifall bei der SPD – Jens Spahn [CDU/CSU]: Das …!)' -> Einheiten mit
    Art (Beifall, Lachen, Zuruf, Zwischenruf …), beteiligten Fraktionen und ggf. Person und Wortlaut."""
    zeilen = []
    for r in rufe.itertuples(index=False):
        inhalt = r.Text.strip()
        inhalt = inhalt[1:-1] if inhalt.startswith("(") and inhalt.endswith(")") else inhalt.strip("()")
        for k, teil in enumerate(re.split(r"\s+[–-]\s+(?=[A-ZÄÖÜ(])", inhalt)):
            teil = teil.strip()
            m = RUFER.match(teil)
            if m and not REAKTION.match(teil):
                zeilen.append({"Sitzung": r.Sitzung, "Beitrag_Nr": r.Beitrag_Nr, "Ruf_Nr": r.Ruf_Nr, "Teil": k + 1,
                               "Art": "Zwischenruf" if m.group("text") else "Zuruf",
                               "Person": m.group("name").strip(), "Fraktionen": [kanon_fraktion(m.group("fraktion")) or m.group("fraktion")],
                               "Wortlaut": (m.group("text") or "").strip(), "Text": teil})
                continue
            m = REAKTION.match(teil)
            if not m and zeilen and zeilen[-1]["Art"] == "Zwischenruf" and zeilen[-1]["Ruf_Nr"] == r.Ruf_Nr \
                    and zeilen[-1]["Beitrag_Nr"] == r.Beitrag_Nr:
                zeilen[-1]["Wortlaut"] += " – " + teil           # weiterer Satz desselben Rufers
                zeilen[-1]["Text"] += " – " + teil
                continue
            art = re.sub(r"^(?:Lebhafter|Langanhaltender|Lang anhaltender|Anhaltender|Erneuter|Weiterer|Weitere|Starker|"
                         r"Vereinzelter|Vereinzelte|Allgemeiner)\s+", "", m.group("art")) if m else "Sonstiges"
            art = {"Zurufe": "Zuruf", "Gegenrufe": "Gegenruf", "Abg.": "Zuruf", "Die Abgeordneten": "Sonstiges"}.get(art, art)
            m_p = RUFER.search(teil)
            zeilen.append({"Sitzung": r.Sitzung, "Beitrag_Nr": r.Beitrag_Nr, "Ruf_Nr": r.Ruf_Nr, "Teil": k + 1,
                           "Art": art, "Person": m_p.group("name").strip() if m_p and art in ("Zuruf", "Gegenruf") else None,
                           "Fraktionen": [p for p, rx in PARTEI_IM_TEXT if re.search(rx, teil)],
                           "Wortlaut": "", "Text": teil})
    return pd.DataFrame(zeilen, columns=["Sitzung", "Beitrag_Nr", "Ruf_Nr", "Teil", "Art", "Person",
                                         "Fraktionen", "Wortlaut", "Text"])

In [13]:
# =====================================================================
# Orchestrierung je Protokoll (mit Stufen-Cache) und Korpus
# =====================================================================
PIPELINE_VERSION = "kom82-nlp-1.0"


def code_fingerabdruck(*objekte) -> str:
    """Hash über den Bytecode aller Methoden/Funktionen – ändert sich der Code einer Stufe, ändert
    sich der Cache-Schlüssel automatisch (kein vergessenes Versionshochzählen)."""
    h = hashlib.sha256()
    def code(c):
        h.update(c.co_code)
        for k in c.co_consts:
            code(k) if hasattr(k, "co_code") else h.update(repr(k).encode())
        h.update(repr(c.co_names).encode())
    for o in objekte:
        funktionen = [o] if callable(o) and hasattr(o, "__code__") else [
            v.__func__ if isinstance(v, (staticmethod, classmethod)) else v for v in vars(o).values()]
        for f in funktionen:
            if hasattr(f, "__code__"):
                code(f.__code__)
        h.update(repr({k: v for k, v in vars(o).items() if isinstance(v, (str, int, float, tuple))}
                      if isinstance(o, type) else "").encode())
    return h.hexdigest()[:12]


class Protokoll:
    """Ein Plenarprotokoll: PDF -> Layout -> Bereiche -> IV -> Beiträge -> Sitzungsmodell.
    Alle Tabellen tragen Sitzung und Zeilennummern als Beleg (Seite, Spalte, Druckseite, Quadrant)."""

    TABELLEN = ("zeilen", "iv", "debatten", "beitraege", "zwischenrufe", "iv_personen", "abstimmungslisten", "meta")

    def __init__(self, sitzung: int, pdf: Path, leser: PdfWoerter, layout: Layout):
        self.sitzung, self.pdf, self.leser, self.layout = sitzung, Path(pdf), leser, layout

    def verarbeite(self) -> dict[str, pd.DataFrame]:
        t0 = time.perf_counter()
        woerter, seiten = self.leser.lese(self.pdf)
        z = self.layout.zeilen(woerter, seiten)
        z, hinweise = Bereiche.markiere(z)
        trenner = Silbentrennung.aus_zeilen(z.loc[z["bereich"].isin(["sitzung", "anlagen"]), "text"])
        iv = Inhaltsverzeichnis(self.sitzung).parse(z, trenner)
        b, rufe, zb = Beitraege(self.sitzung).parse(z, trenner)
        b, personen = Verknuepfung.verknuepfe(iv, b)
        sm = Sitzungsmodell(self.sitzung, iv, b)
        debatten = sm.debatten()
        b = sm.beitraege(debatten)
        zr = zerlege_zwischenrufe(rufe)
        listen = Abstimmungslisten.parse(zb, self.sitzung)
        z = z.merge(zb[["zeile", "beitrag", "typ"]], on="zeile", how="left")
        datum = re.search(r"den (\d{1,2})\. (\w+) (\d{4})", " ".join(z.loc[z["seite"].eq(1), "text"]))
        meta = pd.DataFrame([{
            "Sitzung": self.sitzung, "Datum": _datum(datum), "Seiten": int(seiten["seite"].max()),
            "Woerter_PDF": int(len(woerter)), "Zeilen": int(len(z)),
            "Zeichen_PDF": int(woerter["text"].str.len().sum()),
            "Zeichen_Zeilen": int(z["text"].str.replace(" ", "").str.len().sum()),
            "IV_Personen": int(len(personen)), "IV_Personen_verknuepft": int(personen["verknuepft"].sum()),
            "IV_Personen_Anlage_ohne_Kopf": int((~personen["verknuepft"] & personen["Block"].astype(str).str.startswith("Anlage")).sum()),
            "Beitraege": int(len(b)), "Reden": int(b.loc[b["Beitragsart"].eq("Rede"), "Rede_ID"].nunique()),
            "Zwischenrufe": int(len(rufe)), "Silbentrennung": dict(trenner.stat),
            "Hinweise": hinweise, "Sekunden": round(time.perf_counter() - t0, 2),
            "Backend": self.leser.version()}])
        return {"zeilen": z, "iv": iv, "debatten": debatten, "beitraege": b, "zwischenrufe": zr,
                "iv_personen": personen, "abstimmungslisten": listen, "meta": meta}


MONATSNUMMER = {m: i for i, m in enumerate(MONATE.split("|"), 1)}


def _datum(m) -> str | None:
    if not m or m.group(2) not in MONATSNUMMER:
        return None
    return f"{m.group(3)}-{MONATSNUMMER[m.group(2)]:02d}-{int(m.group(1)):02d}"


class StufenCache:
    """Pickle je Protokoll; Schlüssel = SHA-256 des PDFs + Pipelineversion + Parameter + Backend.
    Geänderte Eingaben, Verfahren oder Parameter erzeugen automatisch neue Einträge."""

    def __init__(self, ordner: Path, version: str, parameter: dict):
        self.ordner = Path(ordner)
        self.ordner.mkdir(parents=True, exist_ok=True)
        self.tag = hashlib.sha256(json.dumps([version, parameter], sort_keys=True, default=str).encode()).hexdigest()[:10]

    def pfad(self, pdf: Path) -> Path:
        sha = hashlib.sha256(Path(pdf).read_bytes()).hexdigest()[:16]
        return self.ordner / f"{Path(pdf).stem}_{sha}_{self.tag}.pkl"

    def lade(self, pdf: Path):
        p = self.pfad(pdf)
        if p.is_file():
            with p.open("rb") as f:
                return pickle.load(f)
        return None

    def speichere(self, pdf: Path, daten) -> None:
        p = self.pfad(pdf)
        tmp = p.with_suffix(".part")
        with tmp.open("wb") as f:
            pickle.dump(daten, f, protocol=pickle.HIGHEST_PROTOCOL)
        tmp.replace(p)


class Korpus:
    """Verarbeitet eine Menge von Sitzungen (fehlertolerant) und stellt verkettete Tabellen bereit."""

    def __init__(self, pdf_dir: Path, cache_dir: Path, logger: logging.Logger,
                 backend: str = "auto", layout: LayoutParameter | None = None):
        self.pdf_dir, self.log = Path(pdf_dir), logger
        self.leser = PdfWoerter(backend)
        self.layout = Layout(layout)
        code = code_fingerabdruck(PdfWoerter, Layout, Silbentrennung, Bereiche, Inhaltsverzeichnis, Beitraege,
                                  Verknuepfung, Sitzungsmodell, Abstimmungslisten, Protokoll, parse_kopf,
                                  zerlege_zwischenrufe, _druckseite, saetze)
        self.cache = StufenCache(cache_dir, PIPELINE_VERSION,
                                 {"layout": asdict(self.layout.p), "backend": self.leser.version(), "code": code})
        self.daten: dict[int, dict] = {}
        self.fehler: dict[int, str] = {}

    def pdf(self, sitzung: int) -> Path:
        return self.pdf_dir / f"20_{sitzung:03d}.pdf"

    def verarbeite(self, sitzungen: Iterable[int], cache: bool = True) -> "Korpus":
        for s in dict.fromkeys(int(x) for x in sitzungen):
            pdf = self.pdf(s)
            if not pdf.is_file():
                self.fehler[s] = "PDF fehlt"
                continue
            d = self.cache.lade(pdf) if cache else None
            quelle = "Cache"
            if d is None:
                try:
                    d = Protokoll(s, pdf, self.leser, self.layout).verarbeite()
                except Exception as exc:                       # Protokoll markieren, Lauf fortsetzen
                    self.fehler[s] = f"{type(exc).__name__}: {exc}"
                    self.log.error("Protokoll 20/%d fehlgeschlagen: %s", s, exc)
                    continue
                self.cache.speichere(pdf, d)
                quelle = "PDF"
            self.daten[s] = d
            m = d["meta"].iloc[0]
            self.log.info("20/%d | %s | Beiträge=%d | IV verknüpft %d/%d | %.1fs", s, quelle, m["Beitraege"],
                          m["IV_Personen_verknuepft"], m["IV_Personen"] - m["IV_Personen_Anlage_ohne_Kopf"], m["Sekunden"])
        return self

    def tabelle(self, name: str) -> pd.DataFrame:
        teile = [d[name] for _, d in sorted(self.daten.items()) if name in d and len(d[name])]
        return pd.concat(teile, ignore_index=True) if teile else pd.DataFrame()


class Personenstamm:
    """Eine Person = Name ohne akademische Titel. Der Ortszusatz ('Detlef Müller (Chemnitz)') gehört
    nur dazu, wenn derselbe Name im Korpus mit verschiedenen Zusätzen vorkommt. Partei der Person:
    Mehrheit der Fraktionsangaben in ihren Köpfen, sonst in benannten Zwischenrufen 'Name [Fraktion]',
    sonst 'nicht im PDF belegt' (Evidenzprinzip). Partei eines Beitrags: Kopfangabe, sonst Person."""

    @staticmethod
    def bilde(beitraege: pd.DataFrame, zwischenrufe: pd.DataFrame,
              listen: pd.DataFrame | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
        b = beitraege.loc[beitraege["Rolle"].ne("Sitzungsleitung")].copy()
        b["Schluessel"] = b["Name"].map(personen_schluessel)
        zus = b.assign(Z=b["Zusatz"].where(b["Zusatz"].map(lambda x: isinstance(x, str)), "")) \
               .groupby("Schluessel")["Z"].nunique()
        mehrdeutig = set(zus[zus > 1].index)
        b["Person_ID"] = [s + (f" ({z})" if s in mehrdeutig and isinstance(z, str) else "")
                          for s, z in zip(b["Schluessel"], b["Zusatz"])]
        kopf = b.dropna(subset=["Fraktion_Kopf"]).groupby("Person_ID")["Fraktion_Kopf"].agg(lambda s: Counter(s).most_common(1)[0][0])
        ruf = (zwischenrufe.loc[zwischenrufe["Person"].notna()]
               .assign(Person_ID=lambda d: d["Person"].map(personen_schluessel),
                       F=lambda d: d["Fraktionen"].map(lambda f: f[0] if f else None))
               .dropna(subset=["F"]).groupby("Person_ID")["F"].agg(lambda s: Counter(s).most_common(1)[0][0]))
        liste = (pd.Series(dtype=object) if listen is None or listen.empty else
                 listen.assign(P=lambda d: d["Name"].map(personen_schluessel))
                 .groupby("P")["Fraktion"].agg(lambda s: Counter(s).most_common(1)[0][0]))
        anzeige = b.groupby("Person_ID")["Name"].agg(lambda s: Counter(s).most_common(1)[0][0])
        personen = pd.DataFrame({"Person_ID": anzeige.index, "Redner": anzeige.values})
        def quelle(p):
            k = p.split(" (")[0]
            for name, q in (("Kopf", kopf.get(p)), ("Abstimmungsliste", liste.get(k)), ("Zwischenruf", ruf.get(k))):
                if isinstance(q, str):
                    return q, name
            return "nicht im PDF belegt", "-"
        q = [quelle(p) for p in personen["Person_ID"]]
        personen["Partei"], personen["Partei_Quelle"] = [a for a, _ in q], [b for _, b in q]
        personen["Rollen"] = personen["Person_ID"].map(b.groupby("Person_ID")["Rolle"].agg(lambda s: sorted(set(s))))
        personen["Fraktionen"] = personen["Person_ID"].map(b.groupby("Person_ID")["Fraktion_Kopf"].agg(lambda s: sorted(set(s.dropna()))))
        personen["Funktionen"] = personen["Person_ID"].map(b.groupby("Person_ID")["Funktion"].agg(lambda s: sorted(set(s.dropna()))[:5]))
        partei = personen.set_index("Person_ID")["Partei"]
        b["Partei"] = [fk if isinstance(fk, str) else partei.get(pid) for fk, pid in zip(b["Fraktion_Kopf"], b["Person_ID"])]
        b["Partei_Gruppe"] = np.where(b["Rolle"].isin(["Bundesregierung", "Länder", "Sonstige"]), b["Rolle"], b["Partei"])
        b["Redner"] = b["Person_ID"].map(personen.set_index("Person_ID")["Redner"])
        cols = ["Beitrag_ID", "Person_ID", "Redner", "Partei", "Partei_Gruppe"]
        return personen, beitraege.merge(b[cols], on="Beitrag_ID", how="left")


def struktur_pruefung(korpus: Korpus) -> pd.DataFrame:
    """Referenzfreie Kontrollen je Protokoll (laufen immer, auch im Betrieb): IV-Soll vs. Text-Ist,
    Zeichenbilanz, Typverteilung der Zeilen, Beiträge ohne Debatte."""
    zeilen = []
    for s, d in sorted(korpus.daten.items()):
        m, z, b = d["meta"].iloc[0], d["zeilen"], d["beitraege"]
        soll = m["IV_Personen"] - m["IV_Personen_Anlage_ohne_Kopf"]
        typen = z["typ"].value_counts()
        zeilen.append({"Sitzung": s, "Seiten": m["Seiten"], "IV_Redner_Soll": soll,
                       "IV_Redner_im_Text": m["IV_Personen_verknuepft"],
                       "IV_Abdeckung": m["IV_Personen_verknuepft"] / max(soll, 1),
                       "Beitraege": len(b), "ohne_IV_Anker": int((b["IV_ID"].isna() & b["Rolle"].ne("Sitzungsleitung")).sum()),
                       "Zeilen_Rede": int(typen.get("rede", 0)), "Zeilen_Zwischenruf": int(typen.get("zwischenruf", 0)),
                       "Zeilen_Einschub": int(typen.get("einschub", 0)), "Zeilen_Tabelle": int(typen.get("tabelle", 0)),
                       "Text_ohne_Beitrag": _anteil_rahmen(z),
                       "Hinweise": "; ".join(m["Hinweise"]) if isinstance(m["Hinweise"], list) else ""})
    return pd.DataFrame(zeilen)


def _anteil_rahmen(z: pd.DataFrame) -> float:
    """Zeichenanteil im Sitzungsverlauf/Anlagen, der keinem Beitrag zugeordnet ist (vor dem ersten Kopf)."""
    t = z.loc[z["bereich"].isin(["sitzung", "anlagen"])]
    n = t["text"].str.len()
    return float(n[t["typ"].eq("rahmen") | t["typ"].isna()].sum() / max(n.sum(), 1))

In [14]:
# ---------------------------------------------------------------------
# Korpus strukturieren (Cache je PDF; ein fehlerhaftes Protokoll stoppt den Lauf nicht)
# ---------------------------------------------------------------------
with stufe("4_struktur"):
    KORPUS = Korpus(PDF_DIR, CACHE_DIR / "struktur", logger, backend=PDF_BACKEND).verarbeite(VERARBEITUNG)
META = KORPUS.tabelle("meta")
IV, DEBATTEN = KORPUS.tabelle("iv"), KORPUS.tabelle("debatten")
BEITRAEGE_ROH, ZWISCHENRUFE = KORPUS.tabelle("beitraege"), KORPUS.tabelle("zwischenrufe")
ABSTIMMUNGSLISTEN = KORPUS.tabelle("abstimmungslisten")
PERSONEN, BEITRAEGE = Personenstamm.bilde(BEITRAEGE_ROH, ZWISCHENRUFE, ABSTIMMUNGSLISTEN)
STRUKTUR = struktur_pruefung(KORPUS)
print(f"Protokolle: {len(KORPUS.daten)}/{len(VERARBEITUNG)} | Fehler: {KORPUS.fehler or '–'}")
print(f"Seiten: {META['Seiten'].sum():,} | Beiträge: {len(BEITRAEGE):,} | Personen: {len(PERSONEN):,} | "
      f"IV-Redner im Text gefunden: {STRUKTUR['IV_Redner_im_Text'].sum()}/{STRUKTUR['IV_Redner_Soll'].sum()}")
display(STRUKTUR.describe().loc[["mean", "min", "max"]].round(3))

if BACKEND_VERGLEICH and MODUS == "dev":
    _vgl = {}
    for _b in ("pymupdf", "pdfplumber"):
        _d = Protokoll(_demo, PDF_DIR / f"20_{_demo}.pdf", PdfWoerter(_b), Layout()).verarbeite()
        _m = _d["meta"].iloc[0]
        _vgl[_b] = {"Zeilen": _m["Zeilen"], "Beiträge": _m["Beitraege"], "IV verknüpft": _m["IV_Personen_verknuepft"],
                    "Reden": _m["Reden"], "Zwischenrufe": _m["Zwischenrufe"],
                    "Wörter Reden": int(_d["beitraege"]["Woerter"].sum()), "Sekunden": _m["Sekunden"]}
    display(pd.DataFrame(_vgl))

20:53:50 | INFO    | 20/146 | PDF | Beiträge=461 | IV verknüpft 201/201 | 24.9s
20:54:42 | INFO    | 20/147 | PDF | Beiträge=650 | IV verknüpft 240/241 | 52.1s
20:55:01 | INFO    | 20/148 | PDF | Beiträge=255 | IV verknüpft 83/83 | 18.7s
20:55:35 | INFO    | 20/149 | PDF | Beiträge=378 | IV verknüpft 138/138 | 33.5s
20:55:55 | INFO    | 20/150 | PDF | Beiträge=296 | IV verknüpft 101/101 | 20.2s
20:56:31 | INFO    | 20/151 | PDF | Beiträge=386 | IV verknüpft 144/144 | 35.7s
20:56:44 | INFO    | 20/152 | PDF | Beiträge=113 | IV verknüpft 55/55 | 12.7s
20:57:09 | INFO    | 20/153 | PDF | Beiträge=433 | IV verknüpft 181/181 | 25.1s
20:58:03 | INFO    | 20/154 | PDF | Beiträge=637 | IV verknüpft 245/245 | 53.5s
20:58:34 | INFO    | 20/155 | PDF | Beiträge=299 | IV verknüpft 124/124 | 31.4s
20:58:56 | INFO    | 20/156 | PDF | Beiträge=365 | IV verknüpft 132/132 | 20.9s
20:59:36 | INFO    | 20/157 | PDF | Beiträge=432 | IV verknüpft 179/179 | 40.4s
20:59:51 | INFO    | 20/158 | PDF | Beiträge

Protokolle: 67/67 | Fehler: –
Seiten: 9,218 | Beiträge: 25,087 | Personen: 752 | IV-Redner im Text gefunden: 9697/9704


,Sitzung,Seiten,IV_Redner_Soll,IV_Redner_im_Text,IV_Abdeckung,Beitraege,ohne_IV_Anker,Zeilen_Rede,Zeilen_Zwischenruf,Zeilen_Einschub,Zeilen_Tabelle,Text_ohne_Beitrag
mean,179.0,137.582,144.836,144.731,1.000,374.433,59.224,10502.179,2438.0,467.896,272.433,0.0
min,146.0,36.000,21.000,21.000,0.994,49.000,3.000,1844.000,514.0,28.000,43.000,0.0
max,212.0,274.000,245.000,245.000,1.000,650.000,120.000,21347.000,5147.0,1536.000,2116.000,0.0


## [↩](#toc) 6) Redeeinheiten, Kennzahlen und LLM-taugliches Textformat<a class="anchor" id="k6"></a>

**Metriken der Länge und Häufigkeit**

| Kennzahl | Definition | Begründung |
|:--|:--|:--|
| Anzahl Reden | Redeeinheiten der Art „Rede“ (separat: Kurzintervention, Erwiderung, Frage/Antwort, zu Protokoll) | vergleichbar mit DIP-Aktivitäten |
| **Wörter** (Hauptmaß) | Tokens des Wortlauts ohne Kopf, Zwischenrufe, Tagesordnungstext | unabhängig von Layout und Satzzeichen; robust |
| Zeichen, Sätze, Absätze | Zeichen inkl. Leerzeichen; regelbasierte Satztrennung; Absätze aus dem Einzug | Ergänzung für Stil und Struktur |
| Redezeit (Schätzung) | Wörter / 120 pro Minute | Annahme (Sprechtempo 110–130 W/min), nur zur Orientierung |
| Interaktion | Zwischenrufe je 100 Wörter, Beifall gesamt / aus eigener / aus anderen Fraktionen, Unterbrechungen | Resonanz im Plenum |

**Textformat.** Je Protokoll entsteht `output/text/20_NNN.md` in Lesereihenfolge: Debattenüberschrift, Sprecherkopf als Überschrift mit Fundstellenkommentar, Absätze aus dem Einzug, Zwischenrufe kursiv an ihrer Position, Tagesordnungstext als Zitat. Silbentrennungen am Zeilenende werden über das Vokabular des Protokolls aufgelöst („Bundes-|regierung“ → „Bundesregierung“, „AfD-|Fraktion“ bleibt, „Wirtschafts- und“ bleibt).

In [15]:
# =====================================================================
# Redeeinheiten, Kennzahlen und Beteiligung auf Debattenebene
# =====================================================================
WOERTER_PRO_MINUTE = 120        # Annahme für die Redezeit-Schätzung (Sprechtempo im Plenum ≈ 110–130 Wörter/min)
FORMEL = re.compile(r"^(?:(?:Sehr geehrte[rs]?|Verehrte[rs]?|Liebe|Lieber|Meine (?:sehr )?(?:geehrten |verehrten )?)"
                    r"(?:\s*(?:Frau|Herr|Präsident\w*|Vizepräsident\w*|Kolleg\w*|Damen|Herren|und|liebe|sehr|geehrte\w*|"
                    r"Zuschauer\w*|Gäste|Bürger\w*|Ministerin|Minister|[A-ZÄÖÜ]\w+)[,!]?)+|"
                    r"(?:Vielen|Herzlichen|Recht herzlichen)?\s*Dank(?:e)?(?: schön| sehr)?(?: für (?:Ihre|die|eure) Aufmerksamkeit)?)[.!]?$")
REDEARTEN = ("Rede", "Kurzintervention", "Erwiderung", "Rede (zu Protokoll)", "Zwischenbeitrag", "Zwischenfrage",
             "Geschäftsordnung", "Erklärung", "Erklärung (schriftlich)", "Frage", "Antwort")


class Reden:
    """Fasst Beiträge zu Redeeinheiten zusammen und berechnet Kennzahlen.

    Länge: Wörter (Hauptmaß, unabhängig von Satzzeichen und Layout), Zeichen, Sätze, Absätze und eine
    Redezeit-Schätzung (Wörter / 120 pro Minute). Interaktion: Zwischenrufe je 100 Wörter, Beifall
    gesamt / aus der eigenen Fraktion / aus anderen Fraktionen, Unterbrechungen."""

    @staticmethod
    def bilde(beitraege: pd.DataFrame, zwischenrufe: pd.DataFrame, debatten: pd.DataFrame) -> pd.DataFrame:
        b = beitraege.loc[beitraege["Rolle"].ne("Sitzungsleitung") & beitraege["Rede_ID"].notna()].copy()
        z = zwischenrufe.merge(b[["Sitzung", "Beitrag_Nr", "Rede_ID", "Partei"]], on=["Sitzung", "Beitrag_Nr"], how="inner")
        z["eigen"] = [isinstance(p, str) and p in f for p, f in zip(z["Partei"], z["Fraktionen"])]
        z["fremd"] = [bool(set(f) - {p}) for p, f in zip(z["Partei"], z["Fraktionen"])]
        bei = z.loc[z["Art"].eq("Beifall")]
        reaktion = pd.DataFrame({
            "Zwischenrufe": z.loc[z["Art"].isin(["Zwischenruf", "Zuruf", "Gegenruf"])].groupby("Rede_ID").size(),
            "Beifall": bei.groupby("Rede_ID").size(),
            "Beifall_eigen": bei.loc[bei["eigen"]].groupby("Rede_ID").size(),
            "Beifall_fremd": bei.loc[bei["fremd"]].groupby("Rede_ID").size(),
            "Lachen_Widerspruch": z.loc[z["Art"].isin(["Lachen", "Widerspruch", "Heiterkeit", "Unruhe"])].groupby("Rede_ID").size(),
        }).fillna(0).astype(int)
        agg = (b.sort_values(["Sitzung", "Zeile_von"]).groupby("Rede_ID", sort=False)
               .agg(Sitzung=("Sitzung", "first"), Debatte_ID=("Debatte_ID", "first"),
                    Debattentyp=("Debattentyp", "first"), Beitragsart=("Beitragsart", "first"),
                    Person_ID=("Person_ID", "first"), Redner=("Redner", "first"), Partei=("Partei", "first"),
                    Partei_Gruppe=("Partei_Gruppe", "first"), Rolle=("Rolle", "first"), Funktion=("Funktion", "first"),
                    IV_ID=("IV_ID", "first"), Druckseite=("Druckseite", "first"), Quadrant=("Quadrant", "first"),
                    Bereich=("Bereich", "first"), Teile=("Beitrag_ID", "size"), Beitraege=("Beitrag_ID", list),
                    Text=("Text", lambda s: "\n".join(t for t in s if t)), Absaetze=("Absaetze", "sum"))
               .reset_index())
        agg["Woerter"] = agg["Text"].str.split().str.len()
        agg["Zeichen"] = agg["Text"].str.len()
        agg["Saetze"] = agg["Text"].map(lambda t: len(saetze(t)))
        agg["Minuten_geschaetzt"] = (agg["Woerter"] / WOERTER_PRO_MINUTE).round(2)
        agg["Unterbrechungen"] = agg["Teile"] - 1
        agg = agg.merge(reaktion, left_on="Rede_ID", right_index=True, how="left")
        for c in reaktion.columns:
            agg[c] = agg[c].fillna(0).astype(int)
        agg["Zwischenrufe_je_100W"] = (100 * agg["Zwischenrufe"] / agg["Woerter"].clip(lower=1)).round(2)
        agg["Leer"] = agg["Woerter"].eq(0)
        return agg


def satzliste(reden: pd.DataFrame, min_woerter: int = 3) -> pd.DataFrame:
    """Sätze je Rede für das Sentiment – ohne Anrede-/Dankesformeln und ohne Kurzsätze ('Ja.')."""
    zeilen = []
    for r in reden.loc[~reden["Leer"]].itertuples(index=False):
        for i, s in enumerate(saetze(r.Text)):
            formel = bool(FORMEL.match(s.strip())) or len(s.split()) < min_woerter
            zeilen.append({"Rede_ID": r.Rede_ID, "Satz_Nr": i + 1, "Satz": s, "Woerter": len(s.split()), "Formel": formel})
    return pd.DataFrame(zeilen, columns=["Rede_ID", "Satz_Nr", "Satz", "Woerter", "Formel"])


def beteiligung_debatte(reden: pd.DataFrame, vorlagen: pd.DataFrame) -> pd.DataFrame:
    """Gesetz × Rede: Wer in der Debatte zu einer Gesetzesvorlage spricht, ist beteiligt
    (Zuordnung 'direkt', wenn die Debatte nur diese Vorlage berät, sonst 'verbundene Debatte')."""
    g = vorlagen.loc[vorlagen["Gesetz_final"]]
    n_vorlagen = vorlagen.groupby("Debatte_ID").size()
    r = reden.loc[reden["Beitragsart"].isin(REDEARTEN[:9]) & ~reden["Leer"]]
    m = g[["Vorlage_ID", "Gesetz_ID", "Debatte_ID"]].merge(r[["Rede_ID", "Debatte_ID", "Person_ID", "Redner", "Partei",
                                                              "Partei_Gruppe", "Beitragsart", "Woerter"]], on="Debatte_ID")
    m["Zuordnung"] = np.where(m["Debatte_ID"].map(n_vorlagen).fillna(1).gt(1), "verbundene Debatte", "direkt")
    m["Methode"] = "Debatte"
    m["Score"] = 1.0
    return m


def redner_tabelle(reden: pd.DataFrame, personen: pd.DataFrame, arten: tuple = ("Rede",)) -> pd.DataFrame:
    """Ergebnis-Dataframe der Aufgabe: je Person Name, Partei, Redeanzahl, mittlere und Gesamtlänge
    (+ Median, Sätze, Redezeit-Schätzung, Interaktion, Sentiment)."""
    r = reden.loc[reden["Beitragsart"].isin(arten) & ~reden["Leer"]]
    agg = r.groupby("Person_ID").agg(
        Reden=("Rede_ID", "size"), Woerter_gesamt=("Woerter", "sum"), Woerter_Mittel=("Woerter", "mean"),
        Woerter_Median=("Woerter", "median"), Zeichen_gesamt=("Zeichen", "sum"), Saetze_gesamt=("Saetze", "sum"),
        Minuten_geschaetzt=("Minuten_geschaetzt", "sum"), Sitzungen=("Sitzung", "nunique"),
        Partei_Gruppe=("Partei_Gruppe", lambda s: Counter(s).most_common(1)[0][0]),
        Zwischenrufe_erhalten=("Zwischenrufe", "sum"), Beifall_erhalten=("Beifall", "sum"),
        **({"Sentiment_Mittel": ("Sentiment_norm", "mean")} if "Sentiment_norm" in r else {}))
    agg["Woerter_Mittel"] = agg["Woerter_Mittel"].round(1)
    alle = reden.loc[~reden["Leer"] & reden["Rolle"].ne("Sitzungsleitung")].groupby("Person_ID")["Rede_ID"].size()
    out = personen[["Person_ID", "Redner", "Partei", "Partei_Quelle", "Rollen"]].merge(agg, on="Person_ID", how="inner")
    out["Beitraege_alle_Arten"] = out["Person_ID"].map(alle)
    return out.sort_values(["Reden", "Woerter_gesamt"], ascending=False).reset_index(drop=True)


def parteien_tabelle(reden: pd.DataFrame, arten: tuple = ("Rede",)) -> pd.DataFrame:
    r = reden.loc[reden["Beitragsart"].isin(arten) & ~reden["Leer"]]
    return r.groupby("Partei_Gruppe").agg(
        Reden=("Rede_ID", "size"), Redner=("Person_ID", "nunique"), Woerter_gesamt=("Woerter", "sum"),
        Woerter_Mittel=("Woerter", "mean"), Woerter_Median=("Woerter", "median"),
        Zwischenrufe_je_100W=("Zwischenrufe_je_100W", "mean"), Beifall_Mittel=("Beifall", "mean"),
        **({"Sentiment_Mittel": ("Sentiment_norm", "mean"), "Anteil_negativ": ("Anteil_negativ", "mean")}
           if "Sentiment_norm" in r else {})).round(3).sort_values("Reden", ascending=False)


def gesetzeserwaehnungen(reden: pd.DataFrame, gesetze: pd.DataFrame) -> pd.DataFrame:
    """Passagen, in denen (auch bestehende) Gesetze in Reden genannt werden: Gesetzesnamen
    ('…gesetz', '…gesetzbuch', 'Gesetz zur/über …') je Satz; Verknüpfung mit einem Gesetz der Tagesordnung,
    wenn Kurztitel oder normierter Titel übereinstimmen."""
    def kern(t: str) -> str:            # ohne Ordnungszahl, Jahreszahl und Klammerzusatz
        t = Gesetzesidentitaet.schluessel(re.sub(r"\s*\([^()]*\)\s*$", "", t))
        t = re.sub(r"\b(?:erst|zweit|dritt|viert|funft|sechst|siebt|acht|neunt|zehnt)\w*\b|\b(?:19|20)\d{2}\b", " ", t)
        return " ".join(t.split())
    bekannt = {}
    for g in gesetze.itertuples(index=False):
        for t in (g.Kurztitel, g.Gesetzestitel):
            if isinstance(t, str) and len(t) > 4:
                bekannt[kern(t)] = g.Gesetz_ID
    zeilen = []
    for r in reden.loc[~reden["Leer"] & reden["Rolle"].ne("Sitzungsleitung")].itertuples(index=False):
        for s in saetze(r.Text):
            for m in GESETZ_NENNUNG.finditer(s):
                name = nominativ(m.group(0))
                key = kern(name)
                zeilen.append({"Rede_ID": r.Rede_ID, "Sitzung": r.Sitzung, "Debatte_ID": r.Debatte_ID,
                               "Redner": r.Redner, "Partei_Gruppe": r.Partei_Gruppe, "Gesetz_genannt": name,
                               "Gesetz_ID": bekannt.get(key), "Passage": s})
    return pd.DataFrame(zeilen, columns=["Rede_ID", "Sitzung", "Debatte_ID", "Redner", "Partei_Gruppe",
                                         "Gesetz_genannt", "Gesetz_ID", "Passage"])

In [16]:
# =====================================================================
# LLM-taugliches Textformat: Markdown je Protokoll
# =====================================================================
class TextExport:
    """Markdown in Lesereihenfolge: Debattenüberschriften, Sprecherköpfe als Überschrift, Absätze aus
    dem Einzug, Zwischenrufe kursiv in Klammern an ihrer Position, Tagesordnungstext als Zitat.
    Jede Rede beginnt mit einer Fundstellenzeile (Druckseite/Quadrant) – zitierfähig für LLM/RAG."""

    @staticmethod
    def markdown(d: dict) -> str:
        z, b, deb, meta = d["zeilen"], d["beitraege"], d["debatten"], d["meta"].iloc[0]
        trenner = Silbentrennung.aus_zeilen(z.loc[z["bereich"].isin(["sitzung", "anlagen"]), "text"])
        b = b.set_index("Beitrag_Nr")
        titel = deb.set_index("Debatte_ID")["Titel"].to_dict()
        aus = [f"# Plenarprotokoll 20/{meta['Sitzung']} – {meta['Datum']}", ""]
        aktuelle_debatte, absatz, block, block_typ = None, [], [], None

        def flush_absatz():
            nonlocal absatz
            if absatz:
                aus.extend([trenner.fuege(absatz), ""]); absatz = []

        def flush_block():
            nonlocal block, block_typ
            if block:
                t = trenner.fuege(block)
                aus.extend([f"*{t}*" if block_typ == "zwischenruf" else f"> {t}", ""]); block = []
            block_typ = None

        sv = z.loc[z["bereich"].isin(["sitzung", "anlagen"]) & z["beitrag"].ge(0)]
        letzter = None
        for r in sv.itertuples(index=False):
            nr = int(r.beitrag) + 1
            if nr != letzter:
                flush_absatz(); flush_block()
                info = b.loc[nr]
                if info["Debatte_ID"] != aktuelle_debatte:
                    aktuelle_debatte = info["Debatte_ID"]
                    aus.extend([f"## {aktuelle_debatte}: {str(titel.get(aktuelle_debatte, ''))[:200]}", ""])
                partei = info.get("Partei") if isinstance(info.get("Partei"), str) else info.get("Fraktion_Kopf")
                kopf = info["Kopftext"]
                aus.extend([f"### {kopf}", f"<!-- {info['Beitrag_ID']} | S. {info['Druckseite']} {info['Quadrant']} | "
                            f"{info['Beitragsart']} | {txt(partei) or info['Rolle']} -->", ""])
                letzter = nr
            if r.typ == "kopf":
                continue
            if r.typ in ("zwischenruf", "einschub"):
                flush_absatz()
                if block_typ not in (None, r.typ):
                    flush_block()
                block_typ = r.typ; block.append(r.text)
                continue
            flush_block()
            if r.typ == "rede":
                if 5 <= r.einzug_l <= 16 and absatz:
                    flush_absatz()
                absatz.append(r.text)
        flush_absatz(); flush_block()
        return "\n".join(aus)

In [17]:
# ---------------------------------------------------------------------
# Redeeinheiten und Kennzahlen; LLM-taugliche Markdown-Texte je Protokoll
# ---------------------------------------------------------------------
with stufe("6_reden"):
    REDEN = Reden.bilde(BEITRAEGE, ZWISCHENRUFE, DEBATTEN)
    for s, d in KORPUS.daten.items():
        (TEXT_DIR / f"20_{s:03d}.md").write_text(TextExport.markdown(d), encoding="utf-8")
print(f"Redeeinheiten: {len(REDEN):,} | davon Art 'Rede': {(REDEN['Beitragsart'] == 'Rede').sum():,} | "
      f"Markdown: {len(KORPUS.daten)} Dateien in {TEXT_DIR.name}/")
display(REDEN["Beitragsart"].value_counts().to_frame().T)
display(REDEN.loc[REDEN["Beitragsart"].eq("Rede"), ["Woerter", "Saetze", "Minuten_geschaetzt", "Zwischenrufe",
                                                     "Beifall", "Zwischenrufe_je_100W"]].describe().round(1))

21:26:23 | INFO    | Stufe 6_reden                          24.4 s


Redeeinheiten: 11,046 | davon Art 'Rede': 6,407 | Markdown: 67 Dateien in text/


Beitragsart,Rede,Antwort,Frage,Rede (zu Protokoll),Zwischenbeitrag,Erklärung (schriftlich),Zwischenfrage,Geschäftsordnung,Kurzintervention,Erwiderung,Erklärung
count,6407,1792,1533,362,294,240,172,106,103,30,7


,Woerter,Saetze,Minuten_geschaetzt,Zwischenrufe,Beifall,Zwischenrufe_je_100W
count,6407.0,6407.0,6407.0,6407.0,6407.0,6407.0
mean,568.9,36.2,4.7,6.3,5.2,1.1
std,279.8,16.9,2.3,7.6,3.6,1.1
min,9.0,1.0,0.1,0.0,0.0,0.0
25%,410.0,26.0,3.4,1.0,3.0,0.3
50%,545.0,34.0,4.5,4.0,5.0,0.8
75%,696.0,44.0,5.8,9.0,7.0,1.6
max,4872.0,281.0,40.6,114.0,53.0,14.8


## [↩](#toc) 7) Gesetze: Extraktion, Identität, Abstimmung<a class="anchor" id="k7"></a>

**Regel (deterministisch).** Jede Vorlage ist ein Gegenstand des IV. Der **Titel ist die fett gesetzte Spanne** des Eintrags – wörtlich und damit selbst Beleg; der nicht fette Vorspann bestimmt Vorlagentyp und Beratung („Erste Beratung des … eingebrachten Entwurfs eines“). Gesetzgebung nach Regel: Gesetzentwurf, Beschlussempfehlung zu einem Gesetz oder ein Gesetzesname als Kopf des Titels (nicht bei Anträgen, Verordnungen, Unterrichtungen, Wahlen). Der Titel wird in den Nominativ gebracht („Zweiten Haushaltsfinanzierungsgesetzes 2024“ → „Zweites Haushaltsfinanzierungsgesetz 2024“), der Kurztitel ist der Klammerzusatz am Titelende.

**LLM (Restfelder).** `qwen3:8b` erhält je Vorlage den IV-Text und liefert per JSON-Schema Typ, Gesetzgebung ja/nein, Titel, Kurztitel, Beratung und genannte bestehende Gesetze. **Belegprüfung**: Titel und Gesetzesnamen werden nur übernommen, wenn sie (normalisiert) wörtlich im Eintrag stehen; verworfene Werte werden gezählt. **Hybrid** = Regel ∪ (LLM mit belegtem Titel, der ein Gesetzeswort enthält): das LLM erhöht den Recall, ohne unbelegte Titel einzuführen. Regel, LLM und Hybrid werden in Kapitel 14 gegen DIP verglichen.

**Drucksache als Schlüssel.** Unterpunkte, die ein Gesetz nur über seine Drucksache beraten („hier: Einzelplan 60, Drucksachen 20/7800“), werden als Beratung dieses Gesetzes erkannt. Sammeldrucksachen („20/12868 Nr. 2.1“) sind ausgeschlossen. Über Sitzungen hinweg bilden Vorlagen mit gleicher Haupt-Drucksache oder gleichem normierten Titel ein Gesetz (Union-Find: erste Lesung, 2./3. Lesung, Vermittlungsverfahren).

**Abstimmung (LLM).** Für jede Debatte erhält das LLM die nummerierten Vorlagen und die Äußerungen der Sitzungsleitung ab dem ersten Abstimmungsbegriff; Ergebnis (angenommen, abgelehnt, überwiesen …) nur mit wörtlichem Belegsatz.

**Bestehende Gesetze in Reden.** Gesetzesnamen („…gesetz“, „…gesetzbuch“, „Gesetz zur/über …“) werden satzweise als Passage gespeichert und – wenn möglich – einem Gesetz der Tagesordnung zugeordnet.

In [18]:
# =====================================================================
# Vorlagen und Gesetze – deterministische Kandidaten aus dem IV (Titel = fett gesetzte Spanne)
# =====================================================================
VORLAGENTYP = [
    ("Gesetzentwurf", r"\bEntwurfs?\s+(?:eines|einer)\b|\bGesetzentwurfs?\b"),
    ("Beschlussempfehlung zum Gesetz", r"(?:Beschlussempfehlung|Vermittlungsausschuss).{0,200}\bzu dem\s+(?:\w+\s+){0,2}Gesetz"),
    ("Verordnung", r"\bVerordnung\b"),
    ("Antrag", r"\bAntrag|\bAnträge"),
    ("Unterrichtung", r"\bUnterrichtung\b"),
    ("Bericht", r"\bBericht|Jahresbericht"),
    ("Wahl", r"\bWahl\b|Wahlvorschlag"),
    ("Regierungserklärung", r"Regierungserklärung"),
    ("Aktuelle Stunde", r"Aktuelle Stunde"),
    ("Befragung/Fragestunde", r"Befragung der Bundesregierung|Fragestunde"),
    ("Vereinbarte Debatte", r"Vereinbarte Debatte"),
]
BERATUNG = [("Erste Beratung", r"Erste Beratung"), ("Zweite und dritte Beratung", r"Zweite und dritte Beratung"),
            ("Zweite Beratung", r"Zweite Beratung"), ("Dritte Beratung", r"Dritte Beratung"),
            ("Beschlussempfehlung", r"Beschlussempfehlung"), ("Beratung", r"\bBeratung")]
GESETZ_KOPF = re.compile(r"^(?:[\wÄÖÜäöüß.-]+\s+){0,3}?[\wÄÖÜäöüß-]*(?:[Gg]esetz(?:es|buch(?:es)?)?|Staatsvertrag(?:es|s)?)\b")
GESETZ_NENNUNG = re.compile(
    r"\b(?:[A-ZÄÖÜ][\wäöüß-]*(?:gesetz(?:es|e|buch(?:es)?)?|Gesetzbuch(?:es)?)\b|Gesetz(?:es)?\s+(?:zur|zum|über|für|zu|gegen)\s+"
    r"[^,.;:()–]{3,160}[^,.;:()–\s])")
ORDINAL_GEN = re.compile(r"^(Ersten|Zweiten|Dritten|Vierten|Fünften|Sechsten|Siebten|Achten|Neunten|Zehnten|"
                         r"Elften|Zwölften|\w+zehnten|\w+zigsten)\b")
SAMMEL_DS = re.compile(r"(\d{2}/\s?\d{2,6})\s*Nr\.")


def nominativ(titel: str) -> str:
    """Titel nach 'Entwurfs eines' steht im Genitiv: 'Zweiten Haushaltsfinanzierungsgesetzes 2024'
    -> 'Zweites Haushaltsfinanzierungsgesetz 2024'; nur das Kopfwort wird umgeformt."""
    t = " ".join(str(titel).split())
    t = ORDINAL_GEN.sub(lambda m: m.group(1)[:-1] + "s", t)
    return re.sub(r"\b([\wÄÖÜäöüß-]*[Gg]esetz|[\wÄÖÜäöüß-]*[Gg]esetzbuch|Staatsvertrag)(?:es|s)\b", r"\1", t, count=1)


def kurztitel(titel: str) -> str | None:
    """Klammerzusatz am Titelende: '(Nachtragshaushaltsgesetz 2024)', '(GKV-FKG)'."""
    m = re.search(r"\(([^()]{2,80})\)\s*$", str(titel))
    if m and not re.search(r"Drucksache|Ausschuss|Tagesordnungspunkt", m.group(1)):
        return m.group(1).strip()
    return None


class Vorlagen:
    """Vorlage = Gegenstand einer Debatte laut IV. Typ und Beratung aus dem Vorlagentext, Titel aus
    der fett gesetzten Spanne (wörtlich, Beleg), Drucksachen als Schlüssel. Gesetzgebung nach Regel:
    Gesetzentwurf / Beschlussempfehlung zu einem Gesetz oder Gesetzesname als Kopf des Titels."""

    @staticmethod
    def aus_iv(iv: pd.DataFrame, debatten: pd.DataFrame) -> pd.DataFrame:
        deb = debatten.set_index(["Sitzung", "Debatte_Nr"])
        v = iv.loc[iv["Art"].eq("gegenstand")].copy()
        v = v.loc[[not bool(deb.loc[(s, d), "Anlage"]) for s, d in zip(v["Sitzung"], v["Debatte_Nr"])]]
        v["Debatte_ID"] = [deb.loc[(s, d), "Debatte_ID"] for s, d in zip(v["Sitzung"], v["Debatte_Nr"])]
        v["Debattentyp"] = [deb.loc[(s, d), "Typ"] for s, d in zip(v["Sitzung"], v["Debatte_Nr"])]
        v["Titel"] = [txt(f) or t for f, t in zip(v["Titel_fett"], v["Text"])]
        v["Praefix"] = [t.split(f)[0] if txt(f) and f in t else "" for t, f in zip(v["Text"], v["Titel_fett"].map(txt))]
        v["Vorlagentyp"] = [next((n for n, rx in VORLAGENTYP if re.search(rx, p or t)), "Sonstige")
                            for p, t in zip(v["Praefix"], v["Text"])]
        v["Beratung"] = [next((n for n, rx in BERATUNG if re.search(rx, t)), None) for t in v["Text"]]
        v["Gesetz_Regel"] = [typ in ("Gesetzentwurf", "Beschlussempfehlung zum Gesetz")
                             or (typ not in ("Antrag", "Verordnung", "Unterrichtung", "Wahl") and bool(GESETZ_KOPF.match(ti)))
                             for typ, ti in zip(v["Vorlagentyp"], v["Titel"])]
        v["Gesetzestitel"] = [nominativ(ti) if g else None for g, ti in zip(v["Gesetz_Regel"], v["Titel"])]
        v["Kurztitel"] = [kurztitel(ti) if g else None for g, ti in zip(v["Gesetz_Regel"], v["Titel"])]
        v["Gesetzesbezug"] = [Vorlagen._bezuege(t, ti, g) for t, ti, g in zip(v["Text"], v["Titel"], v["Gesetz_Regel"])]
        v["Drucksache_Haupt"] = [Vorlagen._haupt_ds(t, a) for t, a in zip(v["Text"], v["Anhang"])]
        v = v.rename(columns={"IV_ID": "Vorlage_ID"})
        cols = ["Vorlage_ID", "Sitzung", "Debatte_ID", "Debattentyp", "Block", "Unterblock", "Vorlagentyp",
                "Beratung", "Titel", "Praefix", "Text", "Drucksachen", "Drucksache_Haupt", "Gesetz_Regel",
                "Gesetzestitel", "Kurztitel", "Gesetzesbezug", "Druckseite", "Quadrant", "Zeilen"]
        return v[cols].reset_index(drop=True)

    @staticmethod
    def _bezuege(text: str, titel: str, ist_gesetz: bool) -> list[str]:
        """Genannte (bestehende) Gesetze im Vorlagentext; Teile des eigenen Gesetzestitels zählen nicht."""
        eigen = Gesetzesidentitaet.schluessel(nominativ(titel)) if ist_gesetz else ""
        out = set()
        for m in GESETZ_NENNUNG.finditer(text):
            n = nominativ(m.group(0))
            if Gesetzesidentitaet.schluessel(n) not in eigen:
                out.add(n)
        return sorted(out)

    @staticmethod
    def _haupt_ds(text: str, anhang) -> str | None:
        """Erste Drucksache, die keine Sammeldrucksache ('20/12868 Nr. 2.1') ist."""
        roh = " ".join(x for x in (text, anhang) if isinstance(x, str))
        sammel = {re.sub(r"\s", "", s) for s in SAMMEL_DS.findall(roh)}
        for a, b in DS_NUMMER.findall(roh):
            if f"{a}/{b}" not in sammel:
                return f"{a}/{b}"
        return None


class Gesetzesidentitaet:
    """Gesetz über Sitzungen hinweg (1. Lesung, 2./3. Lesung, Vermittlungsausschuss): Vorlagen mit
    gleicher Haupt-Drucksache oder gleichem normierten Titel bilden ein Gesetz (Union-Find)."""

    @staticmethod
    def schluessel(titel: str) -> str:
        t = unicodedata.normalize("NFKD", str(titel).casefold())
        t = "".join(c for c in t if not unicodedata.combining(c))
        return " ".join(re.sub(r"[^\w\s]", " ", t).split())

    @classmethod
    def bilde(cls, vorlagen: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        g = vorlagen.loc[vorlagen["Gesetz_final"]].copy()
        eltern = {i: i for i in g["Vorlage_ID"]}
        def wurzel(x):
            while eltern[x] != x:
                eltern[x] = eltern[eltern[x]]
                x = eltern[x]
            return x
        def vereine(a, b):
            ra, rb = wurzel(a), wurzel(b)
            if ra != rb:
                eltern[max(ra, rb)] = min(ra, rb)
        g["Titel_Schluessel"] = g["Gesetzestitel_final"].map(cls.schluessel)
        for spalte in ("Drucksache_Haupt", "Titel_Schluessel"):
            for _, ids in g.dropna(subset=[spalte]).groupby(spalte)["Vorlage_ID"]:
                ids = list(ids)
                for x in ids[1:]:
                    vereine(ids[0], x)
        g["Gesetz_Root"] = g["Vorlage_ID"].map(wurzel)
        nummer = {r: i for i, r in enumerate(sorted(g["Gesetz_Root"].unique(), key=lambda r: (int(r.split(":")[0].split("/")[1]), r)), 1)}
        g["Gesetz_ID"] = g["Gesetz_Root"].map(lambda r: f"G{nummer[r]:04d}")
        gesetze = (g.sort_values(["Sitzung", "Vorlage_ID"]).groupby("Gesetz_ID")
                   .agg(Gesetzestitel=("Gesetzestitel_final", lambda s: max(s, key=len)),
                        Kurztitel=("Kurztitel_final", lambda s: next((x for x in s if isinstance(x, str)), None)),
                        Drucksachen=("Drucksachen", lambda s: sorted({d for x in s for d in x})),
                        Sitzungen=("Sitzung", lambda s: sorted(set(s))),
                        Beratungen=("Beratung", lambda s: [b for b in s if isinstance(b, str)]),
                        Vorlagen=("Vorlage_ID", list))
                   .reset_index())
        return gesetze, vorlagen.merge(g[["Vorlage_ID", "Gesetz_ID"]], on="Vorlage_ID", how="left")

In [19]:
# =====================================================================
# LLM-Aufgaben mit JSON-Schema und Belegprüfung (nur wörtlich belegte Werte werden übernommen)
# =====================================================================
def _norm(s: str) -> str:
    return " ".join(str(s).replace("-\n", "").split()).casefold()


def belegt(wert: str | None, quelle: str) -> bool:
    """Wert muss (normalisiert) wörtlich in der Quelle stehen – verhindert erfundene Titel."""
    return isinstance(wert, str) and len(wert.strip()) > 2 and _norm(wert) in _norm(quelle)


SCHEMA_VORLAGE = {
    "type": "object",
    "properties": {
        "vorlagentyp": {"type": "string", "enum": ["Gesetzentwurf", "Beschlussempfehlung zu einem Gesetz", "Antrag",
                                                    "Verordnung", "Unterrichtung/Bericht", "Wahl", "Debatte ohne Vorlage",
                                                    "Sonstiges"]},
        "gesetzgebung": {"type": "boolean"},
        "gesetzestitel": {"type": "string"},
        "kurztitel": {"type": "string"},
        "beratung": {"type": "string", "enum": ["Erste Beratung", "Zweite Beratung", "Zweite und dritte Beratung",
                                                "Dritte Beratung", "Beschlussempfehlung", "ohne Angabe"]},
        "bestehende_gesetze": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["vorlagentyp", "gesetzgebung", "gesetzestitel", "kurztitel", "beratung", "bestehende_gesetze"],
}
SYSTEM_VORLAGE = (
    "Du analysierst Tagesordnungseinträge aus Plenarprotokollen des Deutschen Bundestages. "
    "Bestimme, ob der Eintrag ein Gesetzgebungsverfahren betrifft (Gesetzentwurf, Beschlussempfehlung zu einem Gesetz, "
    "Vermittlungsverfahren zu einem Gesetz, Vertragsgesetz). Anträge, Verordnungen, Berichte und Wahlen sind keine "
    "Gesetzgebung, auch wenn sie ein Gesetz erwähnen. "
    "gesetzestitel: der vollständige Titel des Gesetzes WÖRTLICH aus dem Eintrag (ohne 'Entwurf eines'), sonst ''. "
    "kurztitel: Kurzbezeichnung oder Abkürzung WÖRTLICH aus dem Eintrag (meist in Klammern), sonst ''. "
    "bestehende_gesetze: im Eintrag WÖRTLICH genannte bereits bestehende Gesetze, auf die sich der Eintrag bezieht "
    "(nicht das neue Gesetz selbst). Erfinde nichts; was nicht im Text steht, bleibt leer.")


SCHEMA_ABSTIMMUNG = {
    "type": "object",
    "properties": {"ergebnisse": {"type": "array", "items": {"type": "object", "properties": {
        "vorlage_nr": {"type": "integer"},
        "ergebnis": {"type": "string", "enum": ["angenommen", "abgelehnt", "überwiesen", "erledigt", "vertagt",
                                                 "keine Abstimmung"]},
        "beleg": {"type": "string"}},
        "required": ["vorlage_nr", "ergebnis", "beleg"]}}},
    "required": ["ergebnisse"],
}
SYSTEM_ABSTIMMUNG = (
    "Du liest Äußerungen der Sitzungsleitung im Deutschen Bundestag nach einer Debatte. "
    "Bestimme für jede nummerierte Vorlage das Ergebnis: angenommen, abgelehnt, überwiesen (an Ausschüsse), erledigt, "
    "vertagt oder keine Abstimmung. Gib als beleg den Satz WÖRTLICH an, der das Ergebnis feststellt. "
    "Gibt es keinen solchen Satz, ist das Ergebnis 'keine Abstimmung' und beleg ein leerer String.")


class VorlagenLLM:
    """Klassifiziert alle Vorlagen (Gesetzgebung ja/nein, Titel, Kurztitel, Beratung, bestehende Gesetze).
    Übernommen werden nur wörtlich belegte Titel. Der Vergleich Regel / LLM / Hybrid erfolgt in Kap. 14."""

    def __init__(self, backend: OllamaBackend, logger: logging.Logger):
        self.b, self.log = backend, logger

    def anwenden(self, vorlagen: pd.DataFrame) -> pd.DataFrame:
        aufgaben = [(SYSTEM_VORLAGE, f"Tagesordnungseintrag:\n{t}\n\nDrucksachen: {', '.join(d)}", SCHEMA_VORLAGE)
                    for t, d in zip(vorlagen["Text"], vorlagen["Drucksachen"])]
        t0 = time.perf_counter()
        antworten = self.b.chat_viele(aufgaben)
        self.log.info("LLM Vorlagen: %d Anfragen | %.1fs | %s", len(aufgaben), time.perf_counter() - t0, dict(self.b.statistik))
        v = vorlagen.copy()
        v["LLM_ok"] = [a is not None for a in antworten]
        v["Gesetz_LLM"] = [bool(a and a.get("gesetzgebung")) for a in antworten]
        v["Titel_LLM"] = [a.get("gesetzestitel") if a and belegt(a.get("gesetzestitel"), t) else None
                          for a, t in zip(antworten, v["Text"])]
        v["Kurztitel_LLM"] = [a.get("kurztitel") if a and belegt(a.get("kurztitel"), t) else None
                              for a, t in zip(antworten, v["Text"])]
        v["Bezug_LLM"] = [[g for g in (a.get("bestehende_gesetze") or []) if belegt(g, t)] if a else []
                          for a, t in zip(antworten, v["Text"])]
        v["Titel_verworfen"] = [bool(a and a.get("gesetzestitel") and not belegt(a.get("gesetzestitel"), t))
                                for a, t in zip(antworten, v["Text"])]
        return v


def gesetz_entscheidung(v: pd.DataFrame, modus: str = "hybrid") -> pd.DataFrame:
    """Regel: Gesetz_Regel. LLM: Gesetz_LLM mit belegtem Titel. Hybrid: Regel ODER (LLM mit belegtem Titel,
    der ein Gesetzeswort enthält) – das LLM ergänzt Recall, ohne unbelegte Titel einzuführen.
    Danach Drucksache als Schlüssel: Unterpunkte, die ein Gesetz nur über seine Drucksache beraten
    ('hier: Einzelplan 60, Drucksachen 20/7800'), sind Beratungen dieses Gesetzes."""
    v = v.copy()
    llm_ok = v.get("Gesetz_LLM", pd.Series(False, index=v.index)) & v.get("Titel_LLM", pd.Series(None, index=v.index)).map(
        lambda t: isinstance(t, str) and bool(re.search(r"[Gg]esetz|Staatsvertrag", t)))
    if modus == "regel" or "Gesetz_LLM" not in v:
        v["Gesetz_final"] = v["Gesetz_Regel"]
    elif modus == "llm":
        v["Gesetz_final"] = llm_ok
    else:
        v["Gesetz_final"] = v["Gesetz_Regel"] | llm_ok
    v["Gesetz_Quelle"] = np.select([v["Gesetz_Regel"] & llm_ok, v["Gesetz_Regel"], llm_ok],
                                   ["Regel+LLM", "Regel", "LLM"], "-")
    v["Gesetzestitel"] = v["Gesetzestitel"].astype(object)
    v["Kurztitel"] = v["Kurztitel"].astype(object)
    gesetz = v["Gesetz_final"]
    ds_titel = {d: t for d, t in zip(v.loc[gesetz, "Drucksache_Haupt"], v.loc[gesetz, "Titel"]) if isinstance(d, str)}
    for i in v.index[~gesetz & ~v["Vorlagentyp"].isin(["Antrag", "Verordnung", "Wahl", "Unterrichtung"])]:
        treffer = [d for d in v.at[i, "Drucksachen"] if d in ds_titel]
        if treffer:
            v.at[i, "Gesetz_final"], v.at[i, "Gesetz_Quelle"] = True, "Drucksache"
            v.at[i, "Gesetzestitel"] = nominativ(ds_titel[treffer[0]])
            v.at[i, "Kurztitel"] = kurztitel(ds_titel[treffer[0]])
    titel_llm = v["Titel_LLM"] if "Titel_LLM" in v else pd.Series(None, index=v.index)
    kurz_llm = v["Kurztitel_LLM"] if "Kurztitel_LLM" in v else pd.Series(None, index=v.index)
    v["Gesetzestitel_final"] = [(txt(r) or (nominativ(l) if isinstance(l, str) else None)) if f else None
                                for r, l, f in zip(v["Gesetzestitel"], titel_llm, v["Gesetz_final"])]
    v["Kurztitel_final"] = [(txt(r) or (l if isinstance(l, str) else None)) if f else None
                            for r, l, f in zip(v["Kurztitel"], kurz_llm, v["Gesetz_final"])]
    return v


class AbstimmungLLM:
    """Ergebnis je Vorlage aus den Äußerungen der Sitzungsleitung der Debatte (Abstimmungen,
    Überweisungen). Übernommen nur mit wörtlichem Beleg."""

    def __init__(self, backend: OllamaBackend, logger: logging.Logger, max_zeichen: int = 6000):
        self.b, self.log, self.max = backend, logger, max_zeichen

    def anwenden(self, beitraege: pd.DataFrame, vorlagen: pd.DataFrame) -> pd.DataFrame:
        sl = beitraege.loc[beitraege["Rolle"].eq("Sitzungsleitung")].sort_values(["Sitzung", "Zeile_von"])
        aufruf = sl["Einschub"].str.len().gt(0)
        vorher_id = [f"20/{s}:D{int(d):02d}" if pd.notna(d) else None for s, d in zip(sl["Sitzung"], sl["Debatte_vorher"])]
        # Text eines Sitzungsleitungsbeitrags mit Aufruf: Teil vor dem Aufruf gehört zur Vordebatte (Abstimmung),
        # Teil ab dem Aufruf zur aufgerufenen Debatte
        teile = defaultdict(list)
        for r, vid, auf in zip(sl.itertuples(index=False), vorher_id, aufruf):
            if auf:
                if vid:
                    teile[vid].append((r.Zeile_von, r.Text_vor_Einschub))
                teile[r.Debatte_ID].append((r.Zeile_von + 0.5, r.Text[len(r.Text_vor_Einschub):]))
            else:
                teile[r.Debatte_ID].append((r.Zeile_von, r.Text))
        aufgaben, meta = [], []
        for deb, vg in vorlagen.groupby("Debatte_ID"):
            text = " ".join(t for _, t in sorted(teile.get(deb, [])))
            # Abstimmungen folgen der Aussprache: relevante Passagen ab dem ersten Abstimmungsbegriff
            m = re.search(r"Abstimmung|abstimmen|Überweisung|überwiesen|Wer stimmt", text)
            if not m:
                continue
            ausschnitt = text[max(0, m.start() - 300):][: self.max]
            liste = "\n".join(f"{i + 1}. {t[:300]} (Drucksachen: {', '.join(d)})"
                              for i, (t, d) in enumerate(zip(vg["Text"], vg["Drucksachen"])))
            aufgaben.append((SYSTEM_ABSTIMMUNG, f"Vorlagen:\n{liste}\n\nÄußerungen der Sitzungsleitung:\n{ausschnitt}",
                             SCHEMA_ABSTIMMUNG))
            meta.append((vg["Vorlage_ID"].tolist(), ausschnitt))
        antworten = self.b.chat_viele(aufgaben)
        zeilen = []
        for (ids, quelle), a in zip(meta, antworten):
            for e in (a or {}).get("ergebnisse", []):
                i = int(e.get("vorlage_nr", 0)) - 1
                if 0 <= i < len(ids):
                    ok = e.get("ergebnis") == "keine Abstimmung" or belegt(e.get("beleg"), quelle)
                    zeilen.append({"Vorlage_ID": ids[i], "Ergebnis": e.get("ergebnis") if ok else None,
                                   "Ergebnis_Beleg": e.get("beleg") if ok else None, "Beleg_ok": ok})
        out = pd.DataFrame(zeilen, columns=["Vorlage_ID", "Ergebnis", "Ergebnis_Beleg", "Beleg_ok"])
        return out.drop_duplicates("Vorlage_ID")


SCHEMA_ANTWORT = {"type": "object", "properties": {
    "antwort": {"type": "string"}, "belege": {"type": "array", "items": {"type": "integer"}}},
    "required": ["antwort", "belege"]}
SYSTEM_RAG = ("Beantworte die Frage ausschließlich auf Grundlage der nummerierten Redeauszüge aus dem Deutschen "
              "Bundestag. Nenne die Nummern der verwendeten Auszüge in 'belege'. Reichen die Auszüge nicht aus, "
              "antworte: 'Die bereitgestellten Quellen reichen für eine verlässliche Antwort nicht aus.'")


class RagQA:
    """LLM-gestützte Auswertung: Frage -> Hybrid-Retrieval (optional durch den Graphen auf Debatten/
    Gesetze eingeschränkt) -> Antwort mit Belegnummern, die auf Rede, Redner und Fundstelle zeigen."""

    def __init__(self, suche: HybridSuche, backend: OllamaBackend, reden: pd.DataFrame):
        self.s, self.b = suche, backend
        self.reden = reden.set_index("Rede_ID")

    def frage(self, frage: str, debatten: Iterable[str] | None = None, top: int = 8) -> dict:
        maske = None if debatten is None else self.s.c["Debatte_ID"].isin(list(debatten)).to_numpy()
        # Top-k nach RRF, im Kontext aber in Dokumentreihenfolge (DOS-RAG, Laitenberger et al.): erhält Struktur
        treffer = self.s.suche(frage, top=top, maske=maske).sort_values(["Sitzung", "Rede_ID", "Chunk_Nr"])
        kontext = "\n\n".join(
            f"[{i + 1}] {r.Redner} ({r.Partei_Gruppe}), {r.Rede_ID}: {r.Text[:1200]}" for i, r in enumerate(treffer.itertuples()))
        a = self.b.chat_json(SYSTEM_RAG, f"Frage: {frage}\n\nRedeauszüge:\n{kontext}", SCHEMA_ANTWORT) or {}
        nummern = [i for i in dict.fromkeys(a.get("belege", [])) if isinstance(i, int) and 1 <= i <= len(treffer)]
        belege = pd.DataFrame([{"Nr": i, "Rede_ID": (b := treffer.iloc[i - 1]).Rede_ID, "Redner": b.Redner,
                                "Partei": b.Partei_Gruppe,
                                "Fundstelle": f"S. {self.reden.loc[b.Rede_ID, 'Druckseite']} {self.reden.loc[b.Rede_ID, 'Quadrant']}"}
                               for i in nummern])
        return {"frage": frage, "antwort": a.get("antwort"), "belege": belege, "treffer": treffer}

In [20]:
# ---------------------------------------------------------------------
# Gesetze: Regel (IV) -> LLM (JSON-Schema, Belegprüfung) -> Entscheidung -> Identität -> Abstimmung
# ---------------------------------------------------------------------
LLM_NUTZBAR = LLM_BEREIT or BACKEND.db.execute("SELECT COUNT(*) FROM chat").fetchone()[0] > 0   # Nachvollzug aus Cache
with stufe("7_gesetze"):
    VORLAGEN = Vorlagen.aus_iv(IV, DEBATTEN)
    if LLM_NUTZBAR:
        VORLAGEN = VorlagenLLM(BACKEND, logger).anwenden(VORLAGEN)
    VORLAGEN = gesetz_entscheidung(VORLAGEN, GESETZ_MODUS)
    GESETZE, VORLAGEN = Gesetzesidentitaet.bilde(VORLAGEN)
    ABSTIMMUNG = AbstimmungLLM(BACKEND, logger).anwenden(BEITRAEGE, VORLAGEN) if LLM_NUTZBAR else pd.DataFrame(columns=["Vorlage_ID", "Ergebnis", "Ergebnis_Beleg", "Beleg_ok"])
    VORLAGEN = VORLAGEN.drop(columns=[c for c in ("Ergebnis", "Ergebnis_Beleg", "Beleg_ok") if c in VORLAGEN]).merge(ABSTIMMUNG, on="Vorlage_ID", how="left")
print(f"Vorlagen: {len(VORLAGEN)} | Gesetzesvorlagen: {VORLAGEN['Gesetz_final'].sum()} "
      f"(Quelle: {VORLAGEN.loc[VORLAGEN['Gesetz_final'], 'Gesetz_Quelle'].value_counts().to_dict()}) | Gesetze: {len(GESETZE)}")
display(GESETZE.head(10))
ERWAEHNUNGEN = gesetzeserwaehnungen(REDEN, GESETZE)
print(f"Gesetzesnennungen in Reden: {len(ERWAEHNUNGEN):,} | davon Gesetze der Tagesordnung: {ERWAEHNUNGEN['Gesetz_ID'].notna().sum():,}")
display(ERWAEHNUNGEN["Gesetz_genannt"].value_counts().head(15).to_frame("Nennungen"))

23:06:18 | INFO    | Stufe 7_gesetze                      5994.6 s


KeyboardInterrupt: 

## [↩](#toc) 8) Beteiligung und Hybrid-Retrieval (BM25 + Dense, RRF)<a class="anchor" id="k8"></a>

**Zwei Definitionen von „beteiligt“.**
- **Debatte** (strukturell): Wer in der Debatte spricht, in der eine Gesetzesvorlage beraten wird, ist beteiligt (`Zuordnung` = *direkt* bei einer Vorlage, sonst *verbundene Debatte*).
- **Inhalt**: In verbundenen Debatten (mehrere Vorlagen, z. B. Gesetzentwurf + Oppositionsanträge) wird jede Rede den Vorlagen zugeordnet, die sie tatsächlich behandelt. Direkte Debatten bleiben unverändert.

**Hybrid-RRF je Rede.** Die Vorlagen der Debatte werden nach drei Signalen gerankt und fusioniert:
$$\text{RRF}(v)=\sum_{i\in\{\text{BM25},\ \text{Dense},\ \text{explizit}\}}\frac{1}{k+\text{rang}_i(v)},\quad k=60$$
BM25 und Dense bewerten die Vorlagenanfrage (Titel + Kurztitel) gegen die Chunks der Rede (Maximum); *explizit* = Drucksachennummer oder Kurztitel wörtlich in der Rede. Zugeordnet werden die bestplatzierte Vorlage, jede Vorlage mit RRF ≥ (1 − δ) · Bestwert und jede explizit genannte. RRF benötigt keine Kalibrierung unterschiedlicher Score-Skalen; δ ist der einzige Parameter (DEV-Kalibrierung in Kap. 14).

**Chunks.** Sätze einer Rede werden bis 1.500 Zeichen gepackt; Chunks überschreiten nie Redegrenzen und tragen Rede, Redner, Partei und Debatte (Metadaten bleiben erhalten). Qwen3-Embeddings erhalten für Anfragen eine Aufgabeninstruktion, Dokumente bleiben unverändert. *Late Chunking* (Günther et al.) würde Token-Embeddings des Gesamttexts benötigen, die Ollama nicht bereitstellt – daher satzgepackte Chunks.

In [ ]:
# =====================================================================
# Hybrid-Retrieval: BM25 + Dense (Ollama-Embeddings) + Reciprocal Rank Fusion
# =====================================================================
STOPP = set("""aber alle allem allen aller alles als also am an ander andere anderen auch auf aus bei beim bin bis bist
da dabei damit dann das dass dem den denn der des dessen deshalb die dies diese diesem diesen dieser dieses doch dort du
durch ein eine einem einen einer eines er es etwa euch euer für gegen gibt hab habe haben hat hatte hier hin hinter ich
ihm ihn ihnen ihr ihre ihrem ihren ihrer im in ins ist ja jede jedem jeden jeder jedes jetzt kann kein keine keinem
keinen keiner können man mehr mein meine mit muss müssen nach nicht nichts noch nun nur ob oder ohne sehr sein seine
sich sie sind so soll sollen sondern sowie über um und uns unser unsere unter viel vom von vor wann war waren warum
was weil weiter welche wenn wer werden wie wieder will wir wird wo wurde würde zu zum zur zwar zwischen herr frau
kollege kollegen kolleginnen damen herren präsident präsidentin""".split())


def tokens(text: str) -> list[str]:
    """Kleinschreibung, Umlaut-Faltung, leichte Suffix-Kürzung (deutsche Flexion), Stoppwörter raus.
    Komposita werden zusätzlich in ihr letztes Glied zerlegt ('Haushaltsfinanzierungsgesetz' -> '…gesetz')."""
    t = unicodedata.normalize("NFKD", str(text).casefold())
    t = "".join(c for c in t if not unicodedata.combining(c)).replace("ß", "ss")
    out = []
    for w in re.findall(r"[a-z0-9]+(?:/[0-9]+)?", t):
        if w in STOPP or len(w) < 2:
            continue
        if not w[0].isdigit():
            for suf in ("ungen", "ungen", "ung", "en", "er", "es", "e", "n", "s"):
                if len(w) > 5 and w.endswith(suf):
                    w = w[: -len(suf)]
                    break
        out.append(w)
        if len(w) > 12 and not w[0].isdigit():
            out.append(w[-7:])                       # Kompositum-Kopf (grob)
    return out


class BM25:
    """Okapi BM25 (k1 = 1,5; b = 0,75) über eine Dokumentliste; Bewertung als dichtes Array."""

    def __init__(self, dokumente: list[list[str]], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.n = len(dokumente)
        self.laenge = np.array([len(d) for d in dokumente], dtype=float)
        self.avg = float(self.laenge.mean()) if self.n else 1.0
        self.index: dict[str, list[tuple[int, int]]] = defaultdict(list)
        for i, d in enumerate(dokumente):
            for w, c in Counter(d).items():
                self.index[w].append((i, c))
        self.idf = {w: math.log(1 + (self.n - len(p) + 0.5) / (len(p) + 0.5)) for w, p in self.index.items()}

    def bewerte(self, anfrage: list[str], maske: np.ndarray | None = None) -> np.ndarray:
        s = np.zeros(self.n)
        for w in set(anfrage):
            for i, c in self.index.get(w, ()):
                s[i] += self.idf[w] * c * (self.k1 + 1) / (c + self.k1 * (1 - self.b + self.b * self.laenge[i] / self.avg))
        if maske is not None:
            s[~maske] = -np.inf
        return s


def rrf(rangfolgen: list[np.ndarray], k: int = 60) -> np.ndarray:
    """Reciprocal Rank Fusion (Cormack et al. 2009): Σ 1/(k + Rang). Ränge beginnen bei 1;
    ein Dokument ohne Rang in einer Liste (np.inf) trägt dort nichts bei."""
    return sum(np.where(np.isfinite(r), 1.0 / (k + r), 0.0) for r in rangfolgen)


def raenge(scores: np.ndarray) -> np.ndarray:
    """Rang 1 = bester Score; -inf (ausgeschlossen) -> inf."""
    r = np.full(len(scores), np.inf)
    gueltig = np.isfinite(scores)
    ordnung = np.argsort(-scores[gueltig], kind="stable")
    r[np.flatnonzero(gueltig)[ordnung]] = np.arange(1, gueltig.sum() + 1)
    return r


class Chunks:
    """Strukturbewusste Chunks: Absätze einer Rede, gepackt bis max_zeichen; nie über Redegrenzen."""

    @staticmethod
    def bilde(reden: pd.DataFrame, max_zeichen: int = 1500) -> pd.DataFrame:
        zeilen = []
        for r in reden.loc[~reden["Leer"]].itertuples(index=False):
            puffer, nr = "", 0
            for satz in saetze(r.Text):
                if puffer and len(puffer) + len(satz) > max_zeichen:
                    zeilen.append((r.Rede_ID, nr, puffer)); nr += 1; puffer = ""
                puffer = f"{puffer} {satz}".strip()
            if puffer:
                zeilen.append((r.Rede_ID, nr, puffer))
        c = pd.DataFrame(zeilen, columns=["Rede_ID", "Chunk_Nr", "Text"])
        c.insert(0, "Chunk_ID", c["Rede_ID"] + ":C" + c["Chunk_Nr"].astype(str).str.zfill(2))
        return c.merge(reden[["Rede_ID", "Sitzung", "Debatte_ID", "Redner", "Partei_Gruppe"]], on="Rede_ID")


class HybridSuche:
    """BM25 und Dense über dieselben Chunks; Fusion per RRF. Dense entfällt ohne Backend (nur BM25)."""

    INSTRUKTION = ("Instruct: Gegeben ist der Titel einer Vorlage oder eine Frage zu einer Bundestagsdebatte. "
                   "Finde Redepassagen, die diesen Gegenstand inhaltlich behandeln.\nQuery: ")

    def __init__(self, chunks: pd.DataFrame, backend: OllamaBackend | None = None, k: int = 60, logger=None):
        self.c, self.backend, self.k = chunks.reset_index(drop=True), backend, k
        self.bm25 = BM25([tokens(t) for t in self.c["Text"]])
        self.E = None
        if backend is not None:
            t0 = time.perf_counter()
            self.E = backend.embed(self.c["Text"].tolist())
            if logger:
                logger.info("Dense-Index: %d Chunks | %s | %.1fs", len(self.c), backend.k.embed, time.perf_counter() - t0)

    def scores(self, anfragen: list[str], maske: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray | None]:
        """-> (BM25 [q × n], Dense-Kosinus [q × n] oder None)."""
        lex = np.vstack([self.bm25.bewerte(tokens(a), maske) for a in anfragen])
        if self.E is None:
            return lex, None
        Q = self.backend.embed(anfragen, praefix=self.INSTRUKTION)
        dense = Q @ self.E.T
        if maske is not None:
            dense[:, ~maske] = -np.inf
        return lex, dense

    def suche(self, frage: str, top: int = 10, maske: np.ndarray | None = None) -> pd.DataFrame:
        lex, dense = self.scores([frage], maske)
        r_lex = raenge(lex[0])
        listen = [r_lex] + ([raenge(dense[0])] if dense is not None else [])
        fused = rrf(listen, self.k)
        if maske is not None:
            fused[~maske] = -np.inf
        best = np.argsort(-fused)[:top]
        out = self.c.iloc[best].copy()
        out["RRF"] = fused[best]
        out["Rang_BM25"] = r_lex[best]
        out["Rang_Dense"] = listen[1][best] if dense is not None else np.nan
        return out


class Zuordnung:
    """Rede -> Vorlage innerhalb verbundener Debatten (mehrere Vorlagen): je Rede werden die Vorlagen
    der Debatte nach drei Signalen gerankt und per RRF fusioniert:
      1. lexikalisch – BM25 der Vorlagenanfrage (Titel, Kurztitel) über die Chunks der Rede (Maximum)
      2. semantisch  – Kosinus Instruktions-Anfrage ↔ Chunk-Embedding (Maximum)
      3. explizit    – Drucksachennummer oder Kurztitel wörtlich in der Rede (Rang 1, sonst kein Rang)
    Zugeordnet wird die bestplatzierte Vorlage sowie jede weitere mit fusioniertem Score ≥ (1 − δ) · Bestwert
    oder expliziter Nennung. δ ist der einzige Parameter (Kalibrierung auf DEV gegen DIP, Kap. 14)."""

    def __init__(self, suche: HybridSuche, delta: float = 0.10):
        self.s, self.delta = suche, delta

    def zuordnen(self, reden: pd.DataFrame, vorlagen: pd.DataFrame) -> pd.DataFrame:
        zeilen = []
        n_v = vorlagen.groupby("Debatte_ID").size()
        for deb, vg in vorlagen.loc[vorlagen["Debatte_ID"].map(n_v).gt(1)].groupby("Debatte_ID"):
            rg = reden.loc[reden["Debatte_ID"].eq(deb) & ~reden["Leer"] & reden["Beitragsart"].isin(REDEARTEN[:9])]
            if rg.empty:
                continue
            maske = self.s.c["Debatte_ID"].eq(deb).to_numpy()
            anfragen = [" ".join(x for x in (txt(t), txt(k)) if x) for t, k in zip(vg["Titel"], vg["Kurztitel"])]
            lex, dense = self.s.scores(anfragen, maske)
            chunk_rede = self.s.c["Rede_ID"].to_numpy()
            for rede in rg.itertuples(index=False):
                idx = np.flatnonzero(maske & (chunk_rede == rede.Rede_ID))
                if len(idx) == 0:
                    continue
                s_lex = lex[:, idx].max(axis=1)
                s_den = dense[:, idx].max(axis=1) if dense is not None else None
                explizit = np.array([self._nennt(rede.Text, v) for v in vg.itertuples(index=False)])
                listen = [raenge(s_lex)] + ([raenge(s_den)] if s_den is not None else [])
                listen.append(np.where(explizit, 1.0, np.inf))
                fused = rrf(listen, self.s.k)
                best = fused.max()
                for j, v in enumerate(vg.itertuples(index=False)):
                    zeilen.append({"Rede_ID": rede.Rede_ID, "Vorlage_ID": v.Vorlage_ID, "Debatte_ID": deb,
                                   "RRF": float(fused[j]), "BM25": float(s_lex[j]),
                                   "Dense": float(s_den[j]) if s_den is not None else np.nan,
                                   "Explizit": bool(explizit[j]), "Rang": int(raenge(fused)[j]),
                                   "Zugeordnet": bool(fused[j] >= (1 - self.delta) * best or explizit[j])})
        return pd.DataFrame(zeilen)

    @staticmethod
    def _nennt(text: str, v) -> bool:
        ds = [d for d in (v.Drucksachen if isinstance(v.Drucksachen, list) else []) if d]
        kurz = txt(v.Kurztitel)
        return any(d in text for d in ds) or (len(kurz) > 5 and kurz.casefold() in text.casefold())

In [ ]:
# ---------------------------------------------------------------------
# Beteiligung: Debatte (strukturell) und Inhalt (Hybrid-RRF in verbundenen Debatten)
# ---------------------------------------------------------------------
with stufe("8_beteiligung"):
    BETEILIGUNG_DEBATTE = beteiligung_debatte(REDEN, VORLAGEN)
    CHUNKS = Chunks.bilde(REDEN)
    if DENSE_UMFANG == "verbundene":
        _mehr = VORLAGEN.groupby("Debatte_ID").size()
        CHUNKS = CHUNKS.loc[CHUNKS["Debatte_ID"].map(_mehr).fillna(0).gt(1)]
    SUCHE = HybridSuche(CHUNKS, BACKEND if EMBED_BEREIT else None, k=RRF_K, logger=logger)
    ZUORDNUNG = Zuordnung(SUCHE, delta=ZUORDNUNG_DELTA).zuordnen(REDEN, VORLAGEN)
    _inhalt = (ZUORDNUNG.loc[ZUORDNUNG["Zugeordnet"]] if len(ZUORDNUNG) else pd.DataFrame(columns=["Rede_ID", "Vorlage_ID", "RRF"]))
    # Inhaltliche Sicht: direkte Debatten unverändert, verbundene Debatten nur zugeordnete Vorlagen
    _verb = BETEILIGUNG_DEBATTE["Zuordnung"].eq("verbundene Debatte")
    BETEILIGUNG_INHALT = pd.concat([
        BETEILIGUNG_DEBATTE.loc[~_verb],
        BETEILIGUNG_DEBATTE.loc[_verb].merge(_inhalt[["Rede_ID", "Vorlage_ID", "RRF"]], on=["Rede_ID", "Vorlage_ID"])
        .assign(Methode="Inhalt", Score=lambda d: d.pop("RRF"))], ignore_index=True)
    for _b in (BETEILIGUNG_DEBATTE, BETEILIGUNG_INHALT):
        _b["Sitzung"] = _b["Rede_ID"].str.extract(r"20/(\d+):")[0].astype(int)
print(f"Chunks: {len(CHUNKS):,} | Dense: {'ja' if SUCHE.E is not None else 'nein (nur BM25)'} | "
      f"Beteiligung (Rede × Gesetzesvorlage): Debatte {len(BETEILIGUNG_DEBATTE):,} | Inhalt {len(BETEILIGUNG_INHALT):,}")
GESETZ_BETEILIGTE = (BETEILIGUNG_INHALT.groupby(["Gesetz_ID", "Person_ID", "Redner", "Partei"], dropna=False)
                     .agg(Reden=("Rede_ID", "nunique"), Woerter=("Woerter", "sum"), Arten=("Beitragsart", lambda s: sorted(set(s))))
                     .reset_index().merge(GESETZE[["Gesetz_ID", "Gesetzestitel", "Kurztitel"]], on="Gesetz_ID"))
display(GESETZ_BETEILIGTE.head(10))

## [↩](#toc) 9) Sentiment<a class="anchor" id="k9"></a>

ParlaSent (`classla/xlm-r-parlasent`, XLM-R, auf Parlamentsreden trainiert) bewertet jeden Satz auf einer Skala 0 (negativ) … 5 (positiv). Ausgeschlossen sind Zwischenrufe (bereits beim Zeilenaufbau getrennt), Anrede- und Dankesformeln („Frau Präsidentin! Liebe Kolleginnen und Kollegen!“ erhielten sonst systematisch positive Werte) und Kurzsätze unter drei Wörtern. Je Rede: längengewichteter Mittelwert, normiert auf [−1, 1], und die Anteile negativer (< 5/3) und positiver (> 10/3) Sätze. **Tonalität ist nicht Position**: Eine Rede kann ein Gesetz unterstützen und dabei die Opposition scharf kritisieren. Das LLM dient in der Validierung als zweiter Bewerter (Klähn et al. 2025: Zero-Shot-LLMs sind auf deutschen Daten häufig konkurrenzfähig zu feinjustierten Modellen, aber prompt-sensitiv und nicht deterministisch).

In [ ]:
# =====================================================================
# Sentiment: ParlaSent (XLM-R, auf Parlamentsreden trainiert) satzweise, Aggregation je Rede
# =====================================================================
class Sentiment:
    """Satzweise Regression 0 (negativ) … 5 (positiv) mit 'classla/xlm-r-parlasent'.

    Warum ein Encoder statt LLM: >100.000 Sätze; ein feinjustierter Encoder ist um Größenordnungen
    schneller, deterministisch und auf Parlamentssprache trainiert. Das LLM dient als zweiter Bewerter
    in der Validierung (Kap. 14). Ergebnisse werden je Satz gecacht (SQLite, Schlüssel = Modell + Satz).

    Aggregation je Rede: längengewichteter Mittelwert (0–5), normiert auf [-1, 1]; Anteile negativer
    (< 5/3) und positiver (> 10/3) Sätze. Formelsätze (Anrede, Dank) und Zwischenrufe sind ausgeschlossen."""

    GRENZEN = (5 / 3, 10 / 3)

    def __init__(self, modell: str, cache: Path, logger: logging.Logger, batch: int = 64, geraet: str | None = None):
        self.modell, self.log, self.batch, self.geraet = modell, logger, batch, geraet
        self.db = sqlite3.connect(str(cache))
        self.db.execute("CREATE TABLE IF NOT EXISTS satz (key TEXT PRIMARY KEY, wert REAL)")
        self._pipe = None

    def _laden(self):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.geraet = self.geraet or ("cuda" if torch.cuda.is_available() else "cpu")
        if self.geraet == "cpu":
            self.log.warning("Sentiment läuft auf der CPU (langsam) – CUDA nicht verfügbar")
        self._tok = AutoTokenizer.from_pretrained(self.modell)
        self._mod = AutoModelForSequenceClassification.from_pretrained(self.modell).to(self.geraet).eval()
        if self.geraet == "cuda":
            self._mod = self._mod.half()
        self._pipe = torch

    def bewerte(self, saetze_: list[str]) -> np.ndarray:
        keys = [hashlib.sha1(f"{self.modell}|{s}".encode()).hexdigest() for s in saetze_]
        werte = {}
        for i in range(0, len(keys), 900):
            teil = keys[i:i + 900]
            werte.update(self.db.execute(f"SELECT key, wert FROM satz WHERE key IN ({','.join('?' * len(teil))})", teil))
        offen = sorted({k: s for k, s in zip(keys, saetze_) if k not in werte}.items(), key=lambda x: len(x[1]))
        if offen:
            if self._pipe is None:
                self._laden()
            torch = self._pipe
            t0 = time.perf_counter()
            for i in range(0, len(offen), self.batch):
                teil = offen[i:i + self.batch]
                enc = self._tok([s for _, s in teil], padding=True, truncation=True, max_length=256,
                                return_tensors="pt").to(self.geraet)
                with torch.no_grad():
                    out = self._mod(**enc).logits.float().squeeze(-1).cpu().numpy().reshape(-1)
                self.db.executemany("INSERT OR REPLACE INTO satz VALUES (?,?)",
                                    [(k, float(v)) for (k, _), v in zip(teil, out)])
                werte.update({k: float(v) for (k, _), v in zip(teil, out)})
            self.db.commit()
            self.log.info("Sentiment: %d neue Sätze | %.1fs | %s", len(offen), time.perf_counter() - t0, self.geraet)
        return np.array([werte[k] for k in keys], dtype=float)

    @classmethod
    def klasse(cls, wert: float) -> str:
        return "negativ" if wert < cls.GRENZEN[0] else "positiv" if wert > cls.GRENZEN[1] else "neutral"

    def je_rede(self, saetze_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        s = saetze_df.loc[~saetze_df["Formel"]].copy()
        s["Sentiment"] = self.bewerte(s["Satz"].tolist())
        s["Klasse"] = s["Sentiment"].map(self.klasse)
        s["_gew"] = s["Sentiment"] * s["Woerter"]
        g = s.groupby("Rede_ID")
        agg = pd.DataFrame({
            "Sentiment": g["_gew"].sum() / g["Woerter"].sum(),
            "Saetze_bewertet": g.size(),
            "Anteil_negativ": g["Klasse"].apply(lambda k: (k == "negativ").mean()),
            "Anteil_positiv": g["Klasse"].apply(lambda k: (k == "positiv").mean())})
        agg["Sentiment_norm"] = ((agg["Sentiment"] - 2.5) / 2.5).round(3)
        return agg.reset_index(), s


SCHEMA_SENT = {"type": "object", "properties": {"sentiment": {"type": "string", "enum": ["negativ", "neutral", "positiv"]}},
               "required": ["sentiment"]}
SYSTEM_SENT = ("Bewerte die Tonalität eines einzelnen Satzes aus einer Rede im Deutschen Bundestag: negativ, neutral "
               "oder positiv. Maßgeblich ist die ausgedrückte Haltung (Kritik, Ablehnung, Zustimmung, Lob), nicht das "
               "Thema. Sachaussagen und Forderungen ohne Wertung sind neutral.")


def llm_sentiment(backend: OllamaBackend, saetze_: list[str]) -> list[str | None]:
    antworten = backend.chat_viele([(SYSTEM_SENT, s, SCHEMA_SENT) for s in saetze_])
    return [a.get("sentiment") if a else None for a in antworten]

In [ ]:
# ---------------------------------------------------------------------
# Sentiment je Satz (ParlaSent) -> je Rede
# ---------------------------------------------------------------------
SAETZE = satzliste(REDEN)
SENTIMENT_SAETZE = pd.DataFrame()
if SENTIMENT_AKTIV and UMGEBUNG.get("transformers") and UMGEBUNG.get("torch"):
    with stufe("9_sentiment"):
        SENT = Sentiment(SENTIMENT_MODELL, CACHE_DIR / "sentiment.sqlite", logger)
        _agg, SENTIMENT_SAETZE = SENT.je_rede(SAETZE)
        REDEN = REDEN.drop(columns=[c for c in _agg.columns if c != "Rede_ID" and c in REDEN]).merge(_agg, on="Rede_ID", how="left")
else:
    logger.warning("Sentiment übersprungen (transformers/torch fehlen oder SENTIMENT_AKTIV=False)")
    for _c in ("Sentiment", "Sentiment_norm", "Anteil_negativ", "Anteil_positiv"):
        REDEN[_c] = np.nan
print(f"Sätze: {len(SAETZE):,} | davon Formeln ausgeschlossen: {SAETZE['Formel'].sum():,} | "
      f"bewertet: {len(SENTIMENT_SAETZE):,}")

## [↩](#toc) 10) Ergebnistabellen: Redner und Parteien<a class="anchor" id="k10"></a>

`REDNER` ist der in der Aufgabe geforderte Dataframe: Abgeordnetenname, Parteizugehörigkeit (mit Quelle), Redeanzahl, durchschnittliche und Gesamtlänge (Wörter, Zeichen, Sätze, geschätzte Minuten), Sitzungen, erhaltene Zwischenrufe/Beifall und mittleres Sentiment. `PARTEIEN` aggregiert nach Partei bzw. Gruppe (Rollenregel).

In [ ]:
# ---------------------------------------------------------------------
# Ergebnis-Dataframes der Aufgabe: Redner (Name, Partei, Anzahl, Ø/Gesamtlänge) und Parteien
# ---------------------------------------------------------------------
REDNER = redner_tabelle(REDEN, PERSONEN)
PARTEIEN = parteien_tabelle(REDEN)
print(f"Redner mit mindestens einer Rede: {len(REDNER)}")
display(REDNER.head(15))
display(PARTEIEN)

## [↩](#toc) 11) Graph der Relationen (SQLite, Neo4j-Export)<a class="anchor" id="k11"></a>

Die Fragestellungen sind Pfadfragen: *Person → Rede → Debatte ← Vorlage → Gesetz* über mehrere Lesungen und Sitzungen. Der Property-Graph (`nodes`, `edges` mit JSON-Eigenschaften) liegt in **SQLite** – Standardbibliothek, eine Datei, rekursive CTEs für k-Hop-Nachbarschaften; der kürzeste Pfad wird als Breitensuche mit einer SQL-Abfrage je Ebene berechnet. Ein CSV-Export im Format von `neo4j-admin import` macht das Modell auf Neo4j übertragbar.

| Kante | Bedeutung |
|:--|:--|
| `(Person)-[HAELT]->(Rede)-[IN_DEBATTE]->(Debatte)-[IN_SITZUNG]->(Sitzung)` | Struktur |
| `(Vorlage)-[BERATEN_IN]->(Debatte)`, `(Vorlage)-[GEHOERT_ZU]->(Gesetz)` | Gesetzesweg über Sitzungen |
| `(Rede)-[BETRIFFT {Methode, Score}]->(Vorlage)` | Beteiligung (Kap. 8) |
| `(Person)-[MITGLIED]->(Partei)` | Partei mit Evidenz |
| `(Person)-[ZWISCHENRUF {Anzahl}]->(Rede)`, `(Partei)-[BEIFALL {Anzahl}]->(Rede)` | Interaktion im Plenum |

In [ ]:
# =====================================================================
# Relationen als Graph: Property-Graph in SQLite (rekursive CTEs), Export für Neo4j
# =====================================================================
class ParlamentsGraph:
    """Knoten (id, label, name, props) und gerichtete Kanten (src, dst, typ, props) in SQLite.

    Warum SQLite statt Neo4j: keine Serverinstallation, Teil der Python-Standardbibliothek, eine Datei
    für die Abgabe; rekursive CTEs decken Pfad- und Nachbarschaftsabfragen ab. Für Neo4j wird ein
    importfähiger CSV-Export erzeugt (neo4j-admin import), d. h. das Modell ist übertragbar.

    Schema:  (Person)-[HAELT]->(Rede)-[IN_DEBATTE]->(Debatte)-[IN_SITZUNG]->(Sitzung)
             (Vorlage)-[BERATEN_IN]->(Debatte);  (Vorlage)-[GEHOERT_ZU]->(Gesetz)
             (Rede)-[BETRIFFT {Methode, Score}]->(Vorlage);  (Person)-[MITGLIED]->(Partei)
             (Person)-[ZWISCHENRUF {Anzahl}]->(Rede);  (Partei)-[BEIFALL {Anzahl}]->(Rede)
    """

    def __init__(self, pfad: Path):
        self.pfad = Path(pfad)
        self.db = sqlite3.connect(str(self.pfad))
        self.db.executescript("""
            CREATE TABLE IF NOT EXISTS nodes (id TEXT PRIMARY KEY, label TEXT NOT NULL, name TEXT, props TEXT);
            CREATE TABLE IF NOT EXISTS edges (src TEXT NOT NULL, dst TEXT NOT NULL, typ TEXT NOT NULL, props TEXT,
                                              PRIMARY KEY (src, dst, typ));
            CREATE INDEX IF NOT EXISTS i_edges_dst ON edges(dst, typ);
            CREATE INDEX IF NOT EXISTS i_nodes_label ON nodes(label);
            CREATE VIEW IF NOT EXISTS ungerichtet AS SELECT src AS a, dst AS b, typ FROM edges
                                                     UNION ALL SELECT dst, src, typ FROM edges;""")

    def leeren(self):
        self.db.executescript("DELETE FROM edges; DELETE FROM nodes;")

    @staticmethod
    def _props(r: dict, spalten: list[str]) -> str:
        def wert(x):
            if isinstance(x, (np.integer,)): return int(x)
            if isinstance(x, (np.floating,)): return None if np.isnan(x) else float(x)
            if isinstance(x, (list, tuple, dict, str, int, float, bool)) or x is None: return x
            return None if pd.isna(x) else str(x)
        return json.dumps({c: wert(r.get(c)) for c in spalten}, ensure_ascii=False)

    def knoten(self, df: pd.DataFrame, id_fn: Callable, label: str, name_spalte: str, props: list[str] = ()):
        zeilen = [(id_fn(r), label, str(r.get(name_spalte, "")), self._props(r, list(props))) for r in df.to_dict("records")]
        self.db.executemany("INSERT OR REPLACE INTO nodes VALUES (?,?,?,?)", zeilen)

    def kanten(self, df: pd.DataFrame, src_fn: Callable, dst_fn: Callable, typ: str, props: list[str] = ()):
        zeilen = [(src_fn(r), dst_fn(r), typ, self._props(r, list(props))) for r in df.to_dict("records")]
        self.db.executemany("INSERT OR REPLACE INTO edges VALUES (?,?,?,?)", zeilen)

    def aus_korpus(self, meta, debatten, vorlagen, gesetze, reden, personen, beteiligung, zwischenrufe, beitraege):
        self.leeren()
        self.knoten(meta, lambda r: f"Sitzung:20/{r['Sitzung']}", "Sitzung", "Datum", ["Sitzung", "Datum", "Seiten"])
        self.knoten(debatten, lambda r: f"Debatte:{r['Debatte_ID']}", "Debatte", "Titel", ["Typ", "Bloecke"])
        self.knoten(vorlagen, lambda r: f"Vorlage:{r['Vorlage_ID']}", "Vorlage", "Titel",
                    ["Vorlagentyp", "Beratung", "Drucksachen", "Gesetz_final", "Ergebnis"])
        self.knoten(gesetze, lambda r: f"Gesetz:{r['Gesetz_ID']}", "Gesetz", "Gesetzestitel", ["Kurztitel", "Drucksachen"])
        self.knoten(personen, lambda r: f"Person:{r['Person_ID']}", "Person", "Redner", ["Partei", "Rollen"])
        parteien = pd.DataFrame({"Partei": sorted(set(personen["Partei"]) | {p for f in zwischenrufe["Fraktionen"] for p in f})})
        self.knoten(parteien, lambda r: f"Partei:{r['Partei']}", "Partei", "Partei")
        self.knoten(reden, lambda r: f"Rede:{r['Rede_ID']}", "Rede", "Rede_ID",
                    ["Beitragsart", "Woerter", "Sentiment", "Druckseite", "Quadrant"])
        self.kanten(debatten, lambda r: f"Debatte:{r['Debatte_ID']}", lambda r: f"Sitzung:20/{r['Sitzung']}", "IN_SITZUNG")
        self.kanten(vorlagen, lambda r: f"Vorlage:{r['Vorlage_ID']}", lambda r: f"Debatte:{r['Debatte_ID']}", "BERATEN_IN")
        self.kanten(vorlagen.dropna(subset=["Gesetz_ID"]), lambda r: f"Vorlage:{r['Vorlage_ID']}",
                    lambda r: f"Gesetz:{r['Gesetz_ID']}", "GEHOERT_ZU")
        self.kanten(reden, lambda r: f"Rede:{r['Rede_ID']}", lambda r: f"Debatte:{r['Debatte_ID']}", "IN_DEBATTE")
        self.kanten(reden, lambda r: f"Person:{r['Person_ID']}", lambda r: f"Rede:{r['Rede_ID']}", "HAELT")
        self.kanten(personen, lambda r: f"Person:{r['Person_ID']}", lambda r: f"Partei:{r['Partei']}", "MITGLIED")
        if len(beteiligung):
            self.kanten(beteiligung, lambda r: f"Rede:{r['Rede_ID']}", lambda r: f"Vorlage:{r['Vorlage_ID']}",
                        "BETRIFFT", ["Methode", "Score", "Zuordnung"])
        # Interaktion: benannte Zwischenrufe (Person -> Rede) und Beifall (Partei -> Rede)
        z = zwischenrufe.merge(beitraege[["Sitzung", "Beitrag_Nr", "Rede_ID"]], on=["Sitzung", "Beitrag_Nr"]).dropna(subset=["Rede_ID"])
        rufer = (z.loc[z["Person"].notna()].assign(Person_ID=lambda d: d["Person"].map(personen_schluessel))
                 .groupby(["Person_ID", "Rede_ID"]).size().rename("Anzahl").reset_index())
        neu = rufer.loc[~rufer["Person_ID"].isin(personen["Person_ID"]), ["Person_ID"]].drop_duplicates()
        if len(neu):
            self.knoten(neu.assign(Redner=neu["Person_ID"]), lambda r: f"Person:{r['Person_ID']}", "Person", "Redner")
        self.kanten(rufer, lambda r: f"Person:{r['Person_ID']}", lambda r: f"Rede:{r['Rede_ID']}", "ZWISCHENRUF", ["Anzahl"])
        beifall = (z.loc[z["Art"].eq("Beifall")].explode("Fraktionen").dropna(subset=["Fraktionen"])
                   .groupby(["Fraktionen", "Rede_ID"]).size().rename("Anzahl").reset_index())
        self.kanten(beifall, lambda r: f"Partei:{r['Fraktionen']}", lambda r: f"Rede:{r['Rede_ID']}", "BEIFALL", ["Anzahl"])
        self.db.commit()
        return self

    # ---------------------------------------------------------------- Abfragen
    def sql(self, abfrage: str, parameter: tuple = ()) -> pd.DataFrame:
        return pd.read_sql_query(abfrage, self.db, params=parameter)

    def statistik(self) -> pd.DataFrame:
        return self.sql("""SELECT 'Knoten' AS art, label AS typ, COUNT(*) AS anzahl FROM nodes GROUP BY label
                           UNION ALL SELECT 'Kante', typ, COUNT(*) FROM edges GROUP BY typ ORDER BY art, anzahl DESC""")

    def gesetz_beteiligte(self, gesetz_id: str, methode: str = "Inhalt") -> pd.DataFrame:
        """Beteiligte eines Gesetzes über alle Lesungen/Sitzungen.
        'Inhalt':  Gesetz <-GEHOERT_ZU- Vorlage <-BETRIFFT- Rede <-HAELT- Person -MITGLIED-> Partei
        'Debatte': Gesetz <-GEHOERT_ZU- Vorlage -BERATEN_IN-> Debatte <-IN_DEBATTE- Rede <-HAELT- Person"""
        weg = ("JOIN edges r ON r.dst = v.src AND r.typ = 'BETRIFFT'" if methode == "Inhalt" else
               "JOIN edges bd ON bd.src = v.src AND bd.typ = 'BERATEN_IN' JOIN edges r ON r.dst = bd.dst AND r.typ = 'IN_DEBATTE'")
        return self.sql(f"""
            SELECT p.name AS Redner, pa.name AS Partei, COUNT(DISTINCT r.src) AS Reden,
                   GROUP_CONCAT(DISTINCT substr(r.src, 6, 6)) AS Sitzungen
            FROM edges v {weg}
            JOIN edges h  ON h.dst = r.src AND h.typ = 'HAELT'
            JOIN nodes p  ON p.id = h.src
            LEFT JOIN edges m ON m.src = p.id AND m.typ = 'MITGLIED'
            LEFT JOIN nodes pa ON pa.id = m.dst
            WHERE v.typ = 'GEHOERT_ZU' AND v.dst = ?
            GROUP BY p.id ORDER BY Reden DESC, Redner""", (f"Gesetz:{gesetz_id}",))

    def gesetz_verlauf(self, gesetz_id: str) -> pd.DataFrame:
        return self.sql("""
            SELECT s.name AS Datum, substr(s.id, 9) AS Sitzung, json_extract(vn.props,'$.Beratung') AS Beratung,
                   json_extract(vn.props,'$.Ergebnis') AS Ergebnis, vn.name AS Titel,
                   (SELECT COUNT(*) FROM edges r WHERE r.dst = bd.dst AND r.typ='IN_DEBATTE') AS Reden
            FROM edges v JOIN nodes vn ON vn.id = v.src
            JOIN edges bd ON bd.src = v.src AND bd.typ='BERATEN_IN'
            JOIN edges si ON si.src = bd.dst AND si.typ='IN_SITZUNG' JOIN nodes s ON s.id = si.dst
            WHERE v.typ='GEHOERT_ZU' AND v.dst = ? ORDER BY s.name""", (f"Gesetz:{gesetz_id}",))

    def nachbarschaft(self, knoten: str, tiefe: int = 2, limit: int = 200) -> pd.DataFrame:
        """k-Hop-Nachbarschaft (ungerichtet) per rekursiver CTE."""
        return self.sql("""
            WITH RECURSIVE n(id, tiefe) AS (SELECT ?, 0
                UNION SELECT u.b, n.tiefe + 1 FROM ungerichtet u JOIN n ON u.a = n.id WHERE n.tiefe < ?)
            SELECT n.id, nd.label, nd.name, MIN(n.tiefe) AS tiefe FROM n JOIN nodes nd ON nd.id = n.id
            GROUP BY n.id ORDER BY tiefe, nd.label LIMIT ?""", (knoten, tiefe, limit))

    def kuerzester_pfad(self, von: str, nach: str, max_tiefe: int = 6, typen: tuple[str, ...] | None = None) -> list[str] | None:
        """Kürzester ungerichteter Pfad: Breitensuche, je Ebene eine SQL-Abfrage der Nachbarn
        (eine rekursive CTE mit Pfadzeichenketten wächst auf dichten Graphen exponentiell)."""
        vorg, front = {von: None}, [von]
        for _ in range(max_tiefe):
            if nach in vorg or not front:
                break
            neu = []
            for i in range(0, len(front), 500):
                teil = front[i:i + 500]
                filt = f" AND typ IN ({','.join('?' * len(typen))})" if typen else ""
                for a, b, typ in self.db.execute(
                        f"SELECT a, b, typ FROM ungerichtet WHERE a IN ({','.join('?' * len(teil))}){filt}",
                        teil + list(typen or ())):
                    if b not in vorg:
                        vorg[b] = (a, typ); neu.append(b)
            front = neu
        if nach not in vorg:
            return None
        pfad, x = [nach], nach
        while vorg[x] is not None:
            a, typ = vorg[x]; pfad += [typ, a]; x = a
        return pfad[::-1]

    def interaktion(self) -> pd.DataFrame:
        """Fraktion des Rufers -> Fraktion des Redners: benannte Zwischenrufe und Beifall."""
        return self.sql("""
            SELECT pr.dst AS von, pz.dst AS an, e.typ AS art, SUM(json_extract(e.props,'$.Anzahl')) AS anzahl
            FROM edges e
            JOIN edges h ON h.dst = e.dst AND h.typ = 'HAELT'
            JOIN edges pz ON pz.src = h.src AND pz.typ = 'MITGLIED'
            JOIN edges pr ON pr.src = e.src AND pr.typ = 'MITGLIED'
            WHERE e.typ = 'ZWISCHENRUF'
            GROUP BY von, an, art
            UNION ALL
            SELECT e.src, pz.dst, e.typ, SUM(json_extract(e.props,'$.Anzahl'))
            FROM edges e JOIN edges h ON h.dst = e.dst AND h.typ='HAELT'
            JOIN edges pz ON pz.src = h.src AND pz.typ='MITGLIED'
            WHERE e.typ = 'BEIFALL' GROUP BY e.src, pz.dst""")

    def export_neo4j(self, ordner: Path) -> list[Path]:
        """CSV im Format von neo4j-admin import (id:ID, :LABEL / :START_ID, :END_ID, :TYPE)."""
        ordner = Path(ordner); ordner.mkdir(parents=True, exist_ok=True)
        n = self.sql("SELECT id AS 'id:ID', name, props, label AS ':LABEL' FROM nodes")
        e = self.sql("SELECT src AS ':START_ID', dst AS ':END_ID', typ AS ':TYPE', props FROM edges")
        pfade = [ordner / "neo4j_nodes.csv", ordner / "neo4j_edges.csv"]
        n.to_csv(pfade[0], index=False); e.to_csv(pfade[1], index=False)
        return pfade

In [ ]:
# ---------------------------------------------------------------------
# Graph aufbauen (SQLite) und beispielhaft abfragen
# ---------------------------------------------------------------------
with stufe("11_graph"):
    GRAPH = ParlamentsGraph(OUTPUT_DIR / "kom82_graph.sqlite").aus_korpus(
        META, DEBATTEN, VORLAGEN, GESETZE, REDEN, PERSONEN, BETEILIGUNG_INHALT, ZWISCHENRUFE, BEITRAEGE)
    NEO4J = GRAPH.export_neo4j(ERGEBNIS_DIR / "neo4j")
display(GRAPH.statistik())
if len(GESETZE):
    _g = GESETZ_BETEILIGTE.groupby("Gesetz_ID")["Person_ID"].nunique().idxmax()
    print(f"Beispiel {_g}: {GESETZE.set_index('Gesetz_ID').loc[_g, 'Gesetzestitel']}")
    display(GRAPH.gesetz_verlauf(_g))
    display(GRAPH.gesetz_beteiligte(_g).head(10))
    _p = GRAPH.gesetz_beteiligte(_g)
    if len(_p) > 1:
        _struktur = ("HAELT", "IN_DEBATTE", "BERATEN_IN", "GEHOERT_ZU", "IN_SITZUNG")
        print("Strukturpfad zwischen zwei Beteiligten:", " → ".join(GRAPH.kuerzester_pfad(
            f"Person:{personen_schluessel(_p['Redner'].iloc[0])}", f"Person:{personen_schluessel(_p['Redner'].iloc[-1])}",
            typen=_struktur) or []))
INTERAKTION = GRAPH.interaktion()

## [↩](#toc) 12) Visualisierung<a class="anchor" id="k12"></a>

Abbildungen (PNG in `output/abbildungen/`): Anzahl und Länge der Reden je Partei/Gruppe, Sentimentverteilung und Anteile negativer/positiver Sätze, Top-Redner (Anzahl, mittlere Länge, Sentiment), Häufigkeit vs. Länge je Redner, Interaktionsmatrizen (benannte Zwischenrufe, Beifall: Fraktion → Fraktion des Redners), Beteiligung je Gesetz nach Partei und das Beteiligungsnetz eines Gesetzes aus dem Graphen. Parteien tragen ihre Konventionsfarben; da Schwarz/Grau unbunt sind, stehen Parteinamen immer an Achse oder Legende.

In [ ]:
# =====================================================================
# Visualisierung (matplotlib): je Partei und je Redner – Anzahl, Länge, Sentiment, Interaktion
# =====================================================================
import matplotlib
import matplotlib.pyplot as plt

# Parteifarben (Konvention). Schwarz/Grau sind bewusst unbunt (CDU/CSU, fraktionslos); Identität wird
# daher nie allein über Farbe vermittelt: Parteinamen stehen immer an Achse oder Legende.
FARBEN = {"SPD": "#E3000F", "CDU/CSU": "#3A3A3A", "AfD": "#009EE0", "FDP": "#C9A800",
          "BÜNDNIS 90/DIE GRÜNEN": "#1AA037", "Die Linke": "#BE3075", "BSW": "#F07D00",
          "fraktionslos": "#8A8A8A", "Bundesregierung": "#5B6F86", "Länder": "#9C8F7A", "Sonstige": "#B5B5B5",
          "nicht im PDF belegt": "#CFCFCF"}
REIHENFOLGE = list(FARBEN)
KURZ = {"BÜNDNIS 90/DIE GRÜNEN": "Grüne", "fraktionslos": "fraktionsl.", "Bundesregierung": "Bundesreg."}

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E6E6E6", "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.edgecolor": "#888888"})


def _horizontal(fn, *a, **k):
    """matplotlib ≥ 3.10: orientation='horizontal'; ältere Versionen: vert=False."""
    try:
        return fn(*a, orientation="horizontal", **k)
    except TypeError:
        return fn(*a, vert=False, **k)


class Grafiken:
    def __init__(self, ordner: Path):
        self.ordner = Path(ordner); self.ordner.mkdir(parents=True, exist_ok=True)

    def _speichern(self, fig, name: str):
        fig.tight_layout()
        fig.savefig(self.ordner / f"{name}.png", bbox_inches="tight")
        return fig

    @staticmethod
    def _gruppen(df: pd.DataFrame, spalte: str = "Partei_Gruppe") -> list[str]:
        vorhanden = set(df[spalte].dropna())
        return [g for g in REIHENFOLGE if g in vorhanden] + sorted(vorhanden - set(REIHENFOLGE))

    def parteien(self, reden: pd.DataFrame):
        """Anzahl Reden, Gesamtwörter und Verteilung der Redelänge je Partei/Gruppe."""
        r = reden.loc[reden["Beitragsart"].eq("Rede") & ~reden["Leer"]]
        gr = self._gruppen(r)
        fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
        n = r.groupby("Partei_Gruppe").size().reindex(gr)
        w = r.groupby("Partei_Gruppe")["Woerter"].sum().reindex(gr) / 1000
        lab = [KURZ.get(g, g) for g in gr]
        for a, werte, titel in ((ax[0], n, "Anzahl Reden"), (ax[1], w, "Gesamtlänge (Tsd. Wörter)")):
            a.barh(lab, werte, color=[FARBEN.get(g, "#999") for g in gr], height=0.7, edgecolor="white", linewidth=1)
            a.invert_yaxis(); a.set_title(titel)
            for y, v in enumerate(werte):
                a.text(v, y, f" {v:,.0f}".replace(",", "."), va="center", fontsize=8, color="#333")
        daten = [r.loc[r["Partei_Gruppe"].eq(g), "Woerter"].to_numpy() for g in gr]
        bp = _horizontal(ax[2].boxplot, daten, showfliers=False, patch_artist=True, widths=0.6)
        ax[2].set_yticks(range(1, len(lab) + 1), lab)
        for patch, g in zip(bp["boxes"], gr):
            patch.set_facecolor(FARBEN.get(g, "#999")); patch.set_alpha(0.55)
        ax[2].invert_yaxis(); ax[2].set_title("Redelänge je Rede (Wörter, ohne Ausreißer)")
        return self._speichern(fig, "01_parteien_anzahl_laenge")

    def sentiment_parteien(self, reden: pd.DataFrame):
        r = reden.loc[reden["Beitragsart"].eq("Rede") & reden["Sentiment_norm"].notna()]
        if r.empty:
            return None
        gr = self._gruppen(r)
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
        daten = [r.loc[r["Partei_Gruppe"].eq(g), "Sentiment_norm"].to_numpy() for g in gr]
        vp = _horizontal(ax[0].violinplot, daten, showmedians=True, widths=0.8)
        for body, g in zip(vp["bodies"], gr):
            body.set_facecolor(FARBEN.get(g, "#999")); body.set_alpha(0.55); body.set_edgecolor("white")
        ax[0].set_yticks(range(1, len(gr) + 1), [KURZ.get(g, g) for g in gr]); ax[0].invert_yaxis()
        ax[0].axvline(0, color="#999", lw=0.8); ax[0].set_title("Sentiment je Rede (−1 negativ … +1 positiv)")
        anteil = r.groupby("Partei_Gruppe")[["Anteil_negativ", "Anteil_positiv"]].mean().reindex(gr)
        y = np.arange(len(gr))
        ax[1].barh(y, -anteil["Anteil_negativ"], color="#B2182B", height=0.65, label="negative Sätze")
        ax[1].barh(y, anteil["Anteil_positiv"], color="#2166AC", height=0.65, label="positive Sätze")
        ax[1].set_yticks(y, [KURZ.get(g, g) for g in gr]); ax[1].invert_yaxis(); ax[1].axvline(0, color="#999", lw=0.8)
        ax[1].set_title("Mittlerer Anteil negativer / positiver Sätze"); ax[1].legend(frameon=False, loc="lower right")
        ax[1].xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{abs(v):.0%}"))
        return self._speichern(fig, "02_sentiment_parteien")

    def redner(self, redner: pd.DataFrame, top: int = 25):
        d = redner.sort_values(["Reden", "Woerter_gesamt"], ascending=False).head(top)
        fig, ax = plt.subplots(1, 2, figsize=(14, 0.28 * top + 1.5), sharey=True)
        lab = [f"{n} ({KURZ.get(p, p)})" for n, p in zip(d["Redner"], d["Partei_Gruppe"])]
        farben = [FARBEN.get(p, "#999") for p in d["Partei_Gruppe"]]
        ax[0].barh(lab, d["Reden"], color=farben, height=0.7); ax[0].invert_yaxis(); ax[0].set_title(f"Top {top}: Anzahl Reden")
        ax[1].barh(lab, d["Woerter_Mittel"], color=farben, height=0.7, alpha=0.8)
        if "Sentiment_Mittel" in d:
            for y, (v, s) in enumerate(zip(d["Woerter_Mittel"], d["Sentiment_Mittel"])):
                if pd.notna(s):
                    ax[1].text(v, y, f"  Sentiment {s:+.2f}", va="center", fontsize=7.5, color="#333")
        ax[1].set_title("Mittlere Redelänge (Wörter) und mittleres Sentiment")
        return self._speichern(fig, "03_redner_top")

    def streuung_redner(self, redner: pd.DataFrame):
        d = redner.loc[redner["Reden"] > 0]
        fig, ax = plt.subplots(figsize=(8, 5))
        for g in self._gruppen(d):
            t = d.loc[d["Partei_Gruppe"].eq(g)]
            ax.scatter(t["Reden"], t["Woerter_Mittel"], s=18 + 2 * t["Reden"], color=FARBEN.get(g, "#999"),
                       alpha=0.7, edgecolor="white", linewidth=0.8, label=KURZ.get(g, g))
        for r in d.nlargest(8, "Reden").itertuples():
            ax.annotate(r.Redner, (r.Reden, r.Woerter_Mittel), fontsize=7, xytext=(4, 3), textcoords="offset points")
        ax.set_xlabel("Anzahl Reden"); ax.set_ylabel("Mittlere Länge (Wörter)")
        ax.set_title("Redner: Häufigkeit vs. Länge"); ax.legend(frameon=False, fontsize=7, ncol=2)
        return self._speichern(fig, "04_redner_streuung")

    def interaktion(self, matrix: pd.DataFrame, titel: str, name: str):
        """Heatmap Fraktion (Zeile: Rufer) -> Fraktion (Spalte: Redner), sequentiell (eine Farbe)."""
        if matrix.empty:
            return None
        zeilen = [g for g in REIHENFOLGE if g in matrix.index]
        spalten = [g for g in REIHENFOLGE if g in matrix.columns]
        m = matrix.reindex(index=zeilen, columns=spalten).fillna(0)
        fig, ax = plt.subplots(figsize=(1 + 0.8 * len(spalten), 0.9 + 0.5 * len(zeilen)))
        im = ax.imshow(m.to_numpy(), cmap="Blues", aspect="auto")
        ax.set_xticks(range(len(spalten)), [KURZ.get(s, s) for s in spalten], rotation=35, ha="right")
        ax.set_yticks(range(len(zeilen)), [KURZ.get(s, s) for s in zeilen]); ax.grid(False)
        grenze = m.to_numpy().max() * 0.6
        for i in range(len(zeilen)):
            for j in range(len(spalten)):
                v = m.iat[i, j]
                if v:
                    ax.text(j, i, f"{v:,.0f}".replace(",", "."), ha="center", va="center", fontsize=7.5,
                            color="white" if v > grenze else "#222")
        ax.set_xlabel("an Redner der Fraktion"); ax.set_ylabel("von Fraktion"); ax.set_title(titel)
        fig.colorbar(im, ax=ax, shrink=0.8)
        return self._speichern(fig, name)

    def gesetze(self, beteiligung: pd.DataFrame, gesetze: pd.DataFrame, top: int = 20):
        b = beteiligung.drop_duplicates(["Gesetz_ID", "Person_ID"])
        n = b.groupby(["Gesetz_ID", "Partei_Gruppe"]).size().unstack(fill_value=0)
        n = n.loc[n.sum(axis=1).sort_values(ascending=False).index[:top]]
        titel = gesetze.set_index("Gesetz_ID").apply(lambda r: txt(r["Kurztitel"]) or str(r["Gesetzestitel"])[:60], axis=1)
        fig, ax = plt.subplots(figsize=(11, 0.32 * len(n) + 1.5))
        links = np.zeros(len(n))
        for g in [g for g in REIHENFOLGE if g in n.columns]:
            ax.barh([titel.get(i, i) for i in n.index], n[g], left=links, color=FARBEN.get(g), height=0.7,
                    edgecolor="white", linewidth=1, label=KURZ.get(g, g))
            links += n[g].to_numpy()
        ax.invert_yaxis(); ax.legend(frameon=False, fontsize=7, ncol=4, loc="lower right")
        ax.set_title(f"Gesetze mit den meisten beteiligten Personen (Top {top}, nach Partei/Gruppe)")
        return self._speichern(fig, "06_gesetze_beteiligung")

    def graph(self, g: ParlamentsGraph, gesetz_id: str):
        """Beteiligungsnetz eines Gesetzes: Gesetz – Debatten – Personen (Farbe = Partei)."""
        try:
            import networkx as nx
        except ImportError:
            print("networkx nicht installiert – Netzgrafik übersprungen")
            return None
        knoten = g.sql("""
            SELECT DISTINCT bd.dst AS debatte, h.src AS person, pa.name AS partei, p.name AS name
            FROM edges v JOIN edges bd ON bd.src=v.src AND bd.typ='BERATEN_IN'
            JOIN edges r ON r.dst=bd.dst AND r.typ='IN_DEBATTE' JOIN edges h ON h.dst=r.src AND h.typ='HAELT'
            JOIN nodes p ON p.id=h.src LEFT JOIN edges m ON m.src=p.id AND m.typ='MITGLIED' LEFT JOIN nodes pa ON pa.id=m.dst
            WHERE v.typ='GEHOERT_ZU' AND v.dst=?""", (f"Gesetz:{gesetz_id}",))
        if knoten.empty:
            return None
        G = nx.Graph()
        G.add_node("G", art="gesetz")
        for r in knoten.itertuples():
            G.add_edge("G", r.debatte); G.add_edge(r.debatte, r.person)
            G.nodes[r.person].update(partei=r.partei, name=r.name)
        pos = nx.spring_layout(G, seed=42, k=0.6)
        fig, ax = plt.subplots(figsize=(9, 7)); ax.axis("off")
        nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#CCCCCC", width=0.8)
        farbe = [("#111111" if n == "G" else "#999999" if str(n).startswith("Debatte") else
                  FARBEN.get(G.nodes[n].get("partei"), "#BBBBBB")) for n in G]
        groesse = [500 if n == "G" else 220 if str(n).startswith("Debatte") else 90 for n in G]
        nx.draw_networkx_nodes(G, pos, ax=ax, node_color=farbe, node_size=groesse, edgecolors="white", linewidths=1)
        nx.draw_networkx_labels(G, pos, {n: G.nodes[n].get("name", "") for n in G if not str(n).startswith(("G", "Debatte"))},
                                font_size=6.5, ax=ax)
        ax.set_title(f"Beteiligungsnetz {gesetz_id}: Gesetz – Debatten (grau) – Personen (Parteifarbe)")
        return self._speichern(fig, f"07_graph_{gesetz_id}")

In [ ]:
# ---------------------------------------------------------------------
# Visualisierungen (PNG in output/abbildungen)
# ---------------------------------------------------------------------
GRAFIK = Grafiken(FIGUR_DIR)
GRAFIK.parteien(REDEN); plt.show()
if REDEN["Sentiment_norm"].notna().any():
    GRAFIK.sentiment_parteien(REDEN); plt.show()
GRAFIK.redner(REDNER); plt.show()
GRAFIK.streuung_redner(REDNER); plt.show()
_m = lambda art: (INTERAKTION.loc[INTERAKTION["art"].eq(art)].assign(von=lambda d: d["von"].str.replace("Partei:", ""),
                  an=lambda d: d["an"].str.replace("Partei:", "")).pivot_table(index="von", columns="an", values="anzahl", aggfunc="sum"))
if len(INTERAKTION):
    GRAFIK.interaktion(_m("ZWISCHENRUF"), "Benannte Zwischenrufe: Fraktion des Rufers → Fraktion des Redners", "05a_zwischenrufe"); plt.show()
    GRAFIK.interaktion(_m("BEIFALL"), "Beifall: Fraktion → Fraktion des Redners", "05b_beifall"); plt.show()
if len(BETEILIGUNG_INHALT):
    GRAFIK.gesetze(BETEILIGUNG_INHALT, GESETZE); plt.show()
    GRAFIK.graph(GRAPH, GESETZ_BETEILIGTE.groupby("Gesetz_ID")["Person_ID"].nunique().idxmax()); plt.show()

## [↩](#toc) 13) LLM-basierte Auswertung (RAG mit Belegen)<a class="anchor" id="k13"></a>

Fragen an den Korpus werden in drei Schritten beantwortet: (1) der **Graph** schränkt auf die Debatten eines Gesetzes ein (dokumentweise Zerlegung statt Suche im Gesamtkorpus, vgl. SPD-RAG, Yagiz et al.), (2) **Hybrid-RRF** liefert die Top-Passagen, die im Kontext in Originalreihenfolge stehen, (3) das **LLM** antwortet nur aus diesen Passagen und nennt die verwendeten Belegnummern; jede Nummer zeigt auf Rede, Redner und Fundstelle (Druckseite/Quadrant). Reichen die Passagen nicht, antwortet das Modell ausdrücklich, dass die Quellen nicht ausreichen.

In [ ]:
# ---------------------------------------------------------------------
# LLM-basierte Auswertung (RAG): Graph schränkt ein, Hybrid-RRF findet Belege, LLM antwortet mit Belegnummern
# ---------------------------------------------------------------------
if LLM_BEREIT and len(GESETZE):
    QA = RagQA(SUCHE, BACKEND, REDEN)
    _g = GESETZ_BETEILIGTE.groupby("Gesetz_ID")["Person_ID"].nunique().idxmax()
    _deb = VORLAGEN.loc[VORLAGEN["Gesetz_ID"].eq(_g), "Debatte_ID"].unique()
    _titel = txt(GESETZE.set_index("Gesetz_ID").loc[_g, "Kurztitel"]) or GESETZE.set_index("Gesetz_ID").loc[_g, "Gesetzestitel"]
    ANTWORT = QA.frage(f"Welche Argumente bringen die Fraktionen für und gegen das {_titel} vor?", debatten=_deb)
    print(ANTWORT["antwort"]); display(ANTWORT["belege"])
else:
    print("RAG-Auswertung übersprungen (LLM nicht verfügbar)")

## [↩](#toc) 14) Evaluation<a class="anchor" id="k14"></a>

**Protokoll.** Entwickelt wird nur auf DEV. Nach dem Methoden-Freeze: `EVAL_SPLITS = ("validierung",)`, danach einmalig `("test",)` mit `TEST_FREIGABE = True`; jede Testauswertung wird mit Zeitstempel und Pipelineversion in `test_auswertungen.jsonl` protokolliert. Die δ-Kalibrierung der Zuordnung läuft nur auf DEV.

| Prüfung | Referenz | Kennzahl |
|:--|:--|:--|
| Struktur (immer) | IV des PDFs | IV-Redner im Text gefunden, Beiträge ohne IV-Anker, Zeilentypen |
| Goldfälle | echte Formen aus den Protokollen | Unit-Tests (Kopf über zwei Zeilen, Ortszusatz, römische TOPs, Sammeldrucksache, Kurzintervention, Zwischenruf …) |
| Reden | DIP-Aktivitäten „Rede“ | TP = Σ_Person min(eigen, DIP); Precision/Recall; Parteiübereinstimmung |
| Beitragsarten | DIP-Aktivitätsarten | Gegenüberstellung der Häufigkeiten |
| Textvollständigkeit | amtliches XML (`<rede>`) | Wörter eigen/XML je Rede: Median der Abweichung, Anteil < 5 %, Korrelation |
| Gesetze | DIP-Vorgänge „Gesetzgebung“ | Precision/Recall/F1 für Regel, LLM, Hybrid (Drucksache, Kurztitel, Titelähnlichkeit) |
| Beteiligung | DIP-Vorgangsbezug der Reden | Person × Vorgang je Sitzung, bedingt auf gefundene Gesetze; Debatte vs. Inhalt(δ) |
| Sentiment | manuelle Annotation (falls vorhanden) | κ, Makro-F1 für ParlaSent und LLM |

Die Normierer der Evaluation (Personen, Titel, Parteien) sind **eigene Funktionen**; eine Änderung der Pipeline verändert die Messung nie unbemerkt.

In [ ]:
# =====================================================================
# Evaluation – getrennter Code: eigene Normierer, Kopplung an die Pipeline NUR über Ergebnistabellen
# =====================================================================
import difflib
import xml.etree.ElementTree as ET


_PARTEI_EVAL = {"spd": "SPD", "cdu/csu": "CDU/CSU", "afd": "AfD", "fdp": "FDP", "bündnis90/diegrünen": "BÜNDNIS 90/DIE GRÜNEN",
                "dielinke": "Die Linke", "bsw": "BSW", "fraktionslos": "fraktionslos"}


def _partei_eval(t: str) -> str | None:
    return _PARTEI_EVAL.get(re.sub(r"[\s.\-]+", "", str(t)).casefold())


def _schluessel_person(name: str) -> str:
    n = re.sub(r"\([^()]*\)", " ", str(name))
    n = re.sub(r"\b(?:Dr|Prof|h\.\s?c|med|rer\.\s?nat|phil|jur)\.", " ", n)
    n = unicodedata.normalize("NFKD", n)
    n = "".join(c for c in n if not unicodedata.combining(c)).casefold()
    return " ".join(re.findall(r"[a-z]+(?:-[a-z]+)?", n))


def _schluessel_titel(titel: str) -> str:
    t = unicodedata.normalize("NFKD", str(titel)).casefold()
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"\bentwurfs? (?:eines|einer)\b", " ", t)
    t = re.sub(r"(gesetz|gesetzbuch)es\b", r"\1", t)
    return " ".join(re.findall(r"[a-z0-9]+", t))


INHALT_STOPP = {"der", "die", "das", "des", "dem", "den", "und", "zur", "zum", "zu", "von", "vom", "im", "in", "fur",
                "uber", "mit", "sowie", "eines", "einer", "einem", "an", "auf", "weiterer", "vorschriften"}


def _inhaltswoerter(t: str) -> set[str]:
    return {w for w in t.split() if w not in INHALT_STOPP and not re.fullmatch(r"(?:erst|zweit|dritt|viert|funft|sechst|siebt|acht|neunt|zehnt)\w*", w)}


def titel_aehnlich(a: str, b: str) -> float:
    """Ähnlichkeit zweier Gesetzestitel: max(Sequenzähnlichkeit, Enthaltensein der Inhaltswörter ab 3)."""
    ka, kb = _schluessel_titel(a), _schluessel_titel(b)
    if not ka or not kb:
        return 0.0
    s = difflib.SequenceMatcher(None, ka, kb).ratio()
    wa, wb = _inhaltswoerter(ka), _inhaltswoerter(kb)
    kurz, lang = sorted((wa, wb), key=len)
    zahlen = lambda w: {x for x in w if x.isdigit()}
    if len(kurz) >= 3 and kurz <= lang and zahlen(kurz) <= zahlen(lang):
        s = max(s, 0.9)
    return s


class Referenz:
    """Liest DIP-JSON ({'documents': [...]}) und amtliche XML-Protokolle – ausschließlich hier."""

    def __init__(self, ordner: Path):
        self.ordner = Path(ordner)

    def dip(self, sitzung: int, name: str) -> list[dict]:
        p = self.ordner / "dip" / f"20_{sitzung:03d}" / f"{name}.json"
        if not p.is_file():
            return []
        d = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(d, dict) and "documents" in d:
            return d["documents"]
        return list(d.values()) if isinstance(d, dict) else d

    def xml_reden(self, sitzung: int) -> pd.DataFrame:
        """<rede>: Rednertext = <p> nach dem Rednerabsatz bis zu einem <name>-Element (Sitzungsleitung),
        danach wieder ab dem nächsten <p klasse='redner'>; <kommentar> (Zwischenrufe) zählt nicht."""
        p = self.ordner / "xml" / f"20_{sitzung:03d}.xml"
        if not p.is_file():
            return pd.DataFrame()
        zeilen = []
        for i, rede in enumerate(ET.parse(p).getroot().iter("rede")):
            red = rede.find(".//redner")
            name = " ".join(x for x in (red.findtext(".//vorname"), red.findtext(".//nachname")) if x) if red is not None else ""
            fraktion = red.findtext(".//fraktion") if red is not None else None
            rolle = red.findtext(".//rolle_kurz") if red is not None else None
            woerter, aktiv = 0, True
            for el in rede:
                if el.tag == "name":
                    aktiv = False
                elif el.tag == "p":
                    if el.get("klasse") == "redner":
                        aktiv = True
                        continue
                    if aktiv:
                        woerter += len("".join(el.itertext()).split())
            zeilen.append({"Sitzung": sitzung, "XML_Nr": i + 1, "Rede_XML": rede.get("id"), "Name": name,
                           "Key": _schluessel_person(name), "Fraktion": fraktion, "Rolle": rolle, "Woerter_XML": woerter})
        return pd.DataFrame(zeilen)


class Bewertung:
    def __init__(self, ref: Referenz, sitzungen: list[int]):
        self.ref, self.s = ref, sitzungen

    # ---------------------------------------------------------------- Reden
    def reden_dip(self, reden: pd.DataFrame, arten=("Rede",)) -> tuple[pd.DataFrame, dict]:
        """TP = Σ_Person min(eigen, DIP) je Protokoll (DIP liefert keine Positionen)."""
        zeilen, partei_ok = [], []
        for s in self.s:
            akt = [a for a in self.ref.dip(s, "aktivitaeten") if a.get("aktivitaetsart") in arten]
            if not akt:
                continue
            gold = Counter(_schluessel_person(str(a.get("titel", "")).split(",")[0]) for a in akt)
            e = reden.loc[reden["Sitzung"].eq(s) & reden["Beitragsart"].isin(arten) & ~reden["Leer"]]
            eigen = Counter(e["Redner"].map(_schluessel_person))
            tp = sum(min(n, eigen[k]) for k, n in gold.items())
            zeilen.append({"Sitzung": s, "DIP": sum(gold.values()), "Eigen": sum(eigen.values()), "TP": tp})
            gp = {}
            for a in akt:
                teile = [t.strip() for t in str(a.get("titel", "")).split(",")]
                f = next((_partei_eval(t) for t in teile[1:] if _partei_eval(t)), None)
                if f:
                    gp[_schluessel_person(teile[0])] = f
            for k, p in e.drop_duplicates("Redner").set_index(e.drop_duplicates("Redner")["Redner"].map(_schluessel_person))["Partei"].items():
                if k in gp and isinstance(p, str):
                    partei_ok.append(p == gp[k])
        df = pd.DataFrame(zeilen)
        return df, self._prf(df) | {"Partei_korrekt": float(np.mean(partei_ok)) if partei_ok else np.nan,
                                    "n_Partei": len(partei_ok)}

    ARTEN = {"Rede": ["Rede"], "Kurzintervention": ["Kurzintervention"], "Erwiderung": ["Erwiderung"],
             "Zwischenfrage": ["Zwischenfrage"], "Geschäftsordnung": ["Zur Geschäftsordnung BT"],
             "Frage": ["Frage", "Zusatzfrage"], "Antwort": ["Antwort", "Einleitende Ausführungen und Beantwortung"],
             "Rede (zu Protokoll)": ["Rede (zu Protokoll gegeben)"],
             "Erklärung (schriftlich)": ["Schriftliche Erklärung gem. § 31 Geschäftsordnung BT"],
             "Erklärung": ["Mündliche Erklärung gem. § 31 Geschäftsordnung BT",
                           "Erklärung zur Aussprache gem. § 30 Geschäftsordnung BT"]}

    def arten_dip(self, reden: pd.DataFrame) -> pd.DataFrame:
        """Je Beitragsart (zugeordnet zu DIP-Aktivitätsarten): Anzahl und TP über Personen (Σ min)."""
        zeilen = []
        for eigen_art, dip_arten in self.ARTEN.items():
            t = {"DIP": 0, "Eigen": 0, "TP": 0}
            for s in self.s:
                akt = [a for a in self.ref.dip(s, "aktivitaeten") if a.get("aktivitaetsart") in dip_arten]
                g = Counter(_schluessel_person(str(a.get("titel", "")).split(",")[0]) for a in akt)
                e = Counter(reden.loc[reden["Sitzung"].eq(s) & reden["Beitragsart"].eq(eigen_art) & ~reden["Leer"], "Redner"]
                            .map(_schluessel_person))
                t["DIP"] += sum(g.values()); t["Eigen"] += sum(e.values()); t["TP"] += sum(min(n, e[k]) for k, n in g.items())
            zeilen.append({"Beitragsart": eigen_art, **t})
        df = pd.DataFrame(zeilen).set_index("Beitragsart")
        return df.assign(**self._prf_spalten(df))

    def reden_xml(self, reden: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
        """Textvollständigkeit: i-te XML-Rede einer Person ↔ i-te eigene Einheit derselben Person."""
        paare = []
        for s in self.s:
            x = self.ref.xml_reden(s)
            if x.empty:
                continue
            e = reden.loc[reden["Sitzung"].eq(s) & reden["Bereich"].ne("anlagen") & ~reden["Leer"]].copy()
            e["Key"] = e["Redner"].map(_schluessel_person)
            e["i"] = e.groupby("Key").cumcount(); x["i"] = x.groupby("Key").cumcount()
            m = x.merge(e[["Key", "i", "Rede_ID", "Woerter", "Partei", "Beitragsart"]], on=["Key", "i"], how="left")
            paare.append(m.assign(n_eigen=len(e)))
        if not paare:
            return pd.DataFrame(), {}
        m = pd.concat(paare, ignore_index=True)
        ok = m.dropna(subset=["Rede_ID"])
        rel = (ok["Woerter"] - ok["Woerter_XML"]) / ok["Woerter_XML"].clip(lower=1)
        return m, {"XML_Reden": len(m), "zugeordnet": len(ok), "Anteil_zugeordnet": len(ok) / max(len(m), 1),
                   "Woerter_Verhaeltnis_gesamt": ok["Woerter"].sum() / max(ok["Woerter_XML"].sum(), 1),
                   "Median_abs_Abweichung": float(rel.abs().median()), "Anteil_Abw_unter_5pct": float((rel.abs() < 0.05).mean()),
                   "Korrelation": float(np.corrcoef(ok["Woerter"], ok["Woerter_XML"])[0, 1]) if len(ok) > 2 else np.nan}

    # ---------------------------------------------------------------- Gesetze
    def dip_gesetze(self, s: int) -> list[dict]:
        ds = defaultdict(set)
        for d in self.ref.dip(s, "drucksachen"):
            for b in d.get("vorgangsbezug") or []:
                if d.get("dokumentnummer"):
                    ds[str(b.get("id"))].add(d["dokumentnummer"])
        return [{"Vorgang": str(v.get("id")), "Titel": v.get("titel", ""), "Drucksachen": ds[str(v.get("id"))]}
                for v in self.ref.dip(s, "vorgaenge") if v.get("vorgangstyp") == "Gesetzgebung"]

    def _treffer(self, e: dict, g: dict) -> tuple[float, str]:
        if set(e.get("Drucksachen") or []) & g["Drucksachen"]:
            return 1.0, "Drucksache"
        kurz = e.get("Kurztitel")
        if isinstance(kurz, str) and len(kurz) > 5 and _schluessel_titel(kurz) in _schluessel_titel(g["Titel"]):
            return 0.95, "Kurztitel"
        return titel_aehnlich(e["Titel"], g["Titel"]), "Titel"

    def gesetze_dip(self, varianten: dict[str, pd.DataFrame], schwelle: float = 0.85) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Je Variante (Regel/LLM/Hybrid): eindeutige Zuordnung eigener Gesetzesvorlagen ↔ DIP-Vorgänge
        (Gesetzgebung) je Protokoll, greedy nach Score. Zusätzlich die Abbildung Vorlage -> alle DIP-Vorgänge
        mit Score ≥ Schwelle (für die Beteiligung; parallele Entwürfe gleichen Titels)."""
        zeilen, fehler, abbildung = [], [], []
        for name, df in varianten.items():
            for s in self.s:
                gold = self.dip_gesetze(s)
                eigen = df.loc[df["Sitzung"].eq(s)].to_dict("records")
                if not self.ref.dip(s, "vorgaenge"):
                    continue
                paare = sorted(((i, j, *self._treffer(e, g)) for i, e in enumerate(eigen) for j, g in enumerate(gold)),
                               key=lambda x: -x[2])
                gi, gj, tp = set(), set(), 0
                for i, j, sc, art in paare:
                    if sc >= schwelle and i not in gi and j not in gj:
                        gi.add(i); gj.add(j); tp += 1
                    if name == "Hybrid" and sc >= schwelle:
                        abbildung.append({"Sitzung": s, "Vorlage_ID": eigen[i]["Vorlage_ID"], "Vorgang": gold[j]["Vorgang"],
                                          "Score": sc, "Art": art})
                zeilen.append({"Variante": name, "Sitzung": s, "DIP": len(gold), "Eigen": len(eigen), "TP": tp})
                if name == "Hybrid":
                    fehler += [{"Sitzung": s, "Fehler": "nicht gefunden (FN)", "Titel": g["Titel"]} for j, g in enumerate(gold) if j not in gj]
                    fehler += [{"Sitzung": s, "Fehler": "ohne DIP-Entsprechung (FP)", "Titel": e["Titel"]} for i, e in enumerate(eigen) if i not in gi]
        df = pd.DataFrame(zeilen)
        summe = df.groupby("Variante", sort=False)[["DIP", "Eigen", "TP"]].sum() if len(df) else pd.DataFrame()
        if len(summe):
            summe = summe.assign(**self._prf_spalten(summe))
        return summe, pd.DataFrame(fehler), pd.DataFrame(abbildung)

    def beteiligung_dip(self, varianten: dict[str, pd.DataFrame], abbildung: pd.DataFrame,
                        arten=("Rede",)) -> pd.DataFrame:
        """Person × Vorgang je Sitzung, bedingt auf korrekt gefundene Gesetze (Abbildung aus gesetze_dip)."""
        if abbildung.empty:
            return pd.DataFrame()
        gefunden = set(zip(abbildung["Sitzung"], abbildung["Vorgang"]))
        gold, gekuerzt = set(), 0
        for s in self.s:
            for a in self.ref.dip(s, "aktivitaeten"):
                if a.get("aktivitaetsart") not in arten:
                    continue
                bez = a.get("vorgangsbezug") or []
                gekuerzt += int((a.get("vorgangsbezug_anzahl") or 0) > len(bez))
                k = _schluessel_person(str(a.get("titel", "")).split(",")[0])
                gold |= {(s, k, str(b.get("id"))) for b in bez if (s, str(b.get("id"))) in gefunden}
        zeilen = []
        for name, b in varianten.items():
            b = b.loc[b["Beitragsart"].isin(arten) & b["Sitzung"].isin(self.s)] if "Sitzung" in b else b
            m = b.merge(abbildung[["Vorlage_ID", "Vorgang"]], on="Vorlage_ID")
            pred = set(zip(m["Sitzung"], m["Redner"].map(_schluessel_person), m["Vorgang"]))
            tp = len(pred & gold)
            p, r = tp / max(len(pred), 1), tp / max(len(gold), 1)
            zeilen.append({"Variante": name, "Paare_eigen": len(pred), "Paare_DIP": len(gold), "TP": tp,
                           "Precision": p, "Recall": r, "F1": 2 * p * r / (p + r) if p + r else 0.0,
                           "DIP_gekuerzt": gekuerzt})
        return pd.DataFrame(zeilen)

    # ---------------------------------------------------------------- Sentiment
    @staticmethod
    def kappa(gold: list[str], pred: list[str]) -> dict:
        klassen, n = sorted(set(gold) | set(pred)), len(gold)
        if n == 0:
            return {}
        acc = sum(g == p for g, p in zip(gold, pred)) / n
        f1 = []
        for k in klassen:
            tp = sum(g == p == k for g, p in zip(gold, pred))
            fp = sum(p == k != g for g, p in zip(gold, pred)); fn = sum(g == k != p for g, p in zip(gold, pred))
            f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.0)
        pe = sum((gold.count(k) / n) * (pred.count(k) / n) for k in klassen)
        return {"n": n, "Accuracy": acc, "Makro_F1": float(np.mean(f1)), "Kappa": (acc - pe) / (1 - pe) if pe < 1 else 1.0}

    # ---------------------------------------------------------------- Hilfen
    @staticmethod
    def _prf(df: pd.DataFrame) -> dict:
        if df.empty:
            return {}
        t = df[["DIP", "Eigen", "TP"]].sum()
        p, r = t["TP"] / max(t["Eigen"], 1), t["TP"] / max(t["DIP"], 1)
        return {"Precision": p, "Recall": r, "F1": 2 * p * r / (p + r) if p + r else 0.0, **t.to_dict()}

    @staticmethod
    def _prf_spalten(s: pd.DataFrame) -> dict:
        p = s["TP"] / s["Eigen"].clip(lower=1)
        r = s["TP"] / s["DIP"].clip(lower=1)
        return {"Precision": p.round(3), "Recall": r.round(3), "F1": (2 * p * r / (p + r).replace(0, np.nan)).round(3)}

In [ ]:
# ---------------------------------------------------------------------
# Goldfälle als Unit-Tests (offline, < 1 s): echte Formen aus den Protokollen
# ---------------------------------------------------------------------
def selbsttest():
    f = []
    def pruefe(name, ist, soll):
        if ist != soll:
            f.append(f"{name}: {ist!r} != {soll!r}")
    k = parse_kopf("Dr. Robert Habeck, Bundesminister für Wirtschaft und Klimaschutz:")      # Kopf über zwei Zeilen
    pruefe("Kopf Minister", (k.name, k.rolle), ("Dr. Robert Habeck", "Bundesregierung"))
    k = parse_kopf("Detlef Müller (Chemnitz) (SPD):")
    pruefe("Kopf Ortszusatz", (k.name, k.fraktion, k.zusatz), ("Detlef Müller", "SPD", "Chemnitz"))
    k = parse_kopf("Dr. Markus Söder, Ministerpräsident (Bayern):")
    pruefe("Kopf Land", k.rolle, "Länder")
    pruefe("Kopf Sitzungsleitung", parse_kopf("Vizepräsidentin Katrin Göring-Eckardt:").rolle, "Sitzungsleitung")
    pruefe("Kopf Satz statt Kopf", parse_kopf("Damit ist die Sache für 2024 erledigt:"), None)
    pruefe("IV römischer TOP", IV_BLOCK.match("Tagesordnungspunkt I (Fortsetzung):").group("label"), "I")
    pruefe("IV Seitenverweis", SEITENVERWEIS.search("Katja Mast (SPD) . . . . . . 25814 C").groups(), ("25814", "C"))
    pruefe("Drucksache Sammel", Vorlagen._haupt_ds("Drucksachen 20/12685, 20/12868 Nr. 2.1, 20/13712", None), "20/12685")
    pruefe("Nominativ", nominativ("Zweiten Haushaltsfinanzierungsgesetzes 2024"), "Zweites Haushaltsfinanzierungsgesetz 2024")
    pruefe("Kurztitel", kurztitel("Gesetzes über … Haushaltsjahr 2024 (Nachtragshaushaltsgesetz 2024)"), "Nachtragshaushaltsgesetz 2024")
    st = Silbentrennung(Counter({"bundesregierung": 3}))
    pruefe("Trennung Wort", st.fuege(["die Bundes-", "regierung handelt"]), "die Bundesregierung handelt")
    pruefe("Trennung Koordination", st.fuege(["Wirtschafts-", "und Finanzpolitik"]), "Wirtschafts- und Finanzpolitik")
    pruefe("Trennung Großschreibung", st.fuege(["die AfD-", "Fraktion"]), "die AfD-Fraktion")
    pruefe("Sätze", len(saetze("Dr. Müller kam am 7. Oktober. Er sprach! Gut so?")), 3)
    z = zerlege_zwischenrufe(pd.DataFrame([{"Sitzung": 1, "Beitrag_Nr": 1, "Ruf_Nr": 1,
        "Text": "(Beifall bei der SPD – Jens Spahn [CDU/CSU]: Das fragen wir uns bei Ihnen auch!)"}]))
    pruefe("Zwischenruf Arten", z["Art"].tolist(), ["Beifall", "Zwischenruf"])
    pruefe("Zwischenruf Person", z["Person"].iloc[1], "Jens Spahn")
    pruefe("Hinweis Kurzintervention", Sitzungsmodell._hinweis(
        "Vielen Dank. – Ich erteile jetzt das Wort für eine Kurzintervention an Stefan Keuter.", "Stefan Keuter"), "Kurzintervention")
    pruefe("Hinweis Zwischenfrage (Angesprochener)", Sitzungsmodell._hinweis(
        "Herr Abgeordneter, erlauben Sie eine Zwischenfrage von der Kollegin Hagedorn?", "Josef Rief"), None)
    pruefe("Hinweis Zwischenfrage (aus Antwort des Redners)", Sitzungsmodell._hinweis(
        "Also, ich glaube, Sie erlauben die Zwischenfrage. Ich erlaube die Zwischenfrage von Kollege Rief, ja. Bitte schön, Herr Rief.",
        "Josef Rief"), "Zwischenfrage")
    pruefe("Hinweis Erwiderung", Sitzungsmodell._hinweis("", "Stephan Weil",
        "Herr Ministerpräsident, Sie haben die Möglichkeit, zu erwidern."), "Erwiderung")
    pruefe("RRF", round(float(rrf([np.array([1.0, 2.0]), np.array([2.0, np.inf])], 60)[0]), 5), round(1 / 61 + 1 / 62, 5))
    pruefe("Abstimmungsliste Name", bool(Abstimmungslisten.NAME.match("Michael Brand (Fulda)")), True)
    pruefe("Beleg", belegt("Nachtragshaushaltsgesetz  2024", "… (Nachtragshaushaltsgesetz 2024) …"), True)
    return f


FEHLER_SELBSTTEST = selbsttest()
print("Selbsttest:", "alle Goldfälle bestanden" if not FEHLER_SELBSTTEST else FEHLER_SELBSTTEST)

In [ ]:
# ---------------------------------------------------------------------
# Evaluation (nur dev): Split-Gating, DIP/XML-Referenz, Variantenvergleich
# ---------------------------------------------------------------------
EVAL = {}
if MODUS == "dev":
    if "test" in EVAL_SPLITS and not TEST_FREIGABE:
        raise PermissionError("Test-Split nur mit TEST_FREIGABE = True (einmalig nach dem Methoden-Freeze)")
    _s = [s for s in SPLIT.sitzungen(EVAL_SPLITS) if s in KORPUS.daten]
    BEW = Bewertung(Referenz(VALIDATION_DIR), _s)
    print(f"Evaluation auf {EVAL_SPLITS}: {len(_s)} Protokolle")
    EVAL["struktur"] = STRUKTUR.loc[STRUKTUR["Sitzung"].isin(_s)]
    EVAL["reden_tab"], EVAL["reden"] = BEW.reden_dip(REDEN)
    EVAL["arten"] = BEW.arten_dip(REDEN)
    EVAL["xml_tab"], EVAL["xml"] = BEW.reden_xml(REDEN)
    _var = {"Regel": gesetz_entscheidung(VORLAGEN, "regel"), "LLM": gesetz_entscheidung(VORLAGEN, "llm"),
            "Hybrid": gesetz_entscheidung(VORLAGEN, "hybrid")}
    # je Sitzung zählt ein Gesetz einmal (z. B. mehrere Einzelplan-Unterpunkte desselben Haushaltsgesetzes)
    _var = {k: v.loc[v["Gesetz_final"]].assign(Titel=v["Gesetzestitel_final"].fillna(v["Titel"]),
                                                  Kurztitel=v["Kurztitel_final"])
                 .drop_duplicates(["Sitzung", "Titel"]) for k, v in _var.items()}
    EVAL["gesetze"], EVAL["gesetze_fehler"], _abb = BEW.gesetze_dip(_var)
    if len(_abb):                                            # Abbildung auf alle Vorlagen desselben Gesetzes je Sitzung
        _h = _var["Hybrid"][["Sitzung", "Vorlage_ID", "Titel"]]
        _alle = gesetz_entscheidung(VORLAGEN, "hybrid").loc[lambda d: d["Gesetz_final"]]
        _alle = _alle.assign(Titel=_alle["Gesetzestitel_final"].fillna(_alle["Titel"]))[["Sitzung", "Vorlage_ID", "Titel"]]
        _abb = (_abb.merge(_h, on=["Sitzung", "Vorlage_ID"]).drop(columns="Vorlage_ID")
                .merge(_alle, on=["Sitzung", "Titel"]))
    _b = {"Debatte": BETEILIGUNG_DEBATTE, "Inhalt (δ=%.2f)" % ZUORDNUNG_DELTA: BETEILIGUNG_INHALT}
    for _d in (0.0, 0.05, 0.2, 0.5):                         # δ-Kalibrierung – nur auf DEV zulässig
        if EVAL_SPLITS == ("entwicklung",) and len(ZUORDNUNG):
            _z = ZUORDNUNG.assign(Zugeordnet=(ZUORDNUNG["RRF"] >= (1 - _d) * ZUORDNUNG.groupby("Rede_ID")["RRF"].transform("max"))
                                 | ZUORDNUNG["Explizit"])
            _v = BETEILIGUNG_DEBATTE["Zuordnung"].eq("verbundene Debatte")
            _b[f"Inhalt (δ={_d:.2f})"] = pd.concat([BETEILIGUNG_DEBATTE.loc[~_v], BETEILIGUNG_DEBATTE.loc[_v].merge(
                _z.loc[_z["Zugeordnet"], ["Rede_ID", "Vorlage_ID"]], on=["Rede_ID", "Vorlage_ID"])])
    EVAL["beteiligung"] = BEW.beteiligung_dip(_b, _abb)
    for k in ("reden", "xml"):
        print(k, {a: round(b, 3) if isinstance(b, float) else b for a, b in EVAL[k].items()})
    for k in ("arten", "gesetze", "beteiligung"):
        print(k); display(EVAL[k])
    if "test" in EVAL_SPLITS:
        SPLIT.protokolliere_test(EVAL_DIR, {"reden": EVAL["reden"], "xml": EVAL["xml"],
                                            "gesetze": EVAL["gesetze"].to_dict() if len(EVAL["gesetze"]) else {}})
    _suffix = "_".join(EVAL_SPLITS)
    for k, v in EVAL.items():
        if isinstance(v, pd.DataFrame) and len(v):
            v.to_csv(EVAL_DIR / f"eval_{k}_{_suffix}.csv", encoding="utf-8-sig")
else:
    print("Betrieb: keine Referenzdaten, nur referenzfreie Kontrollen (Kap. 5)")

## [↩](#toc) 15) Export und Laufzeitbericht<a class="anchor" id="k15"></a>

`output/ergebnisse/`: alle Tabellen als CSV (UTF-8 mit BOM für Excel) und – falls `pyarrow` installiert ist – Parquet; Arbeitsmappe `KOM82_Ergebnisse.xlsx` (Gesetze, Beteiligte, Redner, Parteien, Vorlagen, Reden, Struktur); Neo4j-CSV; `laufzeit.csv`. `output/text/`: Markdown je Protokoll. Mit `ZIP_ERSTELLEN = True` entsteht `output/KOM82_Daten.zip` (Ergebnisse, Texte, Abbildungen, PDFs) mit `manifest.json` (SHA-256 je Datei) für den Cloud-Upload.

**Abgabeläufe.** (1) `MODUS = "dev"` → Evaluation DEV, nach Freeze Validierung, dann Test (einmalig). (2) Kernel-Neustart, `MODUS = "betrieb"`, `ZIP_ERSTELLEN = True` → vollständiger Lauf über alle 67 Protokolle; Kennzahlen und Laufzeiten in den Bericht.

In [ ]:
# =====================================================================
# Export: Ergebnistabellen (CSV/Parquet), Arbeitsmappe, Abgabe-ZIP mit Prüfsummen
# =====================================================================
class Export:
    def __init__(self, ordner: Path, logger: logging.Logger):
        self.ordner, self.log = Path(ordner), logger
        self.ordner.mkdir(parents=True, exist_ok=True)

    def tabellen(self, tabellen: dict[str, pd.DataFrame]) -> list[Path]:
        pfade = []
        parquet = _modul_da("pyarrow")
        for name, df in tabellen.items():
            if df is None or not len(df):
                continue
            flach = df.copy()
            for c in flach.columns:
                if flach[c].map(lambda x: isinstance(x, (list, dict, tuple, np.ndarray))).any():
                    flach[c] = flach[c].map(lambda x: json.dumps(list(x) if isinstance(x, (tuple, np.ndarray)) else x,
                                                                 ensure_ascii=False, default=str))
            p = self.ordner / f"{name}.csv"
            flach.to_csv(p, index=False, encoding="utf-8-sig"); pfade.append(p)
            if parquet:
                flach.to_parquet(self.ordner / f"{name}.parquet", index=False); pfade.append(self.ordner / f"{name}.parquet")
        return pfade

    def arbeitsmappe(self, blaetter: dict[str, pd.DataFrame], name: str = "KOM82_Ergebnisse.xlsx") -> Path:
        p = self.ordner / name
        with pd.ExcelWriter(p, engine="openpyxl") as xw:
            for blatt, df in blaetter.items():
                if df is None or not len(df):
                    continue
                d = df.copy()
                for c in d.columns:
                    if d[c].dtype == object:
                        d[c] = d[c].map(lambda x: json.dumps(x, ensure_ascii=False, default=str) if isinstance(x, (list, dict))
                                        else (x[:32000] if isinstance(x, str) else x))
                d.to_excel(xw, sheet_name=blatt[:31], index=False)
        return p

    @staticmethod
    def zip_abgabe(ziel: Path, ordner: dict[str, Path]) -> Path:
        """ZIP mit Ergebnissen, Texten und PDFs; manifest.json mit SHA-256 je Datei."""
        manifest = {}
        with zipfile.ZipFile(ziel, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for praefix, o in ordner.items():
                for f in sorted(Path(o).rglob("*")):
                    if f.is_file() and f.suffix != ".part":
                        arc = f"{praefix}/{f.relative_to(o).as_posix()}"
                        zf.write(f, arc)
                        manifest[arc] = hashlib.sha256(f.read_bytes()).hexdigest()
            zf.writestr("manifest.json", json.dumps(manifest, indent=1))
        return ziel

In [ ]:
# ---------------------------------------------------------------------
# Export: Tabellen (CSV/Parquet), Arbeitsmappe, Laufzeitbericht, optional Abgabe-ZIP; RunPod beenden
# ---------------------------------------------------------------------
with stufe("15_export"):
    EXPORT = Export(ERGEBNIS_DIR, logger)
    _gesetze_aus = (GESETZE.merge(GESETZ_BETEILIGTE.groupby("Gesetz_ID").agg(
        Beteiligte=("Person_ID", "nunique"),
        Beteiligte_Liste=("Redner", lambda s: sorted(set(s)))), on="Gesetz_ID", how="left"))
    _tab = {"sitzungen": META, "debatten": DEBATTEN, "vorlagen": VORLAGEN, "gesetze": _gesetze_aus,
            "gesetz_beteiligte": GESETZ_BETEILIGTE, "beteiligung_debatte": BETEILIGUNG_DEBATTE,
            "beteiligung_inhalt": BETEILIGUNG_INHALT, "zuordnung_rrf": ZUORDNUNG, "reden": REDEN,
            "beitraege": BEITRAEGE, "zwischenrufe": ZWISCHENRUFE, "personen": PERSONEN, "redner": REDNER,
            "parteien": PARTEIEN.reset_index(), "struktur_pruefung": STRUKTUR}
    EXPORT.tabellen(_tab)
    EXPORT.arbeitsmappe({"Gesetze": _gesetze_aus, "Gesetz_Beteiligte": GESETZ_BETEILIGTE, "Redner": REDNER,
                         "Parteien": PARTEIEN.reset_index(), "Vorlagen": VORLAGEN.drop(columns=["Zeilen"], errors="ignore"),
                         "Reden": REDEN.drop(columns=["Text"]), "Struktur": STRUKTUR})
    LAUFZEIT = pd.DataFrame([{"Stufe": k, "Sekunden": v} for k, v in LAUFZEITEN.items()])
    LAUFZEIT.to_csv(ERGEBNIS_DIR / "laufzeit.csv", index=False)
    if ZIP_ERSTELLEN:
        _zip = Export.zip_abgabe(OUTPUT_DIR / "KOM82_Daten.zip", {"ergebnisse": ERGEBNIS_DIR, "text": TEXT_DIR,
                                                                  "abbildungen": FIGUR_DIR, "pdf": PDF_DIR})
        print(f"ZIP: {_zip} ({_zip.stat().st_size / 2**20:.0f} MB)")
if RUNPOD_SITZUNG is not None:
    RUNPOD_SITZUNG.__exit__(None, None, None)
print(f"Modellaufrufe: {dict(BACKEND.statistik)}")
display(LAUFZEIT)

## [↩](#toc) 16) Grenzen und Ausblick<a class="anchor" id="k16"></a>

| Bereich | Grenze | Umgang / Ausblick |
|:--|:--|:--|
| Layout | Typografie (Kopfschrift, Einzüge) ist für WP 20 konstant; ein geänderter Satz bricht die Erkennung | Kontrolle je Protokoll über die IV-Abdeckung (Kap. 5); Schwellen zentral in `LayoutParameter` |
| Backend | Entwickelt und regressionsgeprüft mit pdfplumber; PyMuPDF liefert dieselbe Wörtertabelle, die Gleichheit ist auf dem Zielsystem zu bestätigen | Vergleich: beide Backends auf einem DEV-Protokoll, Kennzahlen aus Kap. 5 müssen übereinstimmen |
| Redeeinheit | DIP und das PDF definieren Reden teils anders (Unterbrechung, Erwiderung) | Fehleranalyse je Person in Kap. 14 |
| Partei | Regierungsmitglieder ohne Mandat haben im PDF keine Parteiangabe | „nicht im PDF belegt“; Gruppe *Bundesregierung* |
| Gesetze | Gesetzgebung nach Titel und Vorspann; Gesetze, die nur in Reden genannt werden, sind „Erwähnungen“, keine Beratung | eigene Tabelle `ERWAEHNUNGEN` mit Passage |
| Beteiligung | DIP verknüpft Reden teils mit allen Vorlagen einer verbundenen Debatte | beide Definitionen ausgewiesen; δ auf DEV kalibriert |
| Sentiment | Modell nicht auf Deutsch feinjustiert; Tonalität ≠ Position | Validierung gegen Annotation; LLM als zweiter Bewerter |
| Retrieval | Dense-Qualität hängt vom Embeddingmodell ab | Austausch über `MODELL_PARAMETER["embed"]`; Cache je Modell-Digest |

**Verbesserungspotenziale.** Positionserkennung (Zustimmung/Ablehnung zum Gesetz) statt Tonalität; Namenslisten namentlicher Abstimmungen auch als Abstimmungsverhalten je Person auswerten; Embedding-Vergleich `qwen3-embedding:0.6b` vs. `4b` vs. `bge-m3` auf der DEV-Zuordnung; Verknüpfung der Gesetze mit dem späteren Inkrafttreten (Bundesgesetzblatt).